In [73]:

from pathlib import Path
import json

# Locate the Phase 7 query file
matches = list(
    Path("/kaggle/input").rglob("train.json")
)

phase7_root = None

for path in matches:
    if path.parent.name.lower() == "train":
        candidate = path.parent.parent
        if (candidate / "Test" / "test.json").exists():
            phase7_root = candidate
            break

if phase7_root is None:
    print("Phase 7 dataset not found.")
    print("Attach the official dataset through Kaggle Add Input.")
else:
    print("Phase 7 root:", phase7_root)

    required_files = [
        "Train/train.json",
        "Train/train_candidates.json",
        "Dev/dev.json",
        "Dev/dev_candidates.json",
        "Test/test.json",
        "Test/test_candidates.json"
    ]

    for name in required_files:
        p = phase7_root / name
        print(("OK" if p.exists() else "MISSING"), name)

    if all((phase7_root / x).exists() for x in required_files):
        for name in required_files:
            with open(phase7_root / name, encoding="utf-8") as f:
                data = json.load(f)
            print(
                f"{name}: {len(data)} records | "
                f"sample keys: {list(data)[:3]}"
            )


Phase 7 dataset not found.
Attach the official dataset through Kaggle Add Input.


In [74]:
from pathlib import Path
import json

# Find all JSON files under Kaggle inputs
all_json = list(Path("/kaggle/input").rglob("*.json"))

# Find a folder containing all six Phase 7 files
phase7_root = None

for folder in set(p.parent for p in all_json):
    files = {p.name.lower() for p in folder.glob("*.json")}

    required = [
        "train_candidates.json",
        "dev_candidates.json",
        "test_candidates.json"
    ]

    has_train = any(
        name.startswith("train") and "candidate" not in name
        for name in files
    )
    has_dev = any(
        name.startswith("dev") and "candidate" not in name
        for name in files
    )
    has_test = any(
        name.startswith("test") and "candidate" not in name
        for name in files
    )

    if all(x in files for x in required) and has_train and has_dev and has_test:
        phase7_root = folder
        break

if phase7_root is None:
    raise FileNotFoundError("Could not locate all six Phase 7 JSON files.")

print("Phase 7 folder:", phase7_root)

def find_file(prefix, candidate=False):
    matches = [
        p for p in phase7_root.glob("*.json")
        if p.name.lower().startswith(prefix)
        and (("candidate" in p.name.lower()) == candidate)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {prefix} file (candidate={candidate}), "
            f"found: {matches}"
        )
    return matches[0]

paths = {
    "train": find_file("train"),
    "train_candidates": find_file("train", True),
    "dev": find_file("dev"),
    "dev_candidates": find_file("dev", True),
    "test": find_file("test"),
    "test_candidates": find_file("test", True),
}

data = {}

for name, path in paths.items():
    with open(path, "r", encoding="utf-8") as f:
        data[name] = json.load(f)

    print(f"{name:18s}: {len(data[name]):,} records | {path.name}")

print("\nAll six files loaded successfully.")

Phase 7 folder: /kaggle/input/datasets/aq1de3/phase7
train             : 827 records | train (4).json
train_candidates  : 4,320 records | train_candidates.json
dev               : 118 records | dev (3).json
dev_candidates    : 1,023 records | dev_candidates.json
test              : 237 records | test (5).json
test_candidates   : 1,727 records | test_candidates.json

All six files loaded successfully.


In [75]:
for name in ["train", "dev", "test"]:
    query_data = data[name]
    candidate_data = data[f"{name}_candidates"]

    print(f"\n{'='*50}")
    print(name.upper())
    print("="*50)

    query_id = next(iter(query_data))
    candidate_id = next(iter(candidate_data))

    print("Sample query ID:", query_id)
    print("Query fields:", list(query_data[query_id].keys()))
    print("Sample candidate ID:", candidate_id)
    print("Candidate fields:", list(candidate_data[candidate_id].keys()))

    print("Query text type:", type(query_data[query_id].get("text")))
    print("Candidate text type:", type(candidate_data[candidate_id].get("text")))

    if "relevant_candidates" in query_data[query_id]:
        print(
            "Relevant candidates:",
            len(query_data[query_id]["relevant_candidates"])
        )


TRAIN
Sample query ID: 0000000599
Query fields: ['text', 'relevant_candidates']
Sample candidate ID: 0000000262
Candidate fields: ['text']
Query text type: <class 'list'>
Candidate text type: <class 'list'>
Relevant candidates: 2

DEV
Sample query ID: 0000012637
Query fields: ['text', 'relevant_candidates']
Sample candidate ID: 0000003060
Candidate fields: ['text']
Query text type: <class 'list'>
Candidate text type: <class 'list'>
Relevant candidates: 11

TEST
Sample query ID: 0000021652
Query fields: ['text']
Sample candidate ID: 0000000987
Candidate fields: ['text']
Query text type: <class 'list'>
Candidate text type: <class 'list'>


In [76]:
from collections import Counter
import statistics

def text_stats(records):
    sentence_counts = []
    char_counts = []

    for case_id, item in records.items():
        text = item.get("text")

        if not isinstance(text, list):
            raise TypeError(f"{case_id}: text is not a list")

        sentence_counts.append(len(text))
        char_counts.append(sum(len(str(s)) for s in text))

    return {
        "cases": len(records),
        "empty_texts": sum(
            not item.get("text") for item in records.values()
        ),
        "avg_sentences": round(statistics.mean(sentence_counts), 2),
        "max_sentences": max(sentence_counts, default=0),
        "avg_characters": round(statistics.mean(char_counts), 2),
        "max_characters": max(char_counts, default=0),
    }


for split in ["train", "dev", "test"]:
    queries = data[split]
    candidates = data[f"{split}_candidates"]

    query_ids = set(queries)
    candidate_ids = set(candidates)

    print(f"\n{'='*60}")
    print(f"{split.upper()} DATA AUDIT")
    print("="*60)

    print("Queries:", len(query_ids))
    print("Candidates:", len(candidate_ids))
    print("Query/candidate ID overlap:", len(query_ids & candidate_ids))

    print("\nQuery text statistics:")
    print(text_stats(queries))

    print("\nCandidate text statistics:")
    print(text_stats(candidates))

    if split in ["train", "dev"]:
        missing_relevant = []
        relevant_counts = []

        for query_id, item in queries.items():
            relevant = item.get("relevant_candidates", [])

            if not isinstance(relevant, list):
                raise TypeError(
                    f"{query_id}: relevant_candidates is not a list"
                )

            relevant_counts.append(len(relevant))

            for candidate_id in relevant:
                if candidate_id not in candidate_ids:
                    missing_relevant.append(
                        (query_id, candidate_id)
                    )

        print("\nRelevance statistics:")
        print("Queries with relevance labels:", len(relevant_counts))
        print("Average relevant candidates:",
              round(statistics.mean(relevant_counts), 2))
        print("Minimum relevant candidates:",
              min(relevant_counts, default=0))
        print("Maximum relevant candidates:",
              max(relevant_counts, default=0))
        print("Missing relevant candidate IDs:", len(missing_relevant))

        if missing_relevant:
            print("First missing examples:", missing_relevant[:5])

    print("\nCandidate text fields valid:",
          all(isinstance(x.get("text"), list)
              for x in candidates.values()))


TRAIN DATA AUDIT
Queries: 827
Candidates: 4320
Query/candidate ID overlap: 827

Query text statistics:
{'cases': 827, 'empty_texts': 0, 'avg_sentences': 59.42, 'max_sentences': 766, 'avg_characters': 39286.19, 'max_characters': 716495}

Candidate text statistics:
{'cases': 4320, 'empty_texts': 0, 'avg_sentences': 53.17, 'max_sentences': 2981, 'avg_characters': 36369.54, 'max_characters': 2423006}

Relevance statistics:
Queries with relevance labels: 827
Average relevant candidates: 6.42
Minimum relevant candidates: 0
Maximum relevant candidates: 52
Missing relevant candidate IDs: 0

Candidate text fields valid: True

DEV DATA AUDIT
Queries: 118
Candidates: 1023
Query/candidate ID overlap: 118

Query text statistics:
{'cases': 118, 'empty_texts': 0, 'avg_sentences': 73.12, 'max_sentences': 1111, 'avg_characters': 48667.47, 'max_characters': 618469}

Candidate text statistics:
{'cases': 1023, 'empty_texts': 0, 'avg_sentences': 71.36, 'max_sentences': 2981, 'avg_characters': 49906.97, 'm

In [77]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "tokenizer.json":
            print(os.path.join(root, file))

/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json


In [78]:
import os
from tokenizers import Tokenizer, decoders

TOKENIZER_PATH = "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"

assert os.path.exists(TOKENIZER_PATH), "Tokenizer file not found!"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = decoders.ByteLevel()

print("Tokenizer loaded successfully!")
print("Vocabulary:", tokenizer.get_vocab_size())
print("BOS:", tokenizer.token_to_id("[BOS]"))
print("EOS:", tokenizer.token_to_id("[EOS]"))
print("PAD:", tokenizer.token_to_id("[PAD]"))

Tokenizer loaded successfully!
Vocabulary: 50000
BOS: 2
EOS: 3
PAD: 0


In [79]:

from tokenizers import Tokenizer, decoders
from collections import Counter
import numpy as np
import os

# ==================================================
# 0. LOAD TOKENIZER
# ==================================================

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "tokenizer/tokenizer.json"
)

assert os.path.exists(TOKENIZER_PATH), "Tokenizer file not found!"

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = decoders.ByteLevel()

print("Tokenizer vocabulary:", tokenizer.get_vocab_size())

print("\nSpecial token IDs:")
for token in ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[MASK]"]:
    print(token, tokenizer.token_to_id(token))


# ==================================================
# 1. SELF-RETRIEVAL AUDIT
# ==================================================

for split in ["train", "dev", "test"]:

    queries = data[split]
    candidates = data[f"{split}_candidates"]

    self_in_pool = [
        qid for qid in queries
        if qid in candidates
    ]

    print(f"\n{split.upper()} SELF-RETRIEVAL")
    print("Query IDs:", len(queries))
    print(
        "Query IDs present in candidate dictionary:",
        len(self_in_pool)
    )

    if split in ["train", "dev"]:
        self_relevant = [
            qid for qid, item in queries.items()
            if qid in item["relevant_candidates"]
        ]

        print(
            "Queries with themselves as relevant:",
            len(self_relevant)
        )


# ==================================================
# 2. TOKEN LENGTH AUDIT
# ==================================================

def measure_token_lengths(records, label):

    lengths = []

    for case_id, item in records.items():

        full_text = "\n".join(
            str(sentence) for sentence in item["text"]
        )

        token_count = len(tokenizer.encode(full_text).ids)
        lengths.append(token_count)

    lengths = np.array(lengths)

    print(f"\n{label}")
    print("-" * 50)
    print("Documents:", len(lengths))
    print("Minimum tokens:", int(lengths.min()))
    print("Median tokens:", int(np.median(lengths)))
    print("Mean tokens:", round(float(lengths.mean()), 2))
    print("90th percentile:", int(np.percentile(lengths, 90)))
    print("95th percentile:", int(np.percentile(lengths, 95)))
    print("99th percentile:", int(np.percentile(lengths, 99)))
    print("Maximum tokens:", int(lengths.max()))

    for limit in [512, 1024, 2048, 4096, 8192]:
        count = int(np.sum(lengths > limit))
        print(
            f"Documents exceeding {limit:5d} tokens: "
            f"{count:5d} ({count / len(lengths) * 100:.2f}%)"
        )

    return lengths


token_lengths = {}

for split in ["train", "dev", "test"]:

    token_lengths[f"{split}_queries"] = measure_token_lengths(
        data[split],
        f"{split.upper()} QUERIES"
    )

    token_lengths[f"{split}_candidates"] = measure_token_lengths(
        data[f"{split}_candidates"],
        f"{split.upper()} CANDIDATES"
    )

print("\nTOKEN LENGTH AUDIT COMPLETED")


Tokenizer vocabulary: 50000

Special token IDs:
[PAD] 0
[UNK] 1
[BOS] 2
[EOS] 3
[MASK] 4

TRAIN SELF-RETRIEVAL
Query IDs: 827
Query IDs present in candidate dictionary: 827
Queries with themselves as relevant: 0

DEV SELF-RETRIEVAL
Query IDs: 118
Query IDs present in candidate dictionary: 118
Queries with themselves as relevant: 0

TEST SELF-RETRIEVAL
Query IDs: 237
Query IDs present in candidate dictionary: 237

TRAIN QUERIES
--------------------------------------------------
Documents: 827
Minimum tokens: 177
Median tokens: 5701
Mean tokens: 8607.54
90th percentile: 15812
95th percentile: 25760
99th percentile: 55915
Maximum tokens: 149015
Documents exceeding   512 tokens:   821 (99.27%)
Documents exceeding  1024 tokens:   797 (96.37%)
Documents exceeding  2048 tokens:   738 (89.24%)
Documents exceeding  4096 tokens:   552 (66.75%)
Documents exceeding  8192 tokens:   273 (33.01%)

TRAIN CANDIDATES
--------------------------------------------------
Documents: 4320
Minimum tokens: 8
Me

In [80]:

MAX_SEQ_LEN = 1024
CONTENT_LIMIT = MAX_SEQ_LEN - 2

BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

def chunk_document(sentences, tokenizer):

    chunks = []
    current_chunk = []

    for sentence in sentences:

        token_ids = tokenizer.encode(str(sentence)).ids

        # Split sentences that exceed the content limit
        while len(token_ids) > CONTENT_LIMIT:

            if current_chunk:
                chunks.append(
                    [BOS_ID] + current_chunk + [EOS_ID]
                )
                current_chunk = []

            chunks.append(
                [BOS_ID]
                + token_ids[:CONTENT_LIMIT]
                + [EOS_ID]
            )

            token_ids = token_ids[CONTENT_LIMIT:]

        # Start another chunk if this sentence does not fit
        if len(current_chunk) + len(token_ids) > CONTENT_LIMIT:

            if current_chunk:
                chunks.append(
                    [BOS_ID] + current_chunk + [EOS_ID]
                )

            current_chunk = []

        current_chunk.extend(token_ids)

    # Save the final chunk
    if current_chunk:
        chunks.append(
            [BOS_ID] + current_chunk + [EOS_ID]
        )

    return chunks


In [81]:

# Select one training query
sample_id = next(iter(data["train"]))

sample_sentences = data["train"][sample_id]["text"]

sample_chunks = chunk_document(
    sample_sentences,
    tokenizer
)

print("Sample case ID:", sample_id)
print("Number of sentences:", len(sample_sentences))
print("Number of chunks:", len(sample_chunks))

print("\nFirst 10 chunk lengths:")
print([len(chunk) for chunk in sample_chunks[:10]])

print("\nMinimum chunk length:", min(map(len, sample_chunks)))
print("Maximum chunk length:", max(map(len, sample_chunks)))

# Validate chunk boundaries
assert all(
    chunk[0] == BOS_ID and chunk[-1] == EOS_ID
    for chunk in sample_chunks
)

# Validate context window
assert all(
    len(chunk) <= MAX_SEQ_LEN
    for chunk in sample_chunks
)

print("\nChunking validation: PASSED")


Sample case ID: 0000000599
Number of sentences: 54
Number of chunks: 8

First 10 chunk lengths:
[981, 947, 892, 942, 782, 951, 868, 839]

Minimum chunk length: 782
Maximum chunk length: 981

Chunking validation: PASSED


In [82]:
import os

print("Searching Kaggle input directory...\n")

checkpoint_matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file == "a4_step_22000.pt":
            full_path = os.path.join(root, file)
            checkpoint_matches.append(full_path)
            print("FOUND:", full_path)

print("\nTotal matches:", len(checkpoint_matches))

Searching Kaggle input directory...

FOUND: /kaggle/input/datasets/aq1de3/a4-step-220000/a4_step_22000.pt

Total matches: 1


In [83]:
CHECKPOINT_PATH = checkpoint_matches[0]

print("Checkpoint:", CHECKPOINT_PATH)
print("Exists:", os.path.exists(CHECKPOINT_PATH))
print("Size (GB):", round(
    os.path.getsize(CHECKPOINT_PATH) / (1024**3), 3
))

Checkpoint: /kaggle/input/datasets/aq1de3/a4-step-220000/a4_step_22000.pt
Exists: True
Size (GB): 0.359


In [5]:

import torch
import torch.nn as nn

# A4 architecture configuration
VOCAB_SIZE = 50000
HIDDEN_SIZE = 512
MAX_POSITION = 1024
NUM_LAYERS = 6
NUM_HEADS = 8
FFN_SIZE = 2048


class A4Attention(nn.Module):

    def __init__(self, hidden_size, num_heads):
        super().__init__()

        self.q_proj = nn.Linear(hidden_size, hidden_size)
        self.k_proj = nn.Linear(hidden_size, hidden_size)
        self.v_proj = nn.Linear(hidden_size, hidden_size)
        self.out_proj = nn.Linear(hidden_size, hidden_size)

        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

    def forward(self, query, key, value, mask=None):

        B, Tq, C = query.shape
        Tk = key.shape[1]

        q = self.q_proj(query).view(
            B, Tq, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = self.k_proj(key).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        v = self.v_proj(value).view(
            B, Tk, self.num_heads, self.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            q, k.transpose(-2, -1)
        ) / (self.head_dim ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attention = torch.softmax(scores, dim=-1)

        output = torch.matmul(attention, v)

        output = output.transpose(
            1, 2
        ).contiguous().view(B, Tq, C)

        return self.out_proj(output)


class A4FFNCheckpoint(nn.Module):

    def __init__(self, hidden_size, ffn_size):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(hidden_size, ffn_size),
            nn.GELU(),
            nn.Linear(ffn_size, hidden_size)
        )

    def forward(self, x):
        return self.net(x)


class A4EncoderLayer(nn.Module):

    def __init__(self, hidden_size, num_heads, ffn_size):
        super().__init__()

        self.norm1 = nn.LayerNorm(hidden_size)
        self.attn = A4Attention(hidden_size, num_heads)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.ffn = A4FFNCheckpoint(hidden_size, ffn_size)

    def forward(self, x, mask=None):

        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)
        x = x + self.ffn(self.norm2(x))

        return x


class A4EncoderForNER(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE, HIDDEN_SIZE
        )

        self.encoder_pos = nn.Embedding(
            MAX_POSITION, HIDDEN_SIZE
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(
                HIDDEN_SIZE,
                NUM_HEADS,
                FFN_SIZE
            )
            for _ in range(NUM_LAYERS)
        ])

    def forward(self, input_ids, attention_mask=None):

        B, T = input_ids.shape

        if T > MAX_POSITION:
            raise ValueError("Sequence length exceeds 1024")

        positions = torch.arange(
            T, device=input_ids.device
        )

        x = self.token_embedding(input_ids)
        x = x + self.encoder_pos(positions).unsqueeze(0)

        for layer in self.encoder_layers:
            x = layer(x, mask=None)

        return x


print("A4 encoder architecture defined successfully!")


A4 encoder architecture defined successfully!


In [85]:
import torch

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state = checkpoint.get("model_state_dict", checkpoint)

encoder = A4EncoderForNER()

encoder_state = {
    key: value
    for key, value in state.items()
    if key.startswith((
        "token_embedding.",
        "encoder_pos.",
        "encoder_layers."
    ))
}

load_result = encoder.load_state_dict(
    encoder_state,
    strict=True
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

encoder = encoder.to(device)
encoder.eval()

print("Strict loading:", load_result)
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("A4 encoder ready!")

Strict loading: <All keys matched successfully>
Device: cuda
GPU: Tesla T4
A4 encoder ready!


In [86]:

import torch
import torch.nn.functional as F

def encode_document_chunks(chunks, encoder, device):

    chunk_embeddings = []

    encoder.eval()

    with torch.inference_mode():

        for i, chunk in enumerate(chunks):

            input_ids = torch.tensor(
                [chunk],
                dtype=torch.long,
                device=device
            )

            hidden_states = encoder(input_ids)

            # Exclude BOS and EOS
            content_states = hidden_states[:, 1:-1, :]

            # Mean pooling
            chunk_embedding = content_states.mean(dim=1)

            chunk_embeddings.append(
                chunk_embedding.squeeze(0).cpu()
            )

    chunk_embeddings = torch.stack(chunk_embeddings)

    # Mean aggregation across chunks
    document_embedding = chunk_embeddings.mean(dim=0)

    # L2 normalization
    document_embedding = F.normalize(
        document_embedding.unsqueeze(0),
        p=2,
        dim=1
    ).squeeze(0)

    return document_embedding, chunk_embeddings


# Extract embedding for sample case
sample_embedding, sample_chunk_embeddings = encode_document_chunks(
    sample_chunks,
    encoder,
    device
)

print("Case ID:", sample_id)
print("Chunk embeddings:", tuple(sample_chunk_embeddings.shape))
print("Document embedding:", tuple(sample_embedding.shape))
print("Finite:", torch.isfinite(sample_embedding).all().item())
print("Embedding norm:", sample_embedding.norm().item())

assert sample_embedding.shape == (512,)
assert sample_chunk_embeddings.shape == (8, 512)
assert torch.isfinite(sample_embedding).all()
assert abs(sample_embedding.norm().item() - 1.0) < 1e-5

print("\nDOCUMENT EMBEDDING TEST: PASSED")


Case ID: 0000000599
Chunk embeddings: (8, 512)
Document embedding: (512,)
Finite: True
Embedding norm: 0.9999999403953552

DOCUMENT EMBEDDING TEST: PASSED


In [87]:

def forward(self, input_ids, attention_mask=None):

    B, T = input_ids.shape

    if T > MAX_POSITION:
        raise ValueError("Sequence length exceeds 1024")

    positions = torch.arange(T, device=input_ids.device)

    x = self.token_embedding(input_ids)
    x = x + self.encoder_pos(positions).unsqueeze(0)

    # True means the position is padding and must not be attended to.
    padding_mask = None

    if attention_mask is not None:
        padding_mask = (~attention_mask.bool())[:, None, None, :]

    for layer in self.encoder_layers:
        x = layer(x, mask=padding_mask)

    return x


In [88]:

from torch.nn.utils.rnn import pad_sequence

# Use three real chunks from the sample case
test_chunks = sample_chunks[:3]

chunk_tensors = [
    torch.tensor(chunk, dtype=torch.long)
    for chunk in test_chunks
]

# Pad only to the longest chunk in this batch
padded = pad_sequence(
    chunk_tensors,
    batch_first=True,
    padding_value=tokenizer.token_to_id("[PAD]")
).to(device)

attention_mask = (
    padded != tokenizer.token_to_id("[PAD]")
)

encoder.eval()

with torch.inference_mode():
    batch_output = encoder(
        padded,
        attention_mask=attention_mask
    )

print("Input shape:", tuple(padded.shape))
print("Output shape:", tuple(batch_output.shape))
print("Finite:", torch.isfinite(batch_output).all().item())

assert batch_output.shape == (
    3, padded.shape[1], 512
)
assert torch.isfinite(batch_output).all()

print("BATCHED ENCODER TEST: PASSED")


Input shape: (3, 981)
Output shape: (3, 981, 512)
Finite: True
BATCHED ENCODER TEST: PASSED


In [89]:

import torch

encoder.eval()

# First chunk encoded individually
single_ids = torch.tensor(
    [sample_chunks[0]],
    dtype=torch.long,
    device=device
)

with torch.inference_mode():
    single_output = encoder(single_ids)

# Same chunk placed in a batch with a shorter chunk
test_chunks = [
    sample_chunks[0],
    sample_chunks[-1][:700]
]

test_tensors = [
    torch.tensor(c, dtype=torch.long)
    for c in test_chunks
]

padded = torch.nn.utils.rnn.pad_sequence(
    test_tensors,
    batch_first=True,
    padding_value=tokenizer.token_to_id("[PAD]")
).to(device)

attention_mask = (
    padded != tokenizer.token_to_id("[PAD]")
)

with torch.inference_mode():
    batch_output = encoder(
        padded,
        attention_mask=attention_mask
    )

# Compare only the real token positions
valid_length = len(sample_chunks[0])

difference = (
    single_output[0] -
    batch_output[0, :valid_length]
).abs().max().item()

print("Maximum absolute difference:", difference)

assert difference < 1e-4, "Padding changed real-token representations!"

print("PADDING MASK VALIDATION: PASSED")


Maximum absolute difference: 3.337860107421875e-06
PADDING MASK VALIDATION: PASSED


In [90]:

import os
import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

BATCH_SIZE = 4
SAVE_EVERY = 25

OUTPUT_DIR = "/kaggle/working/phase7_embeddings"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAD_ID = tokenizer.token_to_id("[PAD]")

encoder.eval()
encoder.to(device)


# --------------------------------------------------
# ENCODE ONE DOCUMENT
# --------------------------------------------------

def encode_document(sentences):

    chunks = chunk_document(sentences, tokenizer)

    # Sort by length to reduce padding waste
    chunks = sorted(chunks, key=len)

    chunk_embeddings = []

    with torch.inference_mode():

        for start in range(0, len(chunks), BATCH_SIZE):

            batch_chunks = chunks[start:start + BATCH_SIZE]

            tensors = [
                torch.tensor(c, dtype=torch.long)
                for c in batch_chunks
            ]

            padded = pad_sequence(
                tensors,
                batch_first=True,
                padding_value=PAD_ID
            ).to(device)

            attention_mask = padded.ne(PAD_ID)

            hidden = encoder(
                padded,
                attention_mask=attention_mask
            )

            # Build a mask that excludes:
            # BOS, EOS and PAD tokens
            content_mask = attention_mask.clone()

            content_mask[:, 0] = False

            lengths = attention_mask.sum(dim=1)

            for i, length in enumerate(lengths):
                content_mask[i, length.item() - 1] = False

            # Mean-pool only valid content tokens
            weights = content_mask.unsqueeze(-1).to(hidden.dtype)

            pooled = (hidden * weights).sum(dim=1)

            counts = weights.sum(dim=1).clamp(min=1)

            pooled = pooled / counts

            chunk_embeddings.append(pooled.cpu())

    chunk_embeddings = torch.cat(chunk_embeddings, dim=0)

    # Aggregate chunk vectors into one document vector
    document_embedding = chunk_embeddings.mean(dim=0)

    document_embedding = F.normalize(
        document_embedding.unsqueeze(0),
        p=2,
        dim=1
    ).squeeze(0)

    return document_embedding


# --------------------------------------------------
# PROCESS ONE COLLECTION WITH RESUME SUPPORT
# --------------------------------------------------

def process_collection(records, collection_name):

    save_path = os.path.join(
        OUTPUT_DIR,
        f"{collection_name}.pt"
    )

    # Resume previously completed documents
    if os.path.exists(save_path):
        embeddings = torch.load(
            save_path,
            map_location="cpu",
            weights_only=False
        )
        print(f"Resuming {collection_name}: {len(embeddings)} done")
    else:
        embeddings = {}

    total = len(records)

    for index, (case_id, item) in enumerate(records.items(), start=1):

        if case_id in embeddings:
            continue

        embedding = encode_document(item["text"])

        embeddings[case_id] = embedding

        # Save progress periodically
        if len(embeddings) % SAVE_EVERY == 0:
            torch.save(embeddings, save_path)

        if index % 25 == 0 or index == total:
            print(
                f"{collection_name}: "
                f"{index}/{total} | "
                f"Saved: {len(embeddings)}/{total}"
            )

    # Final save
    torch.save(embeddings, save_path)

    print(f"\nCompleted: {collection_name}")
    print("Documents:", len(embeddings))
    print("Embedding dimension:", next(iter(embeddings.values())).shape)
    print("Saved to:", save_path)

    return embeddings


print("Embedding pipeline ready.")
print("GPU:", torch.cuda.get_device_name(0))
print("Batch size:", BATCH_SIZE)
print("Output directory:", OUTPUT_DIR)


Embedding pipeline ready.
GPU: Tesla T4
Batch size: 4
Output directory: /kaggle/working/phase7_embeddings


In [91]:

train_query_embeddings = process_collection(
    data["train"],
    "train_queries"
)


Resuming train_queries: 827 done

Completed: train_queries
Documents: 827
Embedding dimension: torch.Size([512])
Saved to: /kaggle/working/phase7_embeddings/train_queries.pt


In [92]:
import os
import torch

path = os.path.join(
    OUTPUT_DIR,
    "train_queries.pt"
)

saved = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Saved documents:", len(saved))
print("Embedding shape:", next(iter(saved.values())).shape)
print("All finite:", all(
    torch.isfinite(v).all().item()
    for v in saved.values()
))

assert len(saved) == 827
assert all(v.shape == (512,) for v in saved.values())

print("TRAIN QUERY EMBEDDINGS VERIFIED")

Saved documents: 827
Embedding shape: torch.Size([512])
All finite: True
TRAIN QUERY EMBEDDINGS VERIFIED


In [93]:
train_candidate_embeddings = process_collection(
    data["train_candidates"],
    "train_candidates"
)

Resuming train_candidates: 4320 done

Completed: train_candidates
Documents: 4320
Embedding dimension: torch.Size([512])
Saved to: /kaggle/working/phase7_embeddings/train_candidates.pt


In [94]:
path = os.path.join(
    OUTPUT_DIR,
    "train_candidates.pt"
)

saved = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Saved candidates:", len(saved))
print("Embedding shape:", next(iter(saved.values())).shape)

print("All finite:", all(
    torch.isfinite(v).all().item()
    for v in saved.values()
))

assert len(saved) == 4320
assert all(v.shape == (512,) for v in saved.values())

print("TRAIN CANDIDATE EMBEDDINGS VERIFIED")

Saved candidates: 4320
Embedding shape: torch.Size([512])
All finite: True
TRAIN CANDIDATE EMBEDDINGS VERIFIED


In [95]:

dev_query_embeddings = process_collection(
    data["dev"],
    "dev_queries"
)


Resuming dev_queries: 118 done

Completed: dev_queries
Documents: 118
Embedding dimension: torch.Size([512])
Saved to: /kaggle/working/phase7_embeddings/dev_queries.pt


In [96]:

path = os.path.join(
    OUTPUT_DIR,
    "dev_queries.pt"
)

saved = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Saved queries:", len(saved))
print("Embedding shape:", next(iter(saved.values())).shape)

print("All finite:", all(
    torch.isfinite(v).all().item()
    for v in saved.values()
))

assert len(saved) == 118
assert all(v.shape == (512,) for v in saved.values())

print("DEV QUERY EMBEDDINGS VERIFIED")


Saved queries: 118
Embedding shape: torch.Size([512])
All finite: True
DEV QUERY EMBEDDINGS VERIFIED


In [97]:

dev_candidate_embeddings = process_collection(
    data["dev_candidates"],
    "dev_candidates"
)


Resuming dev_candidates: 1023 done

Completed: dev_candidates
Documents: 1023
Embedding dimension: torch.Size([512])
Saved to: /kaggle/working/phase7_embeddings/dev_candidates.pt


In [98]:

path = os.path.join(
    OUTPUT_DIR,
    "dev_candidates.pt"
)

saved = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Saved candidates:", len(saved))
print("Embedding shape:", next(iter(saved.values())).shape)

print("All finite:", all(
    torch.isfinite(v).all().item()
    for v in saved.values()
))

assert len(saved) == 1023
assert all(v.shape == (512,) for v in saved.values())

print("DEV CANDIDATE EMBEDDINGS VERIFIED")


Saved candidates: 1023
Embedding shape: torch.Size([512])
All finite: True
DEV CANDIDATE EMBEDDINGS VERIFIED


In [99]:

test_query_embeddings = process_collection(
    data["test"],
    "test_queries"
)


Resuming test_queries: 237 done

Completed: test_queries
Documents: 237
Embedding dimension: torch.Size([512])
Saved to: /kaggle/working/phase7_embeddings/test_queries.pt


In [100]:

path = os.path.join(
    OUTPUT_DIR,
    "test_queries.pt"
)

saved = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Saved queries:", len(saved))
print("Embedding shape:", next(iter(saved.values())).shape)

print("All finite:", all(
    torch.isfinite(v).all().item()
    for v in saved.values()
))

assert len(saved) == 237
assert all(v.shape == (512,) for v in saved.values())

print("TEST QUERY EMBEDDINGS VERIFIED")


Saved queries: 237
Embedding shape: torch.Size([512])
All finite: True
TEST QUERY EMBEDDINGS VERIFIED


In [101]:

test_candidate_embeddings = process_collection(
    data["test_candidates"],
    "test_candidates"
)


Resuming test_candidates: 1727 done

Completed: test_candidates
Documents: 1727
Embedding dimension: torch.Size([512])
Saved to: /kaggle/working/phase7_embeddings/test_candidates.pt


In [102]:

path = os.path.join(
    OUTPUT_DIR,
    "test_candidates.pt"
)

saved = torch.load(
    path,
    map_location="cpu",
    weights_only=False
)

print("Saved candidates:", len(saved))
print("Embedding shape:", next(iter(saved.values())).shape)

print("All finite:", all(
    torch.isfinite(v).all().item()
    for v in saved.values()
))

assert len(saved) == 1727
assert all(v.shape == (512,) for v in saved.values())

print("TEST CANDIDATE EMBEDDINGS VERIFIED")


Saved candidates: 1727
Embedding shape: torch.Size([512])
All finite: True
TEST CANDIDATE EMBEDDINGS VERIFIED


In [103]:

import torch
import torch.nn.functional as F

# Candidate IDs and embedding matrix
candidate_ids = list(dev_candidate_embeddings.keys())

candidate_matrix = torch.stack([
    dev_candidate_embeddings[cid]
    for cid in candidate_ids
]).float()

candidate_matrix = F.normalize(candidate_matrix, p=2, dim=1)

# Query embeddings
query_ids = list(dev_query_embeddings.keys())

query_matrix = torch.stack([
    dev_query_embeddings[qid]
    for qid in query_ids
]).float()

query_matrix = F.normalize(query_matrix, p=2, dim=1)

# Store top-7 predictions
dev_predictions = {}

for i, qid in enumerate(query_ids):

    # Cosine similarity
    scores = torch.matmul(
        candidate_matrix,
        query_matrix[i]
    )

    # Exclude query's own ID
    valid_indices = [
        j for j, cid in enumerate(candidate_ids)
        if cid != qid
    ]

    valid_scores = scores[valid_indices]

    k = min(7, len(valid_indices))

    top_positions = torch.topk(
        valid_scores,
        k=k
    ).indices.tolist()

    selected_indices = [
        valid_indices[pos]
        for pos in top_positions
    ]

    dev_predictions[qid] = [
        candidate_ids[j]
        for j in selected_indices
    ]

print("Queries ranked:", len(dev_predictions))
print("Candidates per query:",
      len(next(iter(dev_predictions.values()))))
print("Example query:", query_ids[0])
print("Top-7:", dev_predictions[query_ids[0]])

assert len(dev_predictions) == 118
assert all(len(v) == 7 for v in dev_predictions.values())
assert all(
    len(set(v)) == 7 for v in dev_predictions.values()
)

print("\nDEV RANKING COMPLETED")


Queries ranked: 118
Candidates per query: 7
Example query: 0000012637
Top-7: ['0001270021', '0000832348', '0000923991', '0000196752', '0001627616', '0000053024', '0000881663']

DEV RANKING COMPLETED


In [104]:

# Micro-level metric counts
total_tp = 0
total_fp = 0
total_fn = 0

for qid, predictions in dev_predictions.items():

    gold = set(data["dev"][qid]["relevant_candidates"])
    predicted = set(predictions)

    # Safety: query itself cannot be retrieved
    assert qid not in predicted

    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    total_tp += tp
    total_fp += fp
    total_fn += fn

micro_precision = (
    total_tp / (total_tp + total_fp)
    if total_tp + total_fp else 0.0
)

micro_recall = (
    total_tp / (total_tp + total_fn)
    if total_tp + total_fn else 0.0
)

micro_f1 = (
    2 * micro_precision * micro_recall /
    (micro_precision + micro_recall)
    if micro_precision + micro_recall else 0.0
)

print("=" * 45)
print("A4 FROZEN ENCODER - DEV RESULTS")
print("=" * 45)

print("Queries:", len(dev_predictions))
print("Total TP:", total_tp)
print("Total FP:", total_fp)
print("Total FN:", total_fn)

print(f"Micro-Precision@7: {micro_precision:.6f}")
print(f"Micro-Recall@7:    {micro_recall:.6f}")
print(f"Micro-F1@7:        {micro_f1:.6f}")

print("=" * 45)


A4 FROZEN ENCODER - DEV RESULTS
Queries: 118
Total TP: 60
Total FP: 766
Total FN: 913
Micro-Precision@7: 0.072639
Micro-Recall@7:    0.061665
Micro-F1@7:        0.066704


In [105]:

import json
import os

baseline_path = os.path.join(
    "/kaggle/working",
    "phase7_exp1_dev_predictions.json"
)

with open(baseline_path, "w") as f:
    json.dump(dev_predictions, f, indent=2)

print("Baseline predictions saved:", baseline_path)


Baseline predictions saved: /kaggle/working/phase7_exp1_dev_predictions.json


In [106]:
total_tp = 0
total_fp = 0
total_fn = 0

for qid, predictions in dev_predictions.items():

    gold = set(data["dev"][qid]["relevant_candidates"])
    predicted = set(predictions)

    total_tp += len(predicted & gold)
    total_fp += len(predicted - gold)
    total_fn += len(gold - predicted)

precision = total_tp / (total_tp + total_fp)
recall = total_tp / (total_tp + total_fn)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("TP:", total_tp)
print("FP:", total_fp)
print("FN:", total_fn)
print(f"Micro-Precision@7: {precision:.6f}")
print(f"Micro-Recall@7: {recall:.6f}")
print(f"Micro-F1@7: {f1:.6f}")

TP: 60
FP: 766
FN: 913
Micro-Precision@7: 0.072639
Micro-Recall@7: 0.061665
Micro-F1@7: 0.066704


In [107]:
total_tp = 0
total_fp = 0
total_fn = 0

for qid, predictions in dev_predictions.items():

    gold = set(data["dev"][qid]["relevant_candidates"])
    predicted = set(predictions)

    total_tp += len(predicted & gold)
    total_fp += len(predicted - gold)
    total_fn += len(gold - predicted)

precision = total_tp / (total_tp + total_fp)
recall = total_tp / (total_tp + total_fn)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("TP:", total_tp)
print("FP:", total_fp)
print("FN:", total_fn)
print(f"Micro-Precision@7: {precision:.6f}")
print(f"Micro-Recall@7: {recall:.6f}")
print(f"Micro-F1@7: {f1:.6f}")

TP: 60
FP: 766
FN: 913
Micro-Precision@7: 0.072639
Micro-Recall@7: 0.061665
Micro-F1@7: 0.066704


In [108]:

import numpy as np
import torch
import torch.nn.functional as F

candidate_ids = list(dev_candidate_embeddings.keys())

candidate_matrix = torch.stack([
    dev_candidate_embeddings[cid]
    for cid in candidate_ids
]).float()

candidate_matrix = F.normalize(candidate_matrix, p=2, dim=1)

query_ids = list(dev_query_embeddings.keys())

query_matrix = torch.stack([
    dev_query_embeddings[qid]
    for qid in query_ids
]).float()

query_matrix = F.normalize(query_matrix, p=2, dim=1)

rank_results = []

for i, qid in enumerate(query_ids):

    scores = candidate_matrix @ query_matrix[i]

    valid_indices = [
        j for j, cid in enumerate(candidate_ids)
        if cid != qid
    ]

    ordered = sorted(
        valid_indices,
        key=lambda j: scores[j].item(),
        reverse=True
    )

    gold = set(data["dev"][qid]["relevant_candidates"])

    # Rank of every relevant candidate (1-based)
    gold_ranks = [
        rank
        for rank, j in enumerate(ordered, start=1)
        if candidate_ids[j] in gold
    ]

    rank_results.append({
        "query_id": qid,
        "gold_count": len(gold),
        "best_gold_rank": min(gold_ranks) if gold_ranks else None,
        "gold_ranks": gold_ranks
    })


# Recall at different ranking cutoffs
print("RELEVANT CANDIDATE RANK ANALYSIS")
print("-" * 45)

for k in [1, 5, 7, 10, 20, 50, 100, 500]:

    retrieved = sum(
        sum(rank <= k for rank in item["gold_ranks"])
        for item in rank_results
    )

    total_gold = sum(item["gold_count"] for item in rank_results)

    recall = retrieved / total_gold if total_gold else 0

    print(f"Recall@{k:3d}: {recall:.6f}")


best_ranks = [
    item["best_gold_rank"]
    for item in rank_results
    if item["best_gold_rank"] is not None
]

print("\nQueries with at least one relevant candidate:",
      len(best_ranks))

print("Median best relevant rank:", np.median(best_ranks))
print("Mean best relevant rank:", np.mean(best_ranks))
print("Best relevant rank <= 7:",
      sum(r <= 7 for r in best_ranks))
print("Best relevant rank > 100:",
      sum(r > 100 for r in best_ranks))


RELEVANT CANDIDATE RANK ANALYSIS
---------------------------------------------
Recall@  1: 0.016444
Recall@  5: 0.054471
Recall@  7: 0.061665
Recall@ 10: 0.073998
Recall@ 20: 0.112025
Recall@ 50: 0.176773
Recall@100: 0.266187
Recall@500: 0.588900

Queries with at least one relevant candidate: 118
Median best relevant rank: 29.5
Mean best relevant rank: 127.98305084745763
Best relevant rank <= 7: 40
Best relevant rank > 100: 36


In [109]:
# Display 15 queries with the worst best-relevant ranks

ranked_queries = sorted(
    rank_results,
    key=lambda x: (
        x["best_gold_rank"]
        if x["best_gold_rank"] is not None
        else float("inf")
    ),
    reverse=True
)

for item in ranked_queries[:15]:
    print(
        "Query:", item["query_id"],
        "| Relevant:", item["gold_count"],
        "| Best rank:", item["best_gold_rank"]
    )

Query: 0000477921 | Relevant: 1 | Best rank: 1011
Query: 0001722031 | Relevant: 1 | Best rank: 976
Query: 0000790482 | Relevant: 1 | Best rank: 858
Query: 0000688536 | Relevant: 4 | Best rank: 838
Query: 0001879785 | Relevant: 1 | Best rank: 729
Query: 0000917632 | Relevant: 1 | Best rank: 712
Query: 0000867830 | Relevant: 2 | Best rank: 645
Query: 0000161423 | Relevant: 1 | Best rank: 610
Query: 0001063842 | Relevant: 2 | Best rank: 588
Query: 0001481214 | Relevant: 6 | Best rank: 516
Query: 0000469517 | Relevant: 5 | Best rank: 445
Query: 0001925650 | Relevant: 5 | Best rank: 423
Query: 0001247083 | Relevant: 2 | Best rank: 412
Query: 0000887203 | Relevant: 5 | Best rank: 301
Query: 0001207635 | Relevant: 5 | Best rank: 294


In [110]:

# Inspect the worst-ranked query
target_qid = "0000477921"

# Get its relevance information
gold_ids = set(
    data["dev"][target_qid]["relevant_candidates"]
)

# Find its ranking
result = next(
    item for item in rank_results
    if item["query_id"] == target_qid
)

# Calculate similarity scores
scores = candidate_matrix @ query_matrix[
    query_ids.index(target_qid)
]

valid_indices = [
    j for j, cid in enumerate(candidate_ids)
    if cid != target_qid
]

ordered = sorted(
    valid_indices,
    key=lambda j: scores[j].item(),
    reverse=True
)

# Print query
print("=" * 65)
print("QUERY CASE:", target_qid)
print("=" * 65)

query_text = data["dev"][target_qid]["text"]

print("\nQUERY TEXT (first 5 sentences):")
for sentence in query_text[:5]:
    print(str(sentence)[:500])

print("\nTotal query sentences:", len(query_text))

# Show top 5 retrieved candidates
print("\n" + "=" * 65)
print("TOP 5 RETRIEVED CANDIDATES")
print("=" * 65)

for rank, idx in enumerate(ordered[:5], start=1):

    cid = candidate_ids[idx]
    text = data["dev_candidates"][cid]["text"]

    print(f"\nRank: {rank}")
    print("Candidate ID:", cid)
    print("Similarity:", round(scores[idx].item(), 6))
    print("Relevant:", cid in gold_ids)

    for sentence in text[:2]:
        print(str(sentence)[:350])

# Show the actual relevant candidate
print("\n" + "=" * 65)
print("GOLD RELEVANT CANDIDATES")
print("=" * 65)

for cid in gold_ids:

    text = data["dev_candidates"][cid]["text"]

    idx = candidate_ids.index(cid)
    rank = ordered.index(idx) + 1

    print("\nCandidate ID:", cid)
    print("Actual rank:", rank)
    print("Similarity:", round(scores[idx].item(), 6))

    for sentence in text[:5]:
        print(str(sentence)[:500])


QUERY CASE: 0000477921

QUERY TEXT (first 5 sentences):
<NAME>, J. 
1. Now that we have had the document of 27th August 1897 translated, I think that the appellants being the tangals are clearly entitled to the money which represents the share assigned to the mosque by <NAME>. By that document he gave the tangals power to collect his share of the estate of <NAME>. They would, I apprehend, have been entitled to demand from the Collector the whole of <NAME>'s share in the particular fund, and having collected it to keep a third for the mosque. In my opinion this i
<NAME>, <NAME> 
2. In this case one <NAME> is said to have held a kanom (usufructuary mortgage) over certain land for Rs. 100. On his death his rights as kanomdar are said to have passed to hits widow, his half-sister and his father's brother's son's son, <NAME>, as sharers under <ORG>, This <NAME> assigned one-third of his share by a registered deed of gift to the seventh and eighth defendants in the suit. The land was taken u

In [111]:

import numpy as np

# Similarity distribution for the selected query
q_index = query_ids.index("0000477921")
scores = candidate_matrix @ query_matrix[q_index]

scores_np = scores.cpu().numpy()

print("Similarity distribution")
print("Minimum:", np.min(scores_np))
print("Maximum:", np.max(scores_np))
print("Mean:", np.mean(scores_np))
print("Median:", np.median(scores_np))
print("Standard deviation:", np.std(scores_np))

print("\nSelected percentiles")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"{p}th percentile:", np.percentile(scores_np, p))

print("\nGold candidate similarity:")
gold_id = "0000057885"
gold_index = candidate_ids.index(gold_id)
print(scores[gold_index].item())


Similarity distribution
Minimum: -0.23349985
Maximum: 1.0000001
Mean: 0.8896878
Median: 0.932695
Standard deviation: 0.13664886

Selected percentiles
1th percentile: 0.1017249
5th percentile: 0.67291534
10th percentile: 0.8110112
25th percentile: 0.89117277
50th percentile: 0.932695
75th percentile: 0.9474657
90th percentile: 0.9554324
95th percentile: 0.9596439
99th percentile: 0.96651095

Gold candidate similarity:
0.14027529954910278


In [112]:

import torch
import torch.nn.functional as F

# Check query and candidate embedding norms
query_norms = torch.linalg.vector_norm(query_matrix, dim=1)
candidate_norms = torch.linalg.vector_norm(candidate_matrix, dim=1)

print("QUERY EMBEDDING NORMS")
print("Minimum:", query_norms.min().item())
print("Maximum:", query_norms.max().item())
print("Mean:", query_norms.mean().item())

print("\nCANDIDATE EMBEDDING NORMS")
print("Minimum:", candidate_norms.min().item())
print("Maximum:", candidate_norms.max().item())
print("Mean:", candidate_norms.mean().item())

# Verify the gold pair using the correct dictionary names
qid = "0000477921"
cid = "0000057885"

q = dev_query_embeddings[qid]
c = dev_candidate_embeddings[cid]

print("\nGOLD PAIR SIMILARITY CHECK")
print("Dot product:", torch.dot(q, c).item())

print(
    "Cosine similarity:",
    F.cosine_similarity(q.unsqueeze(0), c.unsqueeze(0)).item()
)


QUERY EMBEDDING NORMS
Minimum: 0.9999999403953552
Maximum: 1.0000001192092896
Mean: 1.0

CANDIDATE EMBEDDING NORMS
Minimum: 0.9999998807907104
Maximum: 1.0000001192092896
Mean: 1.0

GOLD PAIR SIMILARITY CHECK
Dot product: 0.1402752846479416
Cosine similarity: 0.14027531445026398


In [113]:

qid = "0000477921"
gold_id = "0000057885"

top_ids = [
    "0001733204",
    "0001827392",
    "0000046064",
    "0001652416",
    "0001113945"
]

def document_stats(case_id, collection):
    sentences = collection[case_id]["text"]
    token_count = sum(
        len(tokenizer.encode(str(sentence)).ids)
        for sentence in sentences
    )
    return len(sentences), token_count

print("DOCUMENT LENGTH COMPARISON")
print("-" * 65)

items = [
    ("Query", qid, data["dev"]),
    ("Gold candidate", gold_id, data["dev_candidates"])
]

for i, cid in enumerate(top_ids, start=1):
    items.append((
        f"Retrieved rank {i}",
        cid,
        data["dev_candidates"]
    ))

for label, cid, collection in items:
    sentence_count, token_count = document_stats(cid, collection)

    print(
        f"{label:20s} | ID: {cid} | "
        f"Sentences: {sentence_count:5d} | "
        f"Tokens: {token_count:8d}"
    )


DOCUMENT LENGTH COMPARISON
-----------------------------------------------------------------
Query                | ID: 0000477921 | Sentences:    18 | Tokens:     2454
Gold candidate       | ID: 0000057885 | Sentences:     5 | Tokens:      181
Retrieved rank 1     | ID: 0001733204 | Sentences:    76 | Tokens:     9235
Retrieved rank 2     | ID: 0001827392 | Sentences:    63 | Tokens:     6961
Retrieved rank 3     | ID: 0000046064 | Sentences:    64 | Tokens:     6327
Retrieved rank 4     | ID: 0001652416 | Sentences:    44 | Tokens:     9400
Retrieved rank 5     | ID: 0001113945 | Sentences:    55 | Tokens:     7657


In [114]:

import numpy as np

length_cache = {}

def get_token_length(case_id, collection):
    key = (case_id, id(collection))

    if key not in length_cache:
        length_cache[key] = sum(
            len(tokenizer.encode(str(sentence)).ids)
            for sentence in collection[case_id]["text"]
        )

    return length_cache[key]


gold_lengths = []
top7_lengths = []
query_lengths = []
queries_with_gold = 0

for qid in query_ids:

    gold_ids = set(
        data["dev"][qid]["relevant_candidates"]
    )

    if not gold_ids:
        continue

    queries_with_gold += 1

    query_lengths.append(
        get_token_length(qid, data["dev"])
    )

    # Average length of relevant candidates
    gold_lengths.extend([
        get_token_length(cid, data["dev_candidates"])
        for cid in gold_ids
    ])

    # Retrieve top 7 candidates from the saved ranking
    predictions = dev_predictions[qid]

    top7_lengths.extend([
        get_token_length(cid, data["dev_candidates"])
        for cid in predictions
    ])


print("DEVELOPMENT SET LENGTH DIAGNOSTICS")
print("-" * 55)

print("Queries with gold candidates:", queries_with_gold)

print("\nQuery token length:")
print("Median:", np.median(query_lengths))

print("\nGold candidate token length:")
print("Median:", np.median(gold_lengths))
print("Mean:", np.mean(gold_lengths))

print("\nTop-7 retrieved candidate token length:")
print("Median:", np.median(top7_lengths))
print("Mean:", np.mean(top7_lengths))


DEVELOPMENT SET LENGTH DIAGNOSTICS
-------------------------------------------------------
Queries with gold candidates: 118

Query token length:
Median: 5638.5

Gold candidate token length:
Median: 5176.0
Mean: 12778.602261048305

Top-7 retrieved candidate token length:
Median: 7483.5
Mean: 15285.42857142857


In [115]:

# EXPERIMENT 2 — MODEL INSPECTION

print("Model:", type(encoder).__name__)

print("\nEncoder layers:")
print("Number of layers:", len(encoder.encoder_layers))

print("\nLayer parameter counts:")

for i, layer in enumerate(encoder.encoder_layers):
    total = sum(p.numel() for p in layer.parameters())
    print(f"Layer {i+1}: {total:,} parameters")

print("\nEmbedding parameters:")
print(
    "Token embedding:",
    sum(p.numel() for p in encoder.token_embedding.parameters())
)

print(
    "Position embedding:",
    sum(p.numel() for p in encoder.encoder_pos.parameters())
)

print("\nTotal encoder parameters:")
print(f"{sum(p.numel() for p in encoder.parameters()):,}")


Model: A4EncoderForNER

Encoder layers:
Number of layers: 6

Layer parameter counts:
Layer 1: 3,152,384 parameters
Layer 2: 3,152,384 parameters
Layer 3: 3,152,384 parameters
Layer 4: 3,152,384 parameters
Layer 5: 3,152,384 parameters
Layer 6: 3,152,384 parameters

Embedding parameters:
Token embedding: 25600000
Position embedding: 524288

Total encoder parameters:
45,038,592


In [116]:

# EXPERIMENT 2: PARTIAL / LAYER-WISE FINE-TUNING

# 1. Freeze every parameter initially
for param in encoder.parameters():
    param.requires_grad = False

# 2. Unfreeze encoder layers 4, 5 and 6
# Python indices 3, 4, 5 correspond to layers 4, 5, 6.
for layer in encoder.encoder_layers[3:6]:
    for param in layer.parameters():
        param.requires_grad = True

# 3. Display trainability of every component
print("EXPERIMENT 2 — TRAINABILITY CHECK")
print("-" * 50)

print("Token embedding:",
      any(p.requires_grad for p in encoder.token_embedding.parameters()))

print("Position embedding:",
      any(p.requires_grad for p in encoder.encoder_pos.parameters()))

for i, layer in enumerate(encoder.encoder_layers):
    trainable = any(p.requires_grad for p in layer.parameters())
    print(f"Encoder layer {i+1}:",
          "TRAINABLE" if trainable else "FROZEN")

# 4. Count parameters
total_params = sum(p.numel() for p in encoder.parameters())

trainable_params = sum(
    p.numel() for p in encoder.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("\nPARAMETER SUMMARY")
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Frozen parameters:", f"{frozen_params:,}")
print("Trainable percentage:",
      round(100 * trainable_params / total_params, 2), "%")

# 5. Validate
assert trainable_params == 3 * 3_152_384
assert all(
    not p.requires_grad
    for layer in encoder.encoder_layers[:3]
    for p in layer.parameters()
)
assert all(
    p.requires_grad
    for layer in encoder.encoder_layers[3:]
    for p in layer.parameters()
)

print("\nPARTIAL FINE-TUNING CONFIGURATION: PASSED")


EXPERIMENT 2 — TRAINABILITY CHECK
--------------------------------------------------
Token embedding: False
Position embedding: False
Encoder layer 1: FROZEN
Encoder layer 2: FROZEN
Encoder layer 3: FROZEN
Encoder layer 4: TRAINABLE
Encoder layer 5: TRAINABLE
Encoder layer 6: TRAINABLE

PARAMETER SUMMARY
Total parameters: 45,038,592
Trainable parameters: 9,457,152
Frozen parameters: 35,581,440
Trainable percentage: 21.0 %

PARTIAL FINE-TUNING CONFIGURATION: PASSED


In [117]:

# EXPERIMENT 2 — TRAINING DATA INSPECTION

print("TRAINING DATA OVERVIEW")
print("-" * 55)

print("Number of queries:", len(data["train"]))
print("Number of candidates:", len(data["train_candidates"]))

# Inspect one query
sample_qid = next(iter(data["train"]))

sample_query = data["train"][sample_qid]
gold_ids = sample_query["relevant_candidates"]

print("\nSAMPLE QUERY")
print("Query ID:", sample_qid)
print("Number of sentences:", len(sample_query["text"]))
print("Relevant candidate count:", len(gold_ids))

print("\nFIRST 3 RELEVANT CANDIDATE IDs")

for cid in gold_ids[:3]:
    print(cid)

    candidate = data["train_candidates"].get(cid)

    if candidate is None:
        print("ERROR: Candidate missing!")
    else:
        print("Sentences:", len(candidate["text"]))

print("\nSAMPLE QUERY TEXT")
for sentence in sample_query["text"][:2]:
    print(str(sentence)[:300])

# Validate all training relevance links
missing = []

for qid, record in data["train"].items():
    for cid in record["relevant_candidates"]:
        if cid not in data["train_candidates"]:
            missing.append((qid, cid))

print("\nMISSING RELEVANT CANDIDATES:", len(missing))

assert len(missing) == 0

print("\nTRAINING DATA INSPECTION: PASSED")


TRAINING DATA OVERVIEW
-------------------------------------------------------
Number of queries: 827
Number of candidates: 4320

SAMPLE QUERY
Query ID: 0000000599
Number of sentences: 54
Relevant candidate count: 2

FIRST 3 RELEVANT CANDIDATE IDs
0001292810
Sentences: 30
0001742283
Sentences: 55

SAMPLE QUERY TEXT
PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, J.R. CITATION: 1964 AIR 1545 1964 SCR (3) 573 CITATOR INFO : R 1970 SC 765 (
CIVIL APPELLATE JURISDICTION: Civil Appeal Nos. 30 and 31 of 1963. 

MISSING RELEVANT CANDIDATES: 0

TRAINING DATA INSPECTION: PASSED


In [118]:

# EXPERIMENT 2 — POSITIVE / NEGATIVE DATA AUDIT

total_positive = 0
total_negative = 0
positive_counts = []

for qid, record in data["train"].items():

    relevant = set(record["relevant_candidates"])

    # Exclude the query itself from its candidate pool
    available = set(data["train_candidates"].keys()) - {qid}

    positive = relevant & available
    negative = available - relevant

    total_positive += len(positive)
    total_negative += len(negative)
    positive_counts.append(len(positive))

print("TRAINING PAIR AUDIT")
print("-" * 50)

print("Queries:", len(data["train"]))
print("Positive pairs:", total_positive)
print("Available negative pairs:", total_negative)
print("Average positives per query:",
      round(sum(positive_counts) / len(positive_counts), 2))

print("\nMinimum positives:", min(positive_counts))
print("Maximum positives:", max(positive_counts))

assert total_positive > 0
assert total_negative > 0

print("\nPAIR AUDIT: PASSED")


TRAINING PAIR AUDIT
--------------------------------------------------
Queries: 827
Positive pairs: 5308
Available negative pairs: 3566505
Average positives per query: 6.42

Minimum positives: 0
Maximum positives: 52

PAIR AUDIT: PASSED


In [119]:

# EXPERIMENT 2 — RANDOM NEGATIVE SAMPLING
import random

random.seed(42)

NEGATIVES_PER_QUERY = 4

training_pairs = []

for qid, record in data["train"].items():

    relevant_ids = set(record["relevant_candidates"])

    # Remove query itself and validate positives
    available_ids = set(data["train_candidates"].keys()) - {qid}
    positive_ids = relevant_ids & available_ids

    # Skip queries without positive examples
    if len(positive_ids) == 0:
        continue

    negative_pool = list(available_ids - positive_ids)

    sampled_negatives = random.sample(
        negative_pool,
        min(NEGATIVES_PER_QUERY, len(negative_pool))
    )

    for cid in positive_ids:
        training_pairs.append({
            "query_id": qid,
            "candidate_id": cid,
            "label": 1
        })

    for cid in sampled_negatives:
        training_pairs.append({
            "query_id": qid,
            "candidate_id": cid,
            "label": 0
        })

print("TRAINING PAIR GENERATION")
print("-" * 50)

print("Total pairs:", len(training_pairs))
print("Positive pairs:", sum(p["label"] == 1 for p in training_pairs))
print("Negative pairs:", sum(p["label"] == 0 for p in training_pairs))

print("\nFirst 5 examples:")
for pair in training_pairs[:5]:
    print(pair)

# Validate sampling
for pair in training_pairs:
    assert pair["candidate_id"] != pair["query_id"]

    if pair["label"] == 1:
        assert pair["candidate_id"] in data["train"][
            pair["query_id"]
        ]["relevant_candidates"]
    else:
        assert pair["candidate_id"] not in data["train"][
            pair["query_id"]
        ]["relevant_candidates"]

print("\nRANDOM NEGATIVE SAMPLING: PASSED")


TRAINING PAIR GENERATION
--------------------------------------------------
Total pairs: 8576
Positive pairs: 5308
Negative pairs: 3268

First 5 examples:
{'query_id': '0000000599', 'candidate_id': '0001292810', 'label': 1}
{'query_id': '0000000599', 'candidate_id': '0001742283', 'label': 1}
{'query_id': '0000000599', 'candidate_id': '0000000262', 'label': 0}
{'query_id': '0000000599', 'candidate_id': '0001781847', 'label': 0}
{'query_id': '0000000599', 'candidate_id': '0001912440', 'label': 0}

RANDOM NEGATIVE SAMPLING: PASSED


In [120]:

import json
import os

PAIR_PATH = "/kaggle/working/phase7_exp2_training_pairs.json"

with open(PAIR_PATH, "w") as f:
    json.dump(training_pairs, f, indent=2)

# Reload and verify
with open(PAIR_PATH, "r") as f:
    saved_pairs = json.load(f)

assert len(saved_pairs) == len(training_pairs)
assert saved_pairs == training_pairs

print("Saved:", PAIR_PATH)
print("Pairs:", len(saved_pairs))
print("Positive:", sum(p["label"] == 1 for p in saved_pairs))
print("Negative:", sum(p["label"] == 0 for p in saved_pairs))
print("SAVE AND RELOAD: PASSED")


Saved: /kaggle/working/phase7_exp2_training_pairs.json
Pairs: 8576
Positive: 5308
Negative: 3268
SAVE AND RELOAD: PASSED


In [121]:

# EXPERIMENT 2 — RESTORE CHECKPOINT AND CONFIGURE TRAINABLE LAYERS

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

state = checkpoint.get("model_state_dict", checkpoint)

# Create a fresh encoder using your already-defined class
encoder = A4EncoderForNER()

encoder_state = {
    key: value
    for key, value in state.items()
    if key.startswith((
        "token_embedding.",
        "encoder_pos.",
        "encoder_layers."
    ))
}

# Strict loading verifies checkpoint compatibility
encoder.load_state_dict(encoder_state, strict=True)

# Freeze everything
for param in encoder.parameters():
    param.requires_grad = False

# Unfreeze encoder layers 4, 5 and 6
for layer in encoder.encoder_layers[3:6]:
    for param in layer.parameters():
        param.requires_grad = True

# Move to GPU
encoder = encoder.to(device)

# Verify trainable parameters
trainable = sum(
    p.numel() for p in encoder.parameters()
    if p.requires_grad
)

total = sum(p.numel() for p in encoder.parameters())

print("Checkpoint:", CHECKPOINT_PATH)
print("Trainable parameters:", trainable)
print("Total parameters:", total)
print("Trainable percentage:", round(100 * trainable / total, 2))

for i, layer in enumerate(encoder.encoder_layers):
    status = "TRAINABLE" if any(
        p.requires_grad for p in layer.parameters()
    ) else "FROZEN"
    print(f"Encoder layer {i+1}: {status}")

assert trainable == 9_457_152
assert all(
    not p.requires_grad
    for p in encoder.token_embedding.parameters()
)
assert all(
    p.requires_grad
    for layer in encoder.encoder_layers[3:6]
    for p in layer.parameters()
)

print("\nCHECKPOINT RESTORE: PASSED")


Checkpoint: /kaggle/input/datasets/aq1de3/a4-step-220000/a4_step_22000.pt
Trainable parameters: 9457152
Total parameters: 45038592
Trainable percentage: 21.0
Encoder layer 1: FROZEN
Encoder layer 2: FROZEN
Encoder layer 3: FROZEN
Encoder layer 4: TRAINABLE
Encoder layer 5: TRAINABLE
Encoder layer 6: TRAINABLE

CHECKPOINT RESTORE: PASSED


In [122]:

# EXPERIMENT 2 — LOSS INPUT VALIDATION

import torch
import torch.nn.functional as F

labels = torch.tensor(
    [pair["label"] for pair in training_pairs],
    dtype=torch.float32
)

print("PyTorch version:", torch.__version__)
print("Training labels shape:", labels.shape)
print("Positive labels:", (labels == 1).sum().item())
print("Negative labels:", (labels == 0).sum().item())

assert labels.ndim == 1
assert torch.all((labels == 0) | (labels == 1))
assert (labels == 1).any()
assert (labels == 0).any()

print("\nLOSS INPUT VALIDATION: PASSED")


PyTorch version: 2.11.0+cu130
Training labels shape: torch.Size([8576])
Positive labels: 5308
Negative labels: 3268

LOSS INPUT VALIDATION: PASSED


In [123]:

# EXPERIMENT 2 — CONTRASTIVE LOSS SANITY TEST

import torch
import torch.nn.functional as F

def contrastive_loss(query_emb, candidate_emb, labels, margin=0.5):
    similarity = F.cosine_similarity(
        query_emb, candidate_emb, dim=1
    )

    positive_loss = labels * (1 - similarity).pow(2)

    negative_loss = (1 - labels) * torch.clamp(
        similarity - margin, min=0
    ).pow(2)

    loss = (positive_loss + negative_loss).mean()

    return loss, similarity


# Two positive examples: similar embeddings
# Two negative examples: dissimilar embeddings
query_emb = F.normalize(
    torch.tensor([
        [1.0, 0.0],
        [1.0, 0.0],
        [1.0, 0.0],
        [1.0, 0.0]
    ]),
    dim=1
)

candidate_emb = F.normalize(
    torch.tensor([
        [0.9, 0.1],
        [0.8, 0.2],
        [-1.0, 0.0],
        [0.0, 1.0]
    ]),
    dim=1
)

test_labels = torch.tensor([1., 1., 0., 0.])

loss, similarities = contrastive_loss(
    query_emb, candidate_emb, test_labels
)

print("Similarities:", similarities)
print("Contrastive loss:", loss.item())

assert torch.isfinite(loss)
assert loss.item() >= 0

print("\nCONTRASTIVE LOSS TEST: PASSED")


Similarities: tensor([ 0.9939,  0.9701, -1.0000,  0.0000])
Contrastive loss: 0.00023221912852022797

CONTRASTIVE LOSS TEST: PASSED


In [124]:

# EXPERIMENT 2 — GPU MEMORY CHECK

import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)

    print("GPU:", gpu.name)
    print("Total GPU memory:",
          round(gpu.total_memory / 1024**3, 2), "GB")

    print("Currently allocated:",
          round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

    print("Currently reserved:",
          round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

    print("\nGPU MEMORY CHECK: PASSED")
else:
    print("CUDA GPU not available")


GPU: Tesla T4
Total GPU memory: 14.56 GB
Currently allocated: 0.24 GB
Currently reserved: 1.37 GB

GPU MEMORY CHECK: PASSED


In [125]:

# EXPERIMENT 2 — GRADIENT FLOW TEST

encoder.train()
encoder.zero_grad(set_to_none=True)

dummy_ids = torch.randint(
    low=5,
    high=50000,
    size=(1, 32),
    device=device
)

dummy_mask = torch.ones_like(dummy_ids)

output = encoder(
    dummy_ids,
    attention_mask=dummy_mask
)

dummy_loss = output.pow(2).mean()
dummy_loss.backward()

print("Training mode:", encoder.training)
print("Dummy output shape:", tuple(output.shape))
print("Dummy loss:", dummy_loss.item())

for i, layer in enumerate(encoder.encoder_layers):
    grads = [
        p.grad for p in layer.parameters()
        if p.requires_grad
    ]

    has_gradient = any(g is not None for g in grads)
    finite = all(
        torch.isfinite(g).all().item()
        for g in grads if g is not None
    )

    print(
        f"Layer {i+1}: "
        f"gradient={has_gradient}, finite={finite}"
    )

# Check embeddings remain frozen
assert all(
    p.grad is None
    for p in encoder.token_embedding.parameters()
)

assert all(
    p.grad is None
    for p in encoder.encoder_layers[:3].parameters()
)

assert all(
    p.grad is not None
    for layer in encoder.encoder_layers[3:6]
    for p in layer.parameters()
)

assert torch.isfinite(dummy_loss)

encoder.zero_grad(set_to_none=True)

print("\nGRADIENT FLOW TEST: PASSED")


Training mode: True
Dummy output shape: (1, 32, 512)
Dummy loss: 4.792761325836182
Layer 1: gradient=False, finite=True
Layer 2: gradient=False, finite=True
Layer 3: gradient=False, finite=True
Layer 4: gradient=True, finite=True
Layer 5: gradient=True, finite=True
Layer 6: gradient=True, finite=True

GRADIENT FLOW TEST: PASSED


In [126]:

# EXPERIMENT 2 — TRAINING DOCUMENT LENGTH CHECK

import numpy as np

query_lengths = []
candidate_lengths = []

for record in data["train"].values():
    query_lengths.append(
        sum(len(tokenizer.encode(s).ids) for s in record["text"])
    )

for record in data["train_candidates"].values():
    candidate_lengths.append(
        sum(len(tokenizer.encode(s).ids)
            for s in record["text"])
    )

print("TRAINING DOCUMENT LENGTHS")
print("-" * 50)

print("Queries:", len(query_lengths))
print("Candidates:", len(candidate_lengths))

print("\nQuery token lengths:")
print("Median:", int(np.median(query_lengths)))
print("Mean:", round(float(np.mean(query_lengths)), 2))
print("Maximum:", max(query_lengths))

print("\nCandidate token lengths:")
print("Median:", int(np.median(candidate_lengths)))
print("Mean:", round(float(np.mean(candidate_lengths)), 2))
print("Maximum:", max(candidate_lengths))

print("\nDOCUMENT LENGTH CHECK: PASSED")


TRAINING DOCUMENT LENGTHS
--------------------------------------------------
Queries: 827
Candidates: 4320

Query token lengths:
Median: 5668
Mean: 8549.13
Maximum: 148250

Candidate token lengths:
Median: 4564
Mean: 7903.54
Maximum: 512627

DOCUMENT LENGTH CHECK: PASSED


In [127]:

# EXPERIMENT 2 — CHUNK STRUCTURE INSPECTION

sample_qid = next(iter(data["train"]))

sample_sentences = data["train"][sample_qid]["text"]

sample_chunks = chunk_document(
    sample_sentences,
    tokenizer
)

print("Sample query ID:", sample_qid)
print("Number of sentences:", len(sample_sentences))
print("Number of chunks:", len(sample_chunks))
print("Chunk output type:", type(sample_chunks))

if len(sample_chunks) > 0:
    print("First chunk type:", type(sample_chunks[0]))
    print("First chunk length:", len(sample_chunks[0]))
    print("First chunk token IDs:", sample_chunks[0][:15])

print("\nCHUNK STRUCTURE INSPECTION: PASSED")


Sample query ID: 0000000599
Number of sentences: 54
Number of chunks: 8
Chunk output type: <class 'list'>
First chunk type: <class 'list'>
First chunk length: 981
First chunk token IDs: [2, 4696, 30, 9225, 9934, 41, 34, 808, 18, 3301, 30, 3032, 1724, 2020, 26393]

CHUNK STRUCTURE INSPECTION: PASSED


In [128]:

# EXPERIMENT 2 — CORRECTED TOKENIZED DOCUMENT CACHE

import pickle

CACHE_PATH = "/kaggle/working/phase7_train_token_cache.pkl"

train_query_token_cache = {}
train_candidate_token_cache = {}

# Cache training queries
for doc_id, record in data["train"].items():
    train_query_token_cache[doc_id] = chunk_document(
        record["text"], tokenizer
    )

# Cache training candidates
for doc_id, record in data["train_candidates"].items():
    train_candidate_token_cache[doc_id] = chunk_document(
        record["text"], tokenizer
    )

# Save both collections separately
train_token_cache = {
    "queries": train_query_token_cache,
    "candidates": train_candidate_token_cache
}

with open(CACHE_PATH, "wb") as f:
    pickle.dump(train_token_cache, f)

# Reload and verify
with open(CACHE_PATH, "rb") as f:
    loaded_cache = pickle.load(f)

print("TOKEN CACHE SUMMARY")
print("-" * 50)
print("Query documents:", len(loaded_cache["queries"]))
print("Candidate documents:", len(loaded_cache["candidates"]))
print("Total document entries:",
      len(loaded_cache["queries"]) +
      len(loaded_cache["candidates"]))

print("Sample query chunks:",
      len(loaded_cache["queries"][sample_qid]))

assert len(loaded_cache["queries"]) == len(data["train"])
assert len(loaded_cache["candidates"]) == len(data["train_candidates"])

assert all(
    len(chunk) <= 1024
    for chunks in loaded_cache["queries"].values()
    for chunk in chunks
)

assert all(
    len(chunk) <= 1024
    for chunks in loaded_cache["candidates"].values()
    for chunk in chunks
)

print("\nTOKEN CACHE VALIDATION: PASSED")


TOKEN CACHE SUMMARY
--------------------------------------------------
Query documents: 827
Candidate documents: 4320
Total document entries: 5147
Sample query chunks: 8

TOKEN CACHE VALIDATION: PASSED


In [129]:

# EXPERIMENT 2 — CACHE RELOAD TEST

import pickle
import os

CACHE_PATH = "/kaggle/working/phase7_train_token_cache.pkl"

assert os.path.exists(CACHE_PATH)

with open(CACHE_PATH, "rb") as f:
    restored_cache = pickle.load(f)

print("File exists:", os.path.exists(CACHE_PATH))
print("File size (MB):",
      round(os.path.getsize(CACHE_PATH) / (1024**2), 2))

print("Restored queries:",
      len(restored_cache["queries"]))

print("Restored candidates:",
      len(restored_cache["candidates"]))

# Compare sample document
assert (
    restored_cache["queries"][sample_qid]
    == train_query_token_cache[sample_qid]
)

assert len(restored_cache["queries"]) == 827
assert len(restored_cache["candidates"]) == 4320

print("\nCACHE RELOAD TEST: PASSED")


File exists: True
File size (MB): 107.71
Restored queries: 827
Restored candidates: 4320

CACHE RELOAD TEST: PASSED


In [130]:

# EXPERIMENT 2 — TRAINING BATCH DATA CHECK

from torch.utils.data import Dataset

class LegalPairDataset(Dataset):
    def __init__(self, pairs, query_cache, candidate_cache):
        self.pairs = pairs
        self.query_cache = query_cache
        self.candidate_cache = candidate_cache

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        pair = self.pairs[index]

        qid = pair["query_id"]
        cid = pair["candidate_id"]

        return {
            "query_chunks": self.query_cache[qid],
            "candidate_chunks": self.candidate_cache[cid],
            "label": pair["label"]
        }


train_dataset = LegalPairDataset(
    training_pairs,
    restored_cache["queries"],
    restored_cache["candidates"]
)

sample = train_dataset[0]

print("DATASET CHECK")
print("-" * 50)
print("Dataset size:", len(train_dataset))
print("Query chunks:", len(sample["query_chunks"]))
print("Candidate chunks:", len(sample["candidate_chunks"]))
print("Query first chunk length:",
      len(sample["query_chunks"][0]))
print("Candidate first chunk length:",
      len(sample["candidate_chunks"][0]))
print("Label:", sample["label"])

assert len(train_dataset) == 8576
assert len(sample["query_chunks"]) > 0
assert len(sample["candidate_chunks"]) > 0
assert sample["label"] in (0, 1)

print("\nDATASET CHECK: PASSED")


DATASET CHECK
--------------------------------------------------
Dataset size: 8576
Query chunks: 8
Candidate chunks: 6
Query first chunk length: 981
Candidate first chunk length: 513
Label: 1

DATASET CHECK: PASSED


In [131]:

# EXPERIMENT 2 — PAIR CHUNK STATISTICS

query_chunk_counts = []
candidate_chunk_counts = []

for pair in training_pairs:
    qid = pair["query_id"]
    cid = pair["candidate_id"]

    query_chunk_counts.append(
        len(restored_cache["queries"][qid])
    )

    candidate_chunk_counts.append(
        len(restored_cache["candidates"][cid])
    )

print("PAIR CHUNK STATISTICS")
print("-" * 50)

print("Training pairs:", len(training_pairs))

print("\nQuery chunks per pair:")
print("Mean:", round(sum(query_chunk_counts) / len(query_chunk_counts), 2))
print("Maximum:", max(query_chunk_counts))

print("\nCandidate chunks per pair:")
print("Mean:", round(sum(candidate_chunk_counts) / len(candidate_chunk_counts), 2))
print("Maximum:", max(candidate_chunk_counts))

print("\nPAIR CHUNK STATISTICS: PASSED")


PAIR CHUNK STATISTICS
--------------------------------------------------
Training pairs: 8576

Query chunks per pair:
Mean: 14.67
Maximum: 180

Candidate chunks per pair:
Mean: 10.42
Maximum: 604

PAIR CHUNK STATISTICS: PASSED


In [132]:

# EXPERIMENT 2 — CHUNK COUNT DISTRIBUTION

import numpy as np

query_doc_chunks = [
    len(chunks)
    for chunks in restored_cache["queries"].values()
]

candidate_doc_chunks = [
    len(chunks)
    for chunks in restored_cache["candidates"].values()
]

percentiles = [50, 75, 90, 95, 99, 100]

print("QUERY DOCUMENT CHUNK DISTRIBUTION")
print("-" * 50)

for p in percentiles:
    print(f"P{p}: {np.percentile(query_doc_chunks, p):.1f}")

print("\nCANDIDATE DOCUMENT CHUNK DISTRIBUTION")
print("-" * 50)

for p in percentiles:
    print(f"P{p}: {np.percentile(candidate_doc_chunks, p):.1f}")

print("\nDocuments with more than 20 chunks:")
print("Queries:", sum(x > 20 for x in query_doc_chunks))
print("Candidates:", sum(x > 20 for x in candidate_doc_chunks))

print("\nCHUNK DISTRIBUTION AUDIT: PASSED")


QUERY DOCUMENT CHUNK DISTRIBUTION
--------------------------------------------------
P50: 7.0
P75: 11.0
P90: 19.0
P95: 29.7
P99: 61.7
P100: 180.0

CANDIDATE DOCUMENT CHUNK DISTRIBUTION
--------------------------------------------------
P50: 6.0
P75: 10.0
P90: 19.0
P95: 28.0
P99: 68.8
P100: 604.0

Documents with more than 20 chunks:
Queries: 74
Candidates: 361

CHUNK DISTRIBUTION AUDIT: PASSED


In [133]:

# EXPERIMENT 2 — TOTAL CHUNK COUNT AUDIT

total_query_chunks = sum(
    len(chunks)
    for chunks in restored_cache["queries"].values()
)

total_candidate_chunks = sum(
    len(chunks)
    for chunks in restored_cache["candidates"].values()
)

print("TOTAL CHUNK COUNT")
print("-" * 50)

print("Query documents:", len(restored_cache["queries"]))
print("Query chunks:", total_query_chunks)

print("\nCandidate documents:", len(restored_cache["candidates"]))
print("Candidate chunks:", total_candidate_chunks)

print("\nTotal documents:",
      len(restored_cache["queries"]) +
      len(restored_cache["candidates"]))

print("Total chunks:",
      total_query_chunks + total_candidate_chunks)

assert total_query_chunks > 0
assert total_candidate_chunks > 0

print("\nTOTAL CHUNK AUDIT: PASSED")


TOTAL CHUNK COUNT
--------------------------------------------------
Query documents: 827
Query chunks: 8408

Candidate documents: 4320
Candidate chunks: 40766

Total documents: 5147
Total chunks: 49174

TOTAL CHUNK AUDIT: PASSED


In [134]:

# EXPERIMENT 2 — DIFFERENTIABLE DOCUMENT ENCODING TEST

import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

PAD_ID = 0
BOS_ID = 2
EOS_ID = 3

def encode_document_trainable(chunks, encoder, chunk_batch_size=1):

    chunk_embeddings = []

    for start in range(0, len(chunks), chunk_batch_size):

        batch_chunks = chunks[start:start + chunk_batch_size]

        tensors = [
            torch.tensor(chunk, dtype=torch.long)
            for chunk in batch_chunks
        ]

        input_ids = pad_sequence(
            tensors,
            batch_first=True,
            padding_value=PAD_ID
        ).to(device)

        attention_mask = (input_ids != PAD_ID).long()

        hidden = encoder(
            input_ids,
            attention_mask=attention_mask
        )

        content_mask = (
            (input_ids != PAD_ID) &
            (input_ids != BOS_ID) &
            (input_ids != EOS_ID)
        )

        content_mask = content_mask.unsqueeze(-1)

        pooled = (
            (hidden * content_mask).sum(dim=1) /
            content_mask.sum(dim=1).clamp(min=1)
        )

        chunk_embeddings.append(pooled)

    document_embedding = torch.cat(
        chunk_embeddings, dim=0
    ).mean(dim=0)

    return F.normalize(document_embedding, dim=0)


# Use the first saved training pair
sample_pair = training_pairs[0]

query_chunks = restored_cache["queries"][
    sample_pair["query_id"]
]

candidate_chunks = restored_cache["candidates"][
    sample_pair["candidate_id"]
]

encoder.train()
encoder.zero_grad(set_to_none=True)

query_vector = encode_document_trainable(
    query_chunks, encoder
)

candidate_vector = encode_document_trainable(
    candidate_chunks, encoder
)

similarity = F.cosine_similarity(
    query_vector.unsqueeze(0),
    candidate_vector.unsqueeze(0)
)

label = torch.tensor(
    [float(sample_pair["label"])],
    device=device
)

loss, _ = contrastive_loss(
    query_vector.unsqueeze(0),
    candidate_vector.unsqueeze(0),
    label
)

loss.backward()

print("QUERY VECTOR:", tuple(query_vector.shape))
print("CANDIDATE VECTOR:", tuple(candidate_vector.shape))
print("SIMILARITY:", similarity.item())
print("LOSS:", loss.item())

print(
    "Layer 4 gradient:",
    encoder.encoder_layers[3].attn.q_proj.weight.grad is not None
)

assert query_vector.shape == (512,)
assert candidate_vector.shape == (512,)
assert torch.isfinite(loss)
assert encoder.encoder_layers[3].attn.q_proj.weight.grad is not None

encoder.zero_grad(set_to_none=True)

print("\nDIFFERENTIABLE ENCODING TEST: PASSED")


QUERY VECTOR: (512,)
CANDIDATE VECTOR: (512,)
SIMILARITY: 0.9759544134140015
LOSS: 0.0005781902582384646
Layer 4 gradient: True

DIFFERENTIABLE ENCODING TEST: PASSED


In [135]:

# EXPERIMENT 2 — POST-BACKPROPAGATION GPU MEMORY

torch.cuda.synchronize()

print("GPU:", torch.cuda.get_device_name(0))

print(
    "Allocated memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 3),
    "GB"
)

print(
    "Reserved memory:",
    round(torch.cuda.memory_reserved() / 1024**3, 3),
    "GB"
)

print(
    "Peak allocated memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    "GB"
)

print("\nPOST-BACKPROPAGATION MEMORY CHECK: PASSED")


GPU: Tesla T4
Allocated memory: 0.203 GB
Reserved memory: 2.592 GB
Peak allocated memory: 2.371 GB

POST-BACKPROPAGATION MEMORY CHECK: PASSED


In [136]:

# EXPERIMENT 2 — TRAINING DATALOADER

from torch.utils.data import DataLoader

def legal_pair_collate(batch):
    return {
        "query_chunks": [
            item["query_chunks"] for item in batch
        ],
        "candidate_chunks": [
            item["candidate_chunks"] for item in batch
        ],
        "labels": torch.tensor(
            [item["label"] for item in batch],
            dtype=torch.float32
        )
    }

# One-pair smoke test only; not the final training batch size
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=legal_pair_collate
)

batch = next(iter(train_loader))

print("TRAINING DATALOADER CHECK")
print("-" * 50)
print("Query documents:", len(batch["query_chunks"]))
print("Candidate documents:", len(batch["candidate_chunks"]))
print("Query chunks:", len(batch["query_chunks"][0]))
print("Candidate chunks:", len(batch["candidate_chunks"][0]))
print("Labels:", batch["labels"])

assert len(batch["query_chunks"]) == 1
assert len(batch["candidate_chunks"]) == 1
assert batch["labels"].shape == (1,)

print("\nDATALOADER VALIDATION: PASSED")


TRAINING DATALOADER CHECK
--------------------------------------------------
Query documents: 1
Candidate documents: 1
Query chunks: 51
Candidate chunks: 11
Labels: tensor([1.])

DATALOADER VALIDATION: PASSED


In [137]:

# EXPERIMENT 2 — OPTIMIZER PARAMETER AUDIT

trainable_parameters = [
    p for p in encoder.parameters()
    if p.requires_grad
]

print("Trainable parameter tensors:",
      len(trainable_parameters))

print("Trainable parameters:",
      sum(p.numel() for p in trainable_parameters))

print("Frozen parameters:",
      sum(p.numel() for p in encoder.parameters()
          if not p.requires_grad))

assert len(trainable_parameters) > 0
assert sum(p.numel() for p in trainable_parameters) == 9457152

print("\nOPTIMIZER PARAMETER AUDIT: PASSED")


Trainable parameter tensors: 48
Trainable parameters: 9457152
Frozen parameters: 35581440

OPTIMIZER PARAMETER AUDIT: PASSED


In [138]:
LEARNING_RATE = 0.0001  # Replace with your selected value

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", 0.01)

print("\nADAMW INITIALIZATION: PASSED")

Optimizer: AdamW
Learning rate: 0.0001
Weight decay: 0.01

ADAMW INITIALIZATION: PASSED


In [139]:

# EXPERIMENT 2 — SAVE INITIAL MODEL STATE

INITIAL_CHECKPOINT = (
    "/kaggle/working/phase7_exp2_initial.pt"
)

torch.save({
    "model_state_dict": encoder.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "experiment": "EXP-02 Partial Fine-Tuning",
    "trainable_layers": [4, 5, 6],
    "learning_rate": LEARNING_RATE,
    "weight_decay": 0.01
}, INITIAL_CHECKPOINT)

print("Saved:", INITIAL_CHECKPOINT)
print("File exists:", os.path.exists(INITIAL_CHECKPOINT))
print("\nINITIAL CHECKPOINT SAVE: PASSED")


Saved: /kaggle/working/phase7_exp2_initial.pt
File exists: True

INITIAL CHECKPOINT SAVE: PASSED


In [140]:

# EXPERIMENT 2 — ONE-BATCH TRAINING SMOKE TEST

encoder.train()
optimizer.zero_grad(set_to_none=True)

# Get one pair
sample = train_dataset[0]

# Save weights before update
before = (
    encoder.encoder_layers[3].attn.q_proj.weight
    .detach().clone()
)

# Encode query and candidate
q_vector = encode_document_trainable(
    sample["query_chunks"], encoder
)

c_vector = encode_document_trainable(
    sample["candidate_chunks"], encoder
)

q_vector = q_vector.unsqueeze(0)
c_vector = c_vector.unsqueeze(0)

labels_batch = torch.tensor(
    [float(sample["label"])],
    dtype=torch.float32,
    device=device
)

# Calculate loss
loss, similarities = contrastive_loss(
    q_vector, c_vector, labels_batch
)

# Backpropagation and optimizer update
loss.backward()
optimizer.step()

# Verify weights changed
after = (
    encoder.encoder_layers[3].attn.q_proj.weight
    .detach()
)

weight_change = (after - before).abs().max().item()

print("Training loss:", loss.item())
print("Similarity:", similarities.item())
print("Label:", sample["label"])
print("Maximum weight change:", weight_change)

assert torch.isfinite(loss)
assert weight_change > 0

optimizer.zero_grad(set_to_none=True)

print("\nONE-BATCH TRAINING TEST: PASSED")


Training loss: 0.0005781902582384646
Similarity: 0.9759544134140015
Label: 1
Maximum weight change: 9.961053729057312e-05

ONE-BATCH TRAINING TEST: PASSED


In [141]:

# EXPERIMENT 2 — FROZEN WEIGHT VALIDATION

initial_state = torch.load(
    INITIAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)["model_state_dict"]

frozen_count = 0
changed_frozen = []

for name, param in encoder.named_parameters():
    if not param.requires_grad:
        original = initial_state[name]
        current = param.detach().cpu()

        frozen_count += 1

        if not torch.equal(current, original):
            changed_frozen.append(name)

print("Frozen parameter tensors:", frozen_count)
print("Modified frozen tensors:", len(changed_frozen))

assert len(changed_frozen) == 0

print("\nFROZEN WEIGHT VALIDATION: PASSED")


Frozen parameter tensors: 50
Modified frozen tensors: 0

FROZEN WEIGHT VALIDATION: PASSED


In [142]:

# EXPERIMENT 2 — SAVE CONFIGURATION

import json

EXP2_CONFIG = {
    "experiment": "EXP-02",
    "method": "Partial Fine-Tuning",
    "base_model": "Phase 1 A4",
    "trainable_layers": [4, 5, 6],
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": 0.01,
    "loss": "Contrastive Loss",
    "epochs": 10,
    "training_pairs": len(training_pairs),
    "positive_pairs": 5308,
    "negative_pairs": 3268,
    "gpu": torch.cuda.get_device_name(0),
    "max_sequence_length": 1024
}

CONFIG_PATH = "/kaggle/working/phase7_exp2_config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(EXP2_CONFIG, f, indent=4)

print(json.dumps(EXP2_CONFIG, indent=4))
print("\nCONFIGURATION SAVED:", CONFIG_PATH)


{
    "experiment": "EXP-02",
    "method": "Partial Fine-Tuning",
    "base_model": "Phase 1 A4",
    "trainable_layers": [
        4,
        5,
        6
    ],
    "optimizer": "AdamW",
    "learning_rate": 0.0001,
    "weight_decay": 0.01,
    "loss": "Contrastive Loss",
    "epochs": 10,
    "training_pairs": 8576,
    "positive_pairs": 5308,
    "negative_pairs": 3268,
    "gpu": "Tesla T4",
    "max_sequence_length": 1024
}

CONFIGURATION SAVED: /kaggle/working/phase7_exp2_config.json


In [143]:

# EXPERIMENT 2 — MEMORY-EFFICIENT DOCUMENT ENCODING

from torch.utils.checkpoint import checkpoint
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F

def encode_document_trainable(
    chunks,
    encoder,
    chunk_batch_size=1
):
    chunk_embeddings = []

    for start in range(0, len(chunks), chunk_batch_size):

        batch_chunks = chunks[start:start + chunk_batch_size]

        tensors = [
            torch.tensor(chunk, dtype=torch.long)
            for chunk in batch_chunks
        ]

        input_ids = pad_sequence(
            tensors,
            batch_first=True,
            padding_value=PAD_ID
        ).to(device)

        attention_mask = (input_ids != PAD_ID).long()

        # Recompute encoder activations during backward
        hidden = checkpoint(
            lambda ids, mask: encoder(
                ids,
                attention_mask=mask
            ),
            input_ids,
            attention_mask,
            use_reentrant=False
        )

        content_mask = (
            (input_ids != PAD_ID) &
            (input_ids != BOS_ID) &
            (input_ids != EOS_ID)
        ).unsqueeze(-1)

        pooled = (
            (hidden * content_mask).sum(dim=1) /
            content_mask.sum(dim=1).clamp(min=1)
        )

        chunk_embeddings.append(pooled)

    document_embedding = torch.cat(
        chunk_embeddings,
        dim=0
    ).mean(dim=0)

    return F.normalize(document_embedding, dim=0)


print("Gradient checkpointing: enabled")
print("Chunk batch size: 1")
print("Maximum chunk length: 1024")
print("\nMEMORY-EFFICIENT ENCODER: READY")


Gradient checkpointing: enabled
Chunk batch size: 1
Maximum chunk length: 1024

MEMORY-EFFICIENT ENCODER: READY


In [144]:

# EXPERIMENT 2 — RESTORE CLEAN STARTING WEIGHTS

initial_state = torch.load(
    INITIAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

encoder.load_state_dict(
    initial_state["model_state_dict"],
    strict=True
)

optimizer.load_state_dict(
    initial_state["optimizer_state_dict"]
)

encoder.train()
optimizer.zero_grad(set_to_none=True)

print("Starting checkpoint restored.")
print("Optimizer state restored.")
print("Training mode:", encoder.training)

print("\nCLEAN START: PASSED")


Starting checkpoint restored.
Optimizer state restored.
Training mode: True

CLEAN START: PASSED


In [145]:

# EXPERIMENT 2 — GROUP TRAINING PAIRS BY QUERY

from collections import defaultdict

pairs_by_query = defaultdict(list)

for pair in training_pairs:
    pairs_by_query[pair["query_id"]].append(pair)

eligible_query_ids = [
    qid for qid, pairs in pairs_by_query.items()
    if len(pairs) >= 2
]

print("Total query groups:", len(pairs_by_query))
print("Groups with at least 2 pairs:", len(eligible_query_ids))

# Select one query with at least two pairs
benchmark_qid = eligible_query_ids[0]

benchmark_pairs = pairs_by_query[benchmark_qid][:2]

print("\nBenchmark query:", benchmark_qid)
print("Benchmark pairs:", len(benchmark_pairs))

for pair in benchmark_pairs:
    print(pair)

print("\nGROUPED BATCH: READY")


Total query groups: 817
Groups with at least 2 pairs: 817

Benchmark query: 0000000599
Benchmark pairs: 2
{'query_id': '0000000599', 'candidate_id': '0001292810', 'label': 1}
{'query_id': '0000000599', 'candidate_id': '0001742283', 'label': 1}

GROUPED BATCH: READY


In [146]:

# EXPERIMENT 2 — TWO-PAIR TRAINING BENCHMARK

import time
import torch
import torch.nn.functional as F

encoder.train()
optimizer.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()

# Encode query only once
query_id = benchmark_pairs[0]["query_id"]

query_embedding = encode_document_trainable(
    restored_cache["queries"][query_id],
    encoder,
    chunk_batch_size=1
)

# Encode the two candidate documents
candidate_embeddings = []

for pair in benchmark_pairs:
    candidate_id = pair["candidate_id"]

    candidate_embedding = encode_document_trainable(
        restored_cache["candidates"][candidate_id],
        encoder,
        chunk_batch_size=1
    )

    candidate_embeddings.append(candidate_embedding)

candidate_embeddings = torch.stack(candidate_embeddings)

labels = torch.tensor(
    [pair["label"] for pair in benchmark_pairs],
    dtype=torch.float32,
    device=device
)

query_embeddings = query_embedding.unsqueeze(0).expand_as(
    candidate_embeddings
)

loss, similarities = contrastive_loss(
    query_embeddings,
    candidate_embeddings,
    labels
)

loss.backward()

elapsed = time.time() - start_time

print("Benchmark loss:", loss.item())
print("Similarities:", similarities.detach().cpu().tolist())
print("Labels:", labels.cpu().tolist())
print("Elapsed seconds:", round(elapsed, 2))

if torch.cuda.is_available():
    print(
        "Peak GPU memory (GB):",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2)
    )

# Important: discard test gradients; no optimizer step occurred
optimizer.zero_grad(set_to_none=True)

print("\nTWO-PAIR BENCHMARK: COMPLETED")
print("Model weights were not updated.")


Benchmark loss: 0.001396148232743144
Similarities: [0.9759544134140015, 0.9529457092285156]
Labels: [1.0, 1.0]
Elapsed seconds: 1.24
Peak GPU memory (GB): 0.5

TWO-PAIR BENCHMARK: COMPLETED
Model weights were not updated.


In [147]:

# EXPERIMENT 2 — FULL TRAINING LOOP

import os
import time
import random
import torch
import torch.nn.functional as F

EPOCHS = 10
PAIR_BATCH_SIZE = 4
CHECKPOINT_DIR = "/kaggle/working/phase7_exp2_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Restore clean initial weights and reset optimizer
encoder.load_state_dict(
    initial_state["model_state_dict"],
    strict=True
)

for param in encoder.parameters():
    param.requires_grad = False

for layer in encoder.encoder_layers[3:6]:
    for param in layer.parameters():
        param.requires_grad = True

trainable_parameters = [
    p for p in encoder.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=0.0001,
    weight_decay=0.01
)

# Keep query groups intact
epoch_history = []
global_step = 0

for epoch in range(1, EPOCHS + 1):

    encoder.train()
    epoch_start = time.time()

    shuffled_qids = list(pairs_by_query.keys())
    random.shuffle(shuffled_qids)

    total_loss = 0.0
    total_pairs = 0
    successful_steps = 0

    for qid in shuffled_qids:

        query_pairs = pairs_by_query[qid].copy()
        random.shuffle(query_pairs)

        for start in range(0, len(query_pairs), PAIR_BATCH_SIZE):

            pair_batch = query_pairs[
                start:start + PAIR_BATCH_SIZE
            ]

            optimizer.zero_grad(set_to_none=True)

            # Encode query once for this group
            query_embedding = encode_document_trainable(
                restored_cache["queries"][qid],
                encoder,
                chunk_batch_size=1
            )

            # Encode candidate documents
            candidate_embeddings = []

            for pair in pair_batch:
                candidate_id = pair["candidate_id"]

                candidate_embedding = encode_document_trainable(
                    restored_cache["candidates"][candidate_id],
                    encoder,
                    chunk_batch_size=1
                )

                candidate_embeddings.append(candidate_embedding)

            candidate_embeddings = torch.stack(
                candidate_embeddings
            )

            query_embeddings = query_embedding.unsqueeze(0).expand_as(
                candidate_embeddings
            )

            labels = torch.tensor(
                [p["label"] for p in pair_batch],
                dtype=torch.float32,
                device=device
            )

            loss, similarities = contrastive_loss(
                query_embeddings,
                candidate_embeddings,
                labels
            )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item() * len(pair_batch)
            total_pairs += len(pair_batch)
            successful_steps += 1
            global_step += 1

            # Periodic recovery checkpoint
            if global_step % 100 == 0:
                checkpoint_path = os.path.join(
                    CHECKPOINT_DIR,
                    f"exp2_step_{global_step}.pt"
                )

                torch.save({
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_state_dict": encoder.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch_history": epoch_history
                }, checkpoint_path)

        # Progress within epoch
        if total_pairs and total_pairs % 500 < PAIR_BATCH_SIZE:
            print(
                f"Epoch {epoch}/{EPOCHS} | "
                f"Pairs: {total_pairs}/{len(training_pairs)} | "
                f"Average loss: {total_loss / total_pairs:.6f}"
            )

    average_loss = total_loss / max(total_pairs, 1)
    elapsed = time.time() - epoch_start

    epoch_result = {
        "epoch": epoch,
        "average_loss": average_loss,
        "pairs_processed": total_pairs,
        "optimizer_steps": successful_steps,
        "seconds": elapsed
    }

    epoch_history.append(epoch_result)

    # Save end-of-epoch checkpoint
    epoch_path = os.path.join(
        CHECKPOINT_DIR,
        f"exp2_epoch_{epoch}.pt"
    )

    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": encoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch_history": epoch_history
    }, epoch_path)

    print("\n" + "=" * 55)
    print(f"COMPLETED EPOCH {epoch}/{EPOCHS}")
    print(f"Average loss: {average_loss:.6f}")
    print(f"Pairs processed: {total_pairs}")
    print(f"Optimizer steps: {successful_steps}")
    print(f"Time: {elapsed / 60:.2f} minutes")
    print(f"Checkpoint: {epoch_path}")
    print("=" * 55)

print("\nEXPERIMENT 2 TRAINING FINISHED")


Epoch 1/10 | Pairs: 1002/8576 | Average loss: 0.060837
Epoch 1/10 | Pairs: 1502/8576 | Average loss: 0.056759
Epoch 1/10 | Pairs: 2003/8576 | Average loss: 0.057316
Epoch 1/10 | Pairs: 5002/8576 | Average loss: 0.048428
Epoch 1/10 | Pairs: 7503/8576 | Average loss: 0.044688
Epoch 1/10 | Pairs: 8500/8576 | Average loss: 0.043526

COMPLETED EPOCH 1/10
Average loss: 0.043493
Pairs processed: 8576
Optimizer steps: 2487
Time: 105.94 minutes
Checkpoint: /kaggle/working/phase7_exp2_checkpoints/exp2_epoch_1.pt
Epoch 2/10 | Pairs: 502/8576 | Average loss: 0.035002
Epoch 2/10 | Pairs: 1500/8576 | Average loss: 0.034348
Epoch 2/10 | Pairs: 3501/8576 | Average loss: 0.034366
Epoch 2/10 | Pairs: 5001/8576 | Average loss: 0.034013
Epoch 2/10 | Pairs: 6503/8576 | Average loss: 0.033516
Epoch 2/10 | Pairs: 7003/8576 | Average loss: 0.033761

COMPLETED EPOCH 2/10
Average loss: 0.033814
Pairs processed: 8576
Optimizer steps: 2487
Time: 105.92 minutes
Checkpoint: /kaggle/working/phase7_exp2_checkpoints/e

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 154978624 vs 154978512

In [148]:

import os
import glob

checkpoint_dir = "/kaggle/working/phase7_exp2_checkpoints"

step_files = glob.glob(
    os.path.join(checkpoint_dir, "exp2_step_*.pt")
)

freed_bytes = 0

for path in step_files:
    size = os.path.getsize(path)
    os.remove(path)
    freed_bytes += size

print("Intermediate checkpoints deleted:", len(step_files))
print("Space freed:", round(freed_bytes / (1024**3), 2), "GB")

print("\nPreserved epoch checkpoints:")
for path in sorted(glob.glob(
    os.path.join(checkpoint_dir, "exp2_epoch_*.pt")
)):
    print(os.path.basename(path),
          round(os.path.getsize(path) / (1024**3), 3), "GB")

print("\nCLEANUP COMPLETED")


Intermediate checkpoints deleted: 78
Space freed: 18.5 GB

Preserved epoch checkpoints:
exp2_epoch_1.pt 0.238 GB
exp2_epoch_2.pt 0.238 GB
exp2_epoch_3.pt 0.238 GB

CLEANUP COMPLETED


In [149]:

# EXPERIMENT 2 — RESUME FROM EPOCH 3

import os
import torch

checkpoint_dir = "/kaggle/working/phase7_exp2_checkpoints"
resume_path = os.path.join(checkpoint_dir, "exp2_epoch_3.pt")

# Load completed Epoch 3 checkpoint
resume_state = torch.load(
    resume_path,
    map_location="cpu",
    weights_only=False
)

# Restore model
encoder.load_state_dict(
    resume_state["model_state_dict"],
    strict=True
)

# Restore trainable layers
for param in encoder.parameters():
    param.requires_grad = False

for layer in encoder.encoder_layers[3:6]:
    for param in layer.parameters():
        param.requires_grad = True

trainable_parameters = [
    p for p in encoder.parameters() if p.requires_grad
]

# Recreate optimizer and restore its state
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=0.0001,
    weight_decay=0.01
)

optimizer.load_state_dict(
    resume_state["optimizer_state_dict"]
)

# Restore training progress
global_step = resume_state["global_step"]
epoch_history = resume_state["epoch_history"]

START_EPOCH = resume_state["epoch"] + 1
EPOCHS = 10

encoder.to(device)
encoder.train()

print("Checkpoint restored:", resume_path)
print("Last completed epoch:", resume_state["epoch"])
print("Next epoch:", START_EPOCH)
print("Total epochs:", EPOCHS)
print("Global step:", global_step)
print("Optimizer state restored:", True)
print("Trainable parameters:", sum(
    p.numel() for p in trainable_parameters
))

print("\nRESUME SETUP: PASSED")


Checkpoint restored: /kaggle/working/phase7_exp2_checkpoints/exp2_epoch_3.pt
Last completed epoch: 3
Next epoch: 4
Total epochs: 10
Global step: 7461
Optimizer state restored: True
Trainable parameters: 9457152

RESUME SETUP: PASSED


In [150]:

# FAST ENCODER — EXPERIMENT 2
import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.checkpoint import checkpoint

CHUNK_BATCH_SIZE = 4

def encode_document_fast(chunks, encoder):

    chunk_embeddings = []

    for start in range(0, len(chunks), CHUNK_BATCH_SIZE):

        batch_chunks = chunks[start:start + CHUNK_BATCH_SIZE]

        tensors = [
            torch.tensor(chunk, dtype=torch.long)
            for chunk in batch_chunks
        ]

        input_ids = pad_sequence(
            tensors,
            batch_first=True,
            padding_value=PAD_ID
        ).to(device)

        attention_mask = (input_ids != PAD_ID).long()

        use_checkpoint = len(chunks) > 8

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            if use_checkpoint:
                hidden = checkpoint(
                    lambda ids, mask: encoder(
                        ids, attention_mask=mask
                    ),
                    input_ids,
                    attention_mask,
                    use_reentrant=False
                )
            else:
                hidden = encoder(
                    input_ids,
                    attention_mask=attention_mask
                )

        hidden = hidden.float()

        content_mask = (
            (input_ids != PAD_ID) &
            (input_ids != BOS_ID) &
            (input_ids != EOS_ID)
        ).unsqueeze(-1)

        pooled = (
            (hidden * content_mask).sum(dim=1) /
            content_mask.sum(dim=1).clamp(min=1)
        )

        chunk_embeddings.append(pooled)

    document_embedding = torch.cat(
        chunk_embeddings, dim=0
    ).mean(dim=0)

    return F.normalize(document_embedding, dim=0)


print("Chunk batch size:", CHUNK_BATCH_SIZE)
print("Mixed precision: FP16")
print("Long-document checkpointing: enabled")
print("FAST ENCODER READY")


Chunk batch size: 4
Mixed precision: FP16
Long-document checkpointing: enabled
FAST ENCODER READY


In [151]:

# SPEED AND MEMORY BENCHMARK

import time

encoder.train()
optimizer.zero_grad(set_to_none=True)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

sample_pairs = pairs_by_query[benchmark_qid][:2]

start = time.time()

q_emb = encode_document_fast(
    restored_cache["queries"][benchmark_qid],
    encoder
)

c_embs = []

for pair in sample_pairs:
    c_embs.append(
        encode_document_fast(
            restored_cache["candidates"][pair["candidate_id"]],
            encoder
        )
    )

c_embs = torch.stack(c_embs)
q_embs = q_emb.unsqueeze(0).expand_as(c_embs)

labels = torch.tensor(
    [p["label"] for p in sample_pairs],
    dtype=torch.float32,
    device=device
)

loss, sims = contrastive_loss(q_embs, c_embs, labels)
loss.backward()

elapsed = time.time() - start

print("Loss:", loss.item())
print("Elapsed:", round(elapsed, 2), "seconds")

if torch.cuda.is_available():
    print(
        "Peak GPU memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

optimizer.zero_grad(set_to_none=True)

print("Weights unchanged: optimizer.step() was not called")
print("BENCHMARK COMPLETE")


Loss: 0.00024250303977169096
Elapsed: 0.61 seconds
Peak GPU memory: 4.07 GB
Weights unchanged: optimizer.step() was not called
BENCHMARK COMPLETE


In [ ]:

# EXPERIMENT 2 — FAST RESUMED TRAINING

import os
import time
import random
import torch

EPOCHS = 10
START_EPOCH = 4

CHECKPOINT_DIR = "/kaggle/working/phase7_exp2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for epoch in range(START_EPOCH, EPOCHS + 1):

    encoder.train()
    epoch_start = time.time()

    shuffled_qids = list(pairs_by_query.keys())
    random.shuffle(shuffled_qids)

    total_loss = 0.0
    total_pairs = 0
    successful_steps = 0

    for qid in shuffled_qids:

        query_pairs = pairs_by_query[qid].copy()
        random.shuffle(query_pairs)

        for start in range(0, len(query_pairs), PAIR_BATCH_SIZE):

            pair_batch = query_pairs[
                start:start + PAIR_BATCH_SIZE
            ]

            optimizer.zero_grad(set_to_none=True)

            query_embedding = encode_document_fast(
                restored_cache["queries"][qid],
                encoder
            )

            candidate_embeddings = []

            for pair in pair_batch:
                candidate_embeddings.append(
                    encode_document_fast(
                        restored_cache["candidates"][pair["candidate_id"]],
                        encoder
                    )
                )

            candidate_embeddings = torch.stack(candidate_embeddings)

            query_embeddings = query_embedding.unsqueeze(0).expand_as(
                candidate_embeddings
            )

            labels = torch.tensor(
                [p["label"] for p in pair_batch],
                dtype=torch.float32,
                device=device
            )

            loss, similarities = contrastive_loss(
                query_embeddings,
                candidate_embeddings,
                labels
            )

            if not torch.isfinite(loss):
                optimizer.zero_grad(set_to_none=True)
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=1.0
            )

            optimizer.step()

            total_loss += loss.item() * len(pair_batch)
            total_pairs += len(pair_batch)
            successful_steps += 1
            global_step += 1

        if total_pairs and total_pairs % 500 < PAIR_BATCH_SIZE:
            print(
                f"Epoch {epoch}/{EPOCHS} | "
                f"Pairs: {total_pairs}/{len(training_pairs)} | "
                f"Average loss: {total_loss / total_pairs:.6f}"
            )

    average_loss = total_loss / max(total_pairs, 1)
    elapsed = time.time() - epoch_start

    epoch_history.append({
        "epoch": epoch,
        "average_loss": average_loss,
        "pairs_processed": total_pairs,
        "optimizer_steps": successful_steps,
        "seconds": elapsed
    })

    epoch_path = os.path.join(
        CHECKPOINT_DIR,
        f"exp2_epoch_{epoch}.pt"
    )

    # Save only completed epoch checkpoints
    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": encoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch_history": epoch_history
    }, epoch_path)

    print("\n" + "=" * 55)
    print(f"COMPLETED EPOCH {epoch}/{EPOCHS}")
    print(f"Average loss: {average_loss:.6f}")
    print(f"Pairs processed: {total_pairs}")
    print(f"Time: {elapsed / 60:.2f} minutes")
    print(f"Checkpoint: {epoch_path}")
    print("=" * 55)

print("\nEXPERIMENT 2 COMPLETE")


Epoch 4/10 | Pairs: 501/8576 | Average loss: 0.028778
Epoch 4/10 | Pairs: 2500/8576 | Average loss: 0.026722
Epoch 4/10 | Pairs: 3001/8576 | Average loss: 0.026873
Epoch 4/10 | Pairs: 4003/8576 | Average loss: 0.027094
Epoch 4/10 | Pairs: 5501/8576 | Average loss: 0.027159
Epoch 4/10 | Pairs: 7000/8576 | Average loss: 0.028189
Epoch 4/10 | Pairs: 7503/8576 | Average loss: 0.028238

COMPLETED EPOCH 4/10
Average loss: 0.028488
Pairs processed: 8576
Time: 48.29 minutes
Checkpoint: /kaggle/working/phase7_exp2_checkpoints/exp2_epoch_4.pt
Epoch 5/10 | Pairs: 1501/8576 | Average loss: 0.027732
Epoch 5/10 | Pairs: 2503/8576 | Average loss: 0.027577
Epoch 5/10 | Pairs: 6003/8576 | Average loss: 0.028334
Epoch 5/10 | Pairs: 7002/8576 | Average loss: 0.027445

COMPLETED EPOCH 5/10
Average loss: 0.027521
Pairs processed: 8576
Time: 48.27 minutes
Checkpoint: /kaggle/working/phase7_exp2_checkpoints/exp2_epoch_5.pt
Epoch 6/10 | Pairs: 1003/8576 | Average loss: 0.023456
Epoch 6/10 | Pairs: 1501/8576 |

In [1]:

# RECOVER EXPERIMENT 2 FROM EPOCH 5

import torch
import os

checkpoint_dir = "/kaggle/working/phase7_exp2_checkpoints"
resume_path = os.path.join(checkpoint_dir, "exp2_epoch_5.pt")

resume_state = torch.load(
    resume_path,
    map_location="cpu",
    weights_only=False
)

encoder.load_state_dict(
    resume_state["model_state_dict"],
    strict=True
)

# Freeze all layers except encoder layers 4, 5, 6
for param in encoder.parameters():
    param.requires_grad = False

for layer in encoder.encoder_layers[3:6]:
    for param in layer.parameters():
        param.requires_grad = True

encoder.to(device)

trainable_parameters = [
    p for p in encoder.parameters() if p.requires_grad
]

# Restore optimizer
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=0.0001,
    weight_decay=0.01
)

optimizer.load_state_dict(
    resume_state["optimizer_state_dict"]
)

# Restore training history
global_step = resume_state["global_step"]
epoch_history = resume_state["epoch_history"]

START_EPOCH = 6
EPOCHS = 10

encoder.train()

print("Restored checkpoint: Epoch 5")
print("Next epoch:", START_EPOCH)
print("Final epoch:", EPOCHS)
print("Global step:", global_step)
print("Trainable parameters:", sum(
    p.numel() for p in trainable_parameters
))

print("\nRECOVERY SUCCESSFUL")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/phase7_exp2_checkpoints/exp2_epoch_5.pt'

In [2]:

import os
import glob

matches = glob.glob(
    "/kaggle/input/**/exp2_epoch_5*.pt",
    recursive=True
)

print("Checkpoint files found:", len(matches))

for path in matches:
    print(path)

assert len(matches) > 0, "Epoch 5 checkpoint not found!"

RESUME_PATH = matches[0]

print("\nSelected checkpoint:", RESUME_PATH)
print("File size (GB):",
      round(os.path.getsize(RESUME_PATH) / 1024**3, 3))


Checkpoint files found: 1
/kaggle/input/datasets/aq1de3/exp2-epoch-5/exp2_epoch_5 (1).pt

Selected checkpoint: /kaggle/input/datasets/aq1de3/exp2-epoch-5/exp2_epoch_5 (1).pt
File size (GB): 0.238


In [6]:

# RESTORE EXPERIMENT 2 FROM EPOCH 5

import torch
import os

RESUME_PATH = "/kaggle/input/datasets/aq1de3/exp2-epoch-5/exp2_epoch_5 (1).pt"

resume_state = torch.load(
    RESUME_PATH,
    map_location="cpu",
    weights_only=False
)

# Restore model weights
encoder.load_state_dict(
    resume_state["model_state_dict"],
    strict=True
)

# Freeze all encoder layers
for param in encoder.parameters():
    param.requires_grad = False

# Unfreeze encoder layers 4, 5, 6
for layer in encoder.encoder_layers[3:6]:
    for param in layer.parameters():
        param.requires_grad = True

encoder.to(device)

trainable_parameters = [
    p for p in encoder.parameters() if p.requires_grad
]

# Restore optimizer
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=0.0001,
    weight_decay=0.01
)

optimizer.load_state_dict(
    resume_state["optimizer_state_dict"]
)

# Restore history
global_step = resume_state["global_step"]
epoch_history = resume_state["epoch_history"]

START_EPOCH = resume_state["epoch"] + 1
EPOCHS = 10

encoder.train()

print("Restored epoch:", resume_state["epoch"])
print("Next epoch:", START_EPOCH)
print("Global step:", global_step)
print("Optimizer restored: Yes")
print("Trainable parameters:", sum(
    p.numel() for p in trainable_parameters
))

print("\nRECOVERY SUCCESSFUL")


NameError: name 'encoder' is not defined

In [7]:

# RECREATE A4 ENCODER FROM PHASE 1 CHECKPOINT

import torch
import os

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

A4_CHECKPOINT_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-220000/a4_step_22000.pt"
)

checkpoint = torch.load(
    A4_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

# Instantiate your existing encoder architecture
encoder = A4EncoderForNER(
    vocab_size=50000,
    d_model=512,
    max_seq_len=1024
)

# If your constructor uses different arguments,
# use the exact constructor from your earlier notebook.

# Load Phase 1 encoder weights
# Use the same state-dict extraction that passed
# your earlier strict-loading test.

print("Encoder instantiated:", type(encoder).__name__)
print("Device:", device)


TypeError: A4EncoderForNER.__init__() got an unexpected keyword argument 'vocab_size'

In [8]:

import inspect

print("A4EncoderForNER constructor:")
print(inspect.signature(A4EncoderForNER.__init__))

print("\nClass definition:")
print(inspect.getsource(A4EncoderForNER.__init__))


A4EncoderForNER constructor:
(self)

Class definition:
    def __init__(self):
        super().__init__()

        self.token_embedding = nn.Embedding(
            VOCAB_SIZE, HIDDEN_SIZE
        )

        self.encoder_pos = nn.Embedding(
            MAX_POSITION, HIDDEN_SIZE
        )

        self.encoder_layers = nn.ModuleList([
            A4EncoderLayer(
                HIDDEN_SIZE,
                NUM_HEADS,
                FFN_SIZE
            )
            for _ in range(NUM_LAYERS)
        ])



In [9]:

# RECREATE ENCODER AND RESTORE EPOCH 5

import torch
import os

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

RESUME_PATH = (
    "/kaggle/input/datasets/aq1de3/"
    "exp2-epoch-5/exp2_epoch_5 (1).pt"
)

# Instantiate using your actual constructor
encoder = A4EncoderForNER()

# Load Epoch 5 checkpoint
resume_state = torch.load(
    RESUME_PATH,
    map_location="cpu",
    weights_only=False
)

# Restore trained encoder weights directly
encoder.load_state_dict(
    resume_state["model_state_dict"],
    strict=True
)

encoder.to(device)

# Freeze all layers
for param in encoder.parameters():
    param.requires_grad = False

# Unfreeze encoder layers 4, 5, 6
for layer in encoder.encoder_layers[3:6]:
    for param in layer.parameters():
        param.requires_grad = True

trainable_parameters = [
    p for p in encoder.parameters()
    if p.requires_grad
]

# Restore optimizer
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=0.0001,
    weight_decay=0.01
)

optimizer.load_state_dict(
    resume_state["optimizer_state_dict"]
)

# Restore training history
global_step = resume_state["global_step"]
epoch_history = resume_state["epoch_history"]

START_EPOCH = 6
EPOCHS = 10

encoder.train()

print("Encoder:", type(encoder).__name__)
print("Restored epoch:", resume_state["epoch"])
print("Next epoch:", START_EPOCH)
print("Global step:", global_step)
print("Optimizer restored: YES")
print("Trainable parameters:", sum(
    p.numel() for p in trainable_parameters
))
print("Device:", device)

print("\nEPOCH 5 RECOVERY: SUCCESSFUL")


Encoder: A4EncoderForNER
Restored epoch: 5
Next epoch: 6
Global step: 12435
Optimizer restored: YES
Trainable parameters: 9457152
Device: cuda

EPOCH 5 RECOVERY: SUCCESSFUL


In [10]:

# CHECK TRAINING DATA AFTER KERNEL RESTART

required_objects = [
    "training_pairs",
    "pairs_by_query",
    "restored_cache",
    "encode_document_fast",
    "contrastive_loss"
]

for name in required_objects:
    exists = name in globals()
    print(f"{name}: {'READY' if exists else 'MISSING'}")

print("\nTraining pairs:", len(training_pairs) if "training_pairs" in globals() else "MISSING")
print("Query groups:", len(pairs_by_query) if "pairs_by_query" in globals() else "MISSING")

if "restored_cache" in globals():
    print("Cached queries:", len(restored_cache["queries"]))
    print("Cached candidates:", len(restored_cache["candidates"]))


training_pairs: MISSING
pairs_by_query: MISSING
restored_cache: MISSING
encode_document_fast: MISSING
contrastive_loss: MISSING

Training pairs: MISSING
Query groups: MISSING


In [11]:

# FIND EXPERIMENT 2 DATA AND CACHE FILES

import os

search_roots = [
    "/kaggle/working",
    "/kaggle/input"
]

keywords = [
    "training_pairs",
    "token_cache",
    "train",
    "tokenizer",
    "candidate"
]

print("SEARCHING FOR PHASE 7 FILES\n")

for root in search_roots:
    print(f"\n--- {root} ---")

    for current, dirs, files in os.walk(root):
        for filename in files:
            if any(k in filename.lower() for k in keywords):
                path = os.path.join(current, filename)
                size = os.path.getsize(path) / (1024**2)

                print(f"{filename} | {size:.2f} MB")
                print(path)


SEARCHING FOR PHASE 7 FILES


--- /kaggle/working ---

--- /kaggle/input ---
dev_candidates.json | 49.48 MB
/kaggle/input/datasets/aq1de3/phase7/dev_candidates.json
test_candidates.json | 67.84 MB
/kaggle/input/datasets/aq1de3/phase7/test_candidates.json
train (4).json | 31.65 MB
/kaggle/input/datasets/aq1de3/phase7/train (4).json
train_candidates.json | 152.38 MB
/kaggle/input/datasets/aq1de3/phase7/train_candidates.json
tokenizer.json | 3.37 MB
/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json


In [12]:

import os

DATA_ROOT = "/kaggle/input/datasets/aq1de3/phase7"

print("ALL PHASE 7 DATA FILES\n")

for root, dirs, files in os.walk(DATA_ROOT):
    for filename in sorted(files):
        path = os.path.join(root, filename)
        size_mb = os.path.getsize(path) / (1024**2)
        print(f"{filename:30s} {size_mb:8.2f} MB")
        print(path)


ALL PHASE 7 DATA FILES

dev (3).json                       5.59 MB
/kaggle/input/datasets/aq1de3/phase7/dev (3).json
dev_candidates.json               49.48 MB
/kaggle/input/datasets/aq1de3/phase7/dev_candidates.json
test (5).json                      8.91 MB
/kaggle/input/datasets/aq1de3/phase7/test (5).json
test_candidates.json              67.84 MB
/kaggle/input/datasets/aq1de3/phase7/test_candidates.json
train (4).json                    31.65 MB
/kaggle/input/datasets/aq1de3/phase7/train (4).json
train_candidates.json            152.38 MB
/kaggle/input/datasets/aq1de3/phase7/train_candidates.json


In [13]:

import json
import os

DATA_ROOT = "/kaggle/input/datasets/aq1de3/phase7"

TRAIN_PATH = os.path.join(DATA_ROOT, "train (4).json")
TRAIN_CANDIDATES_PATH = os.path.join(
    DATA_ROOT, "train_candidates.json"
)

print("Loading training queries...")
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

print("Loading training candidates...")
with open(TRAIN_CANDIDATES_PATH, "r", encoding="utf-8") as f:
    train_candidates = json.load(f)

print("\nTRAINING DATA LOADED")
print("Queries:", len(train_data))
print("Candidates:", len(train_candidates))

print("\nQuery sample:")
print(str(train_data[0])[:1000])

print("\nCandidate sample:")
print(str(train_candidates[0])[:500])


Loading training queries...
Loading training candidates...

TRAINING DATA LOADED
Queries: 827
Candidates: 4320

Query sample:


KeyError: 0

In [14]:

print("TRAIN DATA TYPE:", type(train_data))
print("TRAIN DATA KEYS:", list(train_data.keys())[:20])

print("\nTRAIN CANDIDATES TYPE:", type(train_candidates))

if isinstance(train_candidates, dict):
    print("CANDIDATE KEYS:", list(train_candidates.keys())[:20])

print("\nTRAIN DATA SAMPLE:")
for key in list(train_data.keys())[:3]:
    value = train_data[key]
    print(f"\nKey: {key}")
    print("Value type:", type(value))
    print("Sample:", str(value)[:700])


TRAIN DATA TYPE: <class 'dict'>
TRAIN DATA KEYS: ['0000000599', '0000002886', '0000009122', '0000009256', '0000010271', '0000010498', '0000014789', '0000014936', '0000017389', '0000018269', '0000021773', '0000025576', '0000032468', '0000034445', '0000034813', '0000035294', '0000038585', '0000040124', '0000041217', '0000042052']

TRAIN CANDIDATES TYPE: <class 'dict'>
CANDIDATE KEYS: ['0000000262', '0000000599', '0000001566', '0000002408', '0000002442', '0000002463', '0000002833', '0000002843', '0000002886', '0000003060', '0000004354', '0000004475', '0000004617', '0000004772', '0000005192', '0000005493', '0000005947', '0000006759', '0000006974', '0000007685']

TRAIN DATA SAMPLE:

Key: 0000000599
Value type: <class 'dict'>
Sample: {'text': ['PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, J.R. CITATION: 1964 AIR 154

In [15]:

# Inspect one query and its relevance labels

sample_qid = next(iter(train_data))
sample_query = train_data[sample_qid]

print("Sample query ID:", sample_qid)
print("Query fields:", list(sample_query.keys()))

relevant = sample_query.get("relevant_candidates", [])

print("\nRelevant candidates type:", type(relevant))
print("Relevant candidates count:", len(relevant))
print("First 10 relevant IDs:", relevant[:10])

print("\nDataset verification:")
print("Total queries:", len(train_data))
print("Total candidates:", len(train_candidates))

queries_with_positives = sum(
    bool(record.get("relevant_candidates", []))
    for record in train_data.values()
)

print("Queries with positives:", queries_with_positives)
print("Queries without positives:", len(train_data) - queries_with_positives)


Sample query ID: 0000000599
Query fields: ['text', 'relevant_candidates']

Relevant candidates type: <class 'list'>
Relevant candidates count: 2
First 10 relevant IDs: ['0001292810', '0001742283']

Dataset verification:
Total queries: 827
Total candidates: 4320
Queries with positives: 817
Queries without positives: 10


In [16]:

import random
from collections import defaultdict

random.seed(42)

training_pairs = []
positive_count = 0
negative_count = 0
skipped_queries = 0

candidate_ids = set(train_candidates.keys())

for qid, query_record in train_data.items():

    positive_ids = query_record.get("relevant_candidates", [])

    if not positive_ids:
        skipped_queries += 1
        continue

    # Validate positive candidate IDs
    valid_positives = [
        cid for cid in positive_ids
        if cid in candidate_ids and cid != qid
    ]

    # Add all positive pairs
    for cid in valid_positives:
        training_pairs.append({
            "query_id": qid,
            "candidate_id": cid,
            "label": 1
        })
        positive_count += 1

    # Exclude query itself and all positive candidates
    excluded_ids = set(valid_positives)
    excluded_ids.add(qid)

    available_negatives = list(candidate_ids - excluded_ids)

    # Sample four negatives, as in the original experiment
    sampled_negatives = random.sample(
        available_negatives,
        min(4, len(available_negatives))
    )

    for cid in sampled_negatives:
        training_pairs.append({
            "query_id": qid,
            "candidate_id": cid,
            "label": 0
        })
        negative_count += 1

print("TRAINING PAIRS RECREATED")
print("------------------------")
print("Positive pairs:", positive_count)
print("Negative pairs:", negative_count)
print("Total pairs:", len(training_pairs))
print("Skipped queries:", skipped_queries)

# Group pairs by query
pairs_by_query = defaultdict(list)

for pair in training_pairs:
    pairs_by_query[pair["query_id"]].append(pair)

print("\nQuery groups:", len(pairs_by_query))
print("First query group size:", len(next(iter(pairs_by_query.values()))))


TRAINING PAIRS RECREATED
------------------------
Positive pairs: 5308
Negative pairs: 3268
Total pairs: 8576
Skipped queries: 10

Query groups: 817
First query group size: 6


In [17]:

import random
import json
from collections import defaultdict

random.seed(42)

candidate_ids = list(train_candidates.keys())

training_pairs = []
positive_count = 0
negative_count = 0
skipped_queries = 0

for qid, query_record in train_data.items():

    positives = query_record.get("relevant_candidates", [])

    if not positives:
        skipped_queries += 1
        continue

    valid_positives = [
        cid for cid in positives
        if cid in train_candidates and cid != qid
    ]

    for cid in valid_positives:
        training_pairs.append({
            "query_id": qid,
            "candidate_id": cid,
            "label": 1
        })
        positive_count += 1

    excluded = set(valid_positives)
    excluded.add(qid)

    available_negatives = [
        cid for cid in candidate_ids
        if cid not in excluded
    ]

    sampled_negatives = random.sample(
        available_negatives,
        min(4, len(available_negatives))
    )

    for cid in sampled_negatives:
        training_pairs.append({
            "query_id": qid,
            "candidate_id": cid,
            "label": 0
        })
        negative_count += 1

pairs_by_query = defaultdict(list)

for pair in training_pairs:
    pairs_by_query[pair["query_id"]].append(pair)

PAIR_PATH = "/kaggle/working/phase7_exp2_training_pairs.json"

with open(PAIR_PATH, "w", encoding="utf-8") as f:
    json.dump(training_pairs, f)

print("Pairs:", len(training_pairs))
print("Positive:", positive_count)
print("Negative:", negative_count)
print("Query groups:", len(pairs_by_query))
print("Saved:", PAIR_PATH)


Pairs: 8576
Positive: 5308
Negative: 3268
Query groups: 817
Saved: /kaggle/working/phase7_exp2_training_pairs.json


In [18]:

import os
import pickle
from tokenizers import Tokenizer, decoders

TOKENIZER_PATH = (
    "/kaggle/input/datasets/aq1de3/tokenizer/tokenizer.json"
)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.decoder = decoders.ByteLevel()

MAX_SEQ_LEN = 1024
CONTENT_LIMIT = MAX_SEQ_LEN - 2  # Reserve BOS and EOS
BOS_ID = 2
EOS_ID = 3

def chunk_document(sentences, tokenizer):
    """Tokenize a document while preserving sentence boundaries."""
    chunks = []
    current_chunk = []

    for sentence in sentences:
        sentence_ids = tokenizer.encode(str(sentence)).ids

        if not sentence_ids:
            continue

        position = 0

        while position < len(sentence_ids):
            available = CONTENT_LIMIT - len(current_chunk)

            if available == 0:
                chunks.append(
                    [BOS_ID] + current_chunk + [EOS_ID]
                )
                current_chunk = []
                available = CONTENT_LIMIT

            take = min(available, len(sentence_ids) - position)

            current_chunk.extend(
                sentence_ids[position:position + take]
            )
            position += take

            if len(current_chunk) == CONTENT_LIMIT:
                chunks.append(
                    [BOS_ID] + current_chunk + [EOS_ID]
                )
                current_chunk = []

    if current_chunk:
        chunks.append([BOS_ID] + current_chunk + [EOS_ID])

    if not chunks:
        chunks = [[BOS_ID, EOS_ID]]

    return chunks


print("Building query token cache...")
train_query_token_cache = {}

for i, (qid, record) in enumerate(train_data.items(), 1):
    train_query_token_cache[qid] = chunk_document(
        record["text"], tokenizer
    )

    if i % 100 == 0:
        print(f"Queries processed: {i}/{len(train_data)}")


print("\nBuilding candidate token cache...")
train_candidate_token_cache = {}

for i, (cid, record) in enumerate(train_candidates.items(), 1):
    train_candidate_token_cache[cid] = chunk_document(
        record["text"], tokenizer
    )

    if i % 500 == 0:
        print(f"Candidates processed: {i}/{len(train_candidates)}")


restored_cache = {
    "queries": train_query_token_cache,
    "candidates": train_candidate_token_cache
}

CACHE_PATH = "/kaggle/working/phase7_train_token_cache.pkl"

with open(CACHE_PATH, "wb") as f:
    pickle.dump(restored_cache, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\nTOKEN CACHE REBUILT")
print("Queries:", len(restored_cache["queries"]))
print("Candidates:", len(restored_cache["candidates"]))
print("Sample query chunks:",
      len(restored_cache["queries"]["0000000599"]))
print("Sample chunk length:",
      len(restored_cache["queries"]["0000000599"][0]))
print("Cache saved:", CACHE_PATH)
print("Cache size (MB):",
      round(os.path.getsize(CACHE_PATH) / (1024**2), 2))


Building query token cache...
Queries processed: 100/827
Queries processed: 200/827
Queries processed: 300/827
Queries processed: 400/827
Queries processed: 500/827
Queries processed: 600/827
Queries processed: 700/827
Queries processed: 800/827

Building candidate token cache...
Candidates processed: 500/4320
Candidates processed: 1000/4320
Candidates processed: 1500/4320
Candidates processed: 2000/4320
Candidates processed: 2500/4320
Candidates processed: 3000/4320
Candidates processed: 3500/4320
Candidates processed: 4000/4320

TOKEN CACHE REBUILT
Queries: 827
Candidates: 4320
Sample query chunks: 8
Sample chunk length: 1024
Cache saved: /kaggle/working/phase7_train_token_cache.pkl
Cache size (MB): 107.71


In [19]:

import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.checkpoint import checkpoint

CHUNK_BATCH_SIZE = 4

def encode_document_fast(chunks, encoder):
    """
    Encode document chunks using A4 encoder.
    Mean-pool valid tokens, average chunk embeddings,
    and L2-normalize the final document embedding.
    """
    chunk_embeddings = []

    for start in range(0, len(chunks), CHUNK_BATCH_SIZE):
        batch_chunks = chunks[start:start + CHUNK_BATCH_SIZE]

        tensors = [
            torch.tensor(chunk, dtype=torch.long)
            for chunk in batch_chunks
        ]

        input_ids = pad_sequence(
            tensors,
            batch_first=True,
            padding_value=0
        ).to(device)

        attention_mask = (input_ids != 0).long()

        def forward_encoder(ids, mask):
            return encoder(ids, mask)

        use_checkpoint = len(chunks) > 8

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            if use_checkpoint and torch.is_grad_enabled():
                hidden = checkpoint(
                    forward_encoder,
                    input_ids,
                    attention_mask,
                    use_reentrant=False
                )
            else:
                hidden = encoder(input_ids, attention_mask)

        hidden = hidden.float()

        # Exclude PAD, BOS and EOS from mean pooling
        valid_mask = (
            (input_ids != 0) &
            (input_ids != BOS_ID) &
            (input_ids != EOS_ID)
        ).float()

        denominator = valid_mask.sum(dim=1, keepdim=True).clamp(min=1)

        pooled = (
            hidden * valid_mask.unsqueeze(-1)
        ).sum(dim=1) / denominator

        chunk_embeddings.append(pooled)

    document_embedding = torch.cat(
        chunk_embeddings, dim=0
    ).mean(dim=0)

    return F.normalize(document_embedding, p=2, dim=0)


def contrastive_loss(
    query_emb,
    candidate_emb,
    labels,
    margin=0.5
):
    similarity = F.cosine_similarity(
        query_emb, candidate_emb, dim=1
    )

    positive_loss = labels * (1 - similarity).pow(2)

    negative_loss = (
        (1 - labels) *
        torch.clamp(similarity - margin, min=0).pow(2)
    )

    loss = (positive_loss + negative_loss).mean()

    return loss, similarity


print("Encoding function:", callable(encode_document_fast))
print("Loss function:", callable(contrastive_loss))
print("Chunk batch size:", CHUNK_BATCH_SIZE)
print("Device:", device)


Encoding function: True
Loss function: True
Chunk batch size: 4
Device: cuda


In [20]:

required_objects = [
    "encoder",
    "optimizer",
    "training_pairs",
    "pairs_by_query",
    "restored_cache",
    "encode_document_fast",
    "contrastive_loss",
    "global_step",
    "epoch_history"
]

print("EXPERIMENT 2 RECOVERY CHECK\n")

for name in required_objects:
    print(f"{name}: {'READY' if name in globals() else 'MISSING'}")

print("\nTraining pairs:", len(training_pairs))
print("Query groups:", len(pairs_by_query))
print("Cached queries:", len(restored_cache["queries"]))
print("Cached candidates:", len(restored_cache["candidates"]))
print("Global step:", global_step)
print("Next epoch: 6")


EXPERIMENT 2 RECOVERY CHECK

encoder: READY
optimizer: READY
training_pairs: READY
pairs_by_query: READY
restored_cache: READY
encode_document_fast: READY
contrastive_loss: READY
global_step: READY
epoch_history: READY

Training pairs: 8576
Query groups: 817
Cached queries: 827
Cached candidates: 4320
Global step: 12435
Next epoch: 6


In [21]:

import torch
import time

encoder.train()
optimizer.zero_grad(set_to_none=True)

# Select one query with both positive and negative examples
selected_qid = next(
    qid for qid, group in pairs_by_query.items()
    if any(p["label"] == 1 for p in group)
    and any(p["label"] == 0 for p in group)
)

group = pairs_by_query[selected_qid]

positive_pair = next(p for p in group if p["label"] == 1)
negative_pair = next(p for p in group if p["label"] == 0)

test_pairs = [positive_pair, negative_pair]

print("Test query:", selected_qid)
print("Positive candidate:", positive_pair["candidate_id"])
print("Negative candidate:", negative_pair["candidate_id"])

if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

start_time = time.time()

# Encode query once
query_chunks = restored_cache["queries"][selected_qid]
query_embedding = encode_document_fast(query_chunks, encoder)

# Encode positive and negative candidates
candidate_embeddings = []

for pair in test_pairs:
    cid = pair["candidate_id"]
    chunks = restored_cache["candidates"][cid]
    candidate_embeddings.append(
        encode_document_fast(chunks, encoder)
    )

query_batch = query_embedding.unsqueeze(0).expand(2, -1)
candidate_batch = torch.stack(candidate_embeddings)

labels = torch.tensor(
    [1.0, 0.0],
    dtype=torch.float32,
    device=device
)

loss, similarities = contrastive_loss(
    query_batch,
    candidate_batch,
    labels
)

loss.backward()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.time() - start_time

# Gradient checks
layer4_has_grad = any(
    p.grad is not None and torch.isfinite(p.grad).all()
    for p in encoder.encoder_layers[3].parameters()
)

layer1_has_grad = any(
    p.grad is not None
    for p in encoder.encoder_layers[0].parameters()
)

print("\nSMOKE TEST RESULTS")
print("------------------")
print("Query embedding shape:", tuple(query_embedding.shape))
print("Candidate batch shape:", tuple(candidate_batch.shape))
print("Similarities:", similarities.detach().cpu().tolist())
print("Loss:", loss.item())
print("Layer 4 gradients:", layer4_has_grad)
print("Layer 1 gradients (should be False):", layer1_has_grad)
print("Elapsed time:", round(elapsed, 2), "seconds")

if device.type == "cuda":
    print(
        "Peak GPU memory:",
        round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "GB"
    )

# Clear smoke-test gradients; do not update weights
optimizer.zero_grad(set_to_none=True)

assert query_embedding.shape == (512,)
assert candidate_batch.shape == (2, 512)
assert torch.isfinite(loss)
assert layer4_has_grad
assert not layer1_has_grad

print("\nSMOKE TEST: PASSED")
print("No optimizer update was performed.")


Test query: 0000000599
Positive candidate: 0001292810
Negative candidate: 0000433120

SMOKE TEST RESULTS
------------------
Query embedding shape: (512,)
Candidate batch shape: (2, 512)
Similarities: [0.9424790143966675, 0.7270697951316833]
Loss: 0.027434678748250008
Layer 4 gradients: True
Layer 1 gradients (should be False): False
Elapsed time: 1.36 seconds
Peak GPU memory: 4.23 GB

SMOKE TEST: PASSED
No optimizer update was performed.


In [22]:

import os
import random
import time
import torch

START_EPOCH = 6
END_EPOCH = 6
PAIR_BATCH_SIZE = 4

CHECKPOINT_DIR = "/kaggle/working/phase7_exp2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for epoch in range(START_EPOCH, END_EPOCH + 1):

    encoder.train()
    epoch_start = time.time()

    # Shuffle query order reproducibly for this epoch
    query_ids = list(pairs_by_query.keys())
    random.Random(42 + epoch).shuffle(query_ids)

    total_loss = 0.0
    total_pairs = 0
    epoch_steps = 0

    print(f"\n========== EPOCH {epoch}/10 ==========")

    for qid in query_ids:

        query_pairs = list(pairs_by_query[qid])
        random.shuffle(query_pairs)

        # Process up to four pairs from this query at a time
        for start in range(0, len(query_pairs), PAIR_BATCH_SIZE):

            batch_pairs = query_pairs[
                start:start + PAIR_BATCH_SIZE
            ]

            optimizer.zero_grad(set_to_none=True)

            # Encode query once for this batch
            query_chunks = restored_cache["queries"][qid]
            query_embedding = encode_document_fast(
                query_chunks, encoder
            )

            candidate_embeddings = []
            labels_list = []

            for pair in batch_pairs:
                cid = pair["candidate_id"]

                candidate_chunks = (
                    restored_cache["candidates"][cid]
                )

                candidate_embedding = encode_document_fast(
                    candidate_chunks, encoder
                )

                candidate_embeddings.append(candidate_embedding)
                labels_list.append(float(pair["label"]))

            query_batch = query_embedding.unsqueeze(0).expand(
                len(batch_pairs), -1
            )

            candidate_batch = torch.stack(candidate_embeddings)

            labels = torch.tensor(
                labels_list,
                dtype=torch.float32,
                device=device
            )

            loss, similarities = contrastive_loss(
                query_batch,
                candidate_batch,
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                [p for p in encoder.parameters() if p.requires_grad],
                max_norm=1.0
            )

            optimizer.step()

            batch_size = len(batch_pairs)
            total_loss += loss.item() * batch_size
            total_pairs += batch_size
            epoch_steps += 1
            global_step += 1

            if epoch_steps % 100 == 0:
                print(
                    f"Epoch {epoch} | "
                    f"Steps {epoch_steps} | "
                    f"Pairs {total_pairs}/{len(training_pairs)} | "
                    f"Avg loss {total_loss / total_pairs:.6f}"
                )

    avg_loss = total_loss / total_pairs
    elapsed_minutes = (time.time() - epoch_start) / 60

    epoch_result = {
        "epoch": epoch,
        "avg_loss": avg_loss,
        "pairs": total_pairs,
        "steps": epoch_steps,
        "time_minutes": elapsed_minutes
    }

    epoch_history.append(epoch_result)

    # Save only the completed epoch checkpoint
    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"exp2_epoch_{epoch}.pt"
    )

    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": encoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch_history": epoch_history
    }, checkpoint_path)

    print(f"\n========== EPOCH {epoch} COMPLETE ==========")
    print("Average loss:", round(avg_loss, 6))
    print("Pairs processed:", total_pairs)
    print("Optimizer steps:", epoch_steps)
    print("Elapsed minutes:", round(elapsed_minutes, 2))
    print("Global step:", global_step)
    print("Checkpoint saved:", checkpoint_path)



========== EPOCH 6/10 ==========
Epoch 6 | Steps 100 | Pairs 351/8576 | Avg loss 0.028586
Epoch 6 | Steps 200 | Pairs 684/8576 | Avg loss 0.028424
Epoch 6 | Steps 300 | Pairs 1016/8576 | Avg loss 0.027286
Epoch 6 | Steps 400 | Pairs 1354/8576 | Avg loss 0.027973
Epoch 6 | Steps 500 | Pairs 1688/8576 | Avg loss 0.026613
Epoch 6 | Steps 600 | Pairs 2045/8576 | Avg loss 0.026530
Epoch 6 | Steps 700 | Pairs 2387/8576 | Avg loss 0.028759
Epoch 6 | Steps 800 | Pairs 2740/8576 | Avg loss 0.029074
Epoch 6 | Steps 900 | Pairs 3095/8576 | Avg loss 0.029618
Epoch 6 | Steps 1000 | Pairs 3434/8576 | Avg loss 0.030457
Epoch 6 | Steps 1100 | Pairs 3771/8576 | Avg loss 0.030919
Epoch 6 | Steps 1200 | Pairs 4129/8576 | Avg loss 0.031040
Epoch 6 | Steps 1300 | Pairs 4475/8576 | Avg loss 0.030922
Epoch 6 | Steps 1400 | Pairs 4816/8576 | Avg loss 0.031343
Epoch 6 | Steps 1500 | Pairs 5168/8576 | Avg loss 0.031445
Epoch 6 | Steps 1600 | Pairs 5503/8576 | Avg loss 0.031344
Epoch 6 | Steps 1700 | Pairs 5854

In [28]:

START_EPOCH = 7
END_EPOCH = 7
PAIR_BATCH_SIZE = 4

for epoch in range(START_EPOCH, END_EPOCH + 1):

    encoder.train()
    epoch_start = time.time()

    query_ids = list(pairs_by_query.keys())
    random.Random(42 + epoch).shuffle(query_ids)

    total_loss = 0.0
    total_pairs = 0
    epoch_steps = 0

    print(f"\n========== EPOCH {epoch}/10 ==========")

    for qid in query_ids:

        query_pairs = list(pairs_by_query[qid])
        random.Random(42 + epoch + hash(qid) % 100000).shuffle(query_pairs)

        for start in range(0, len(query_pairs), PAIR_BATCH_SIZE):

            batch_pairs = query_pairs[
                start:start + PAIR_BATCH_SIZE
            ]

            optimizer.zero_grad(set_to_none=True)

            query_chunks = restored_cache["queries"][qid]
            query_embedding = encode_document_fast(
                query_chunks, encoder
            )

            candidate_embeddings = []
            labels_list = []

            for pair in batch_pairs:
                cid = pair["candidate_id"]
                candidate_chunks = restored_cache["candidates"][cid]

                candidate_embedding = encode_document_fast(
                    candidate_chunks, encoder
                )

                candidate_embeddings.append(candidate_embedding)
                labels_list.append(float(pair["label"]))

            query_batch = query_embedding.unsqueeze(0).expand(
                len(batch_pairs), -1
            )

            candidate_batch = torch.stack(candidate_embeddings)

            labels = torch.tensor(
                labels_list,
                dtype=torch.float32,
                device=device
            )

            loss, similarities = contrastive_loss(
                query_batch,
                candidate_batch,
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                [p for p in encoder.parameters() if p.requires_grad],
                max_norm=1.0
            )

            optimizer.step()

            batch_size = len(batch_pairs)
            total_loss += loss.item() * batch_size
            total_pairs += batch_size
            epoch_steps += 1
            global_step += 1

            if epoch_steps % 100 == 0:
                print(
                    f"Epoch {epoch} | "
                    f"Steps {epoch_steps} | "
                    f"Pairs {total_pairs}/8576 | "
                    f"Avg loss {total_loss / total_pairs:.6f}"
                )

    avg_loss = total_loss / total_pairs
    elapsed_minutes = (time.time() - epoch_start) / 60

    epoch_result = {
        "epoch": epoch,
        "avg_loss": avg_loss,
        "pairs": total_pairs,
        "steps": epoch_steps,
        "time_minutes": elapsed_minutes
    }

    epoch_history.append(epoch_result)

    checkpoint_path = (
        f"{CHECKPOINT_DIR}/exp2_epoch_{epoch}.pt"
    )

    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": encoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch_history": epoch_history
    }, checkpoint_path)

    print(f"\n========== EPOCH {epoch} COMPLETE ==========")
    print("Average loss:", round(avg_loss, 6))
    print("Pairs processed:", total_pairs)
    print("Optimizer steps:", epoch_steps)
    print("Elapsed minutes:", round(elapsed_minutes, 2))
    print("Global step:", global_step)
    print("Checkpoint saved:", checkpoint_path)

    assert total_pairs == 8576
    print("EPOCH CHECK: PASSED")



========== EPOCH 7/10 ==========
Epoch 7 | Steps 100 | Pairs 330/8576 | Avg loss 0.035414
Epoch 7 | Steps 200 | Pairs 682/8576 | Avg loss 0.035148
Epoch 7 | Steps 300 | Pairs 1035/8576 | Avg loss 0.032245
Epoch 7 | Steps 400 | Pairs 1374/8576 | Avg loss 0.031798
Epoch 7 | Steps 500 | Pairs 1711/8576 | Avg loss 0.031978
Epoch 7 | Steps 600 | Pairs 2063/8576 | Avg loss 0.031549
Epoch 7 | Steps 700 | Pairs 2403/8576 | Avg loss 0.031816
Epoch 7 | Steps 800 | Pairs 2766/8576 | Avg loss 0.031459
Epoch 7 | Steps 900 | Pairs 3106/8576 | Avg loss 0.030784
Epoch 7 | Steps 1000 | Pairs 3459/8576 | Avg loss 0.030679
Epoch 7 | Steps 1100 | Pairs 3804/8576 | Avg loss 0.030582
Epoch 7 | Steps 1200 | Pairs 4142/8576 | Avg loss 0.030687
Epoch 7 | Steps 1300 | Pairs 4492/8576 | Avg loss 0.030258
Epoch 7 | Steps 1400 | Pairs 4835/8576 | Avg loss 0.030075
Epoch 7 | Steps 1500 | Pairs 5164/8576 | Avg loss 0.030256
Epoch 7 | Steps 1600 | Pairs 5508/8576 | Avg loss 0.030045
Epoch 7 | Steps 1700 | Pairs 5862

In [29]:

START_EPOCH = 8
END_EPOCH = 8
PAIR_BATCH_SIZE = 4

for epoch in range(START_EPOCH, END_EPOCH + 1):

    encoder.train()
    epoch_start = time.time()

    query_ids = list(pairs_by_query.keys())
    random.Random(42 + epoch).shuffle(query_ids)

    total_loss = 0.0
    total_pairs = 0
    epoch_steps = 0

    print(f"\n========== EPOCH {epoch}/10 ==========")

    for qid in query_ids:

        query_pairs = list(pairs_by_query[qid])
        random.Random(42 + epoch).shuffle(query_pairs)

        for start in range(0, len(query_pairs), PAIR_BATCH_SIZE):

            batch_pairs = query_pairs[start:start + PAIR_BATCH_SIZE]

            optimizer.zero_grad(set_to_none=True)

            query_embedding = encode_document_fast(
                restored_cache["queries"][qid], encoder
            )

            candidate_embeddings = []
            labels_list = []

            for pair in batch_pairs:
                cid = pair["candidate_id"]

                candidate_embedding = encode_document_fast(
                    restored_cache["candidates"][cid], encoder
                )

                candidate_embeddings.append(candidate_embedding)
                labels_list.append(float(pair["label"]))

            query_batch = query_embedding.unsqueeze(0).expand(
                len(batch_pairs), -1
            )

            candidate_batch = torch.stack(candidate_embeddings)

            labels = torch.tensor(
                labels_list, dtype=torch.float32, device=device
            )

            loss, similarities = contrastive_loss(
                query_batch, candidate_batch, labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                [p for p in encoder.parameters() if p.requires_grad],
                max_norm=1.0
            )

            optimizer.step()

            batch_size = len(batch_pairs)
            total_loss += loss.item() * batch_size
            total_pairs += batch_size
            epoch_steps += 1
            global_step += 1

            if epoch_steps % 100 == 0:
                print(
                    f"Epoch {epoch} | Steps {epoch_steps} | "
                    f"Pairs {total_pairs}/8576 | "
                    f"Avg loss {total_loss / total_pairs:.6f}"
                )

    avg_loss = total_loss / total_pairs
    elapsed_minutes = (time.time() - epoch_start) / 60

    epoch_result = {
        "epoch": epoch,
        "avg_loss": avg_loss,
        "pairs": total_pairs,
        "steps": epoch_steps,
        "time_minutes": elapsed_minutes
    }

    epoch_history.append(epoch_result)

    checkpoint_path = (
        f"{CHECKPOINT_DIR}/exp2_epoch_{epoch}.pt"
    )

    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": encoder.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch_history": epoch_history
    }, checkpoint_path)

    print(f"\n========== EPOCH {epoch} COMPLETE ==========")
    print("Average loss:", round(avg_loss, 6))
    print("Pairs processed:", total_pairs)
    print("Optimizer steps:", epoch_steps)
    print("Elapsed minutes:", round(elapsed_minutes, 2))
    print("Global step:", global_step)
    print("Checkpoint saved:", checkpoint_path)

    assert total_pairs == 8576
    print("EPOCH CHECK: PASSED")



========== EPOCH 8/10 ==========
Epoch 8 | Steps 100 | Pairs 336/8576 | Avg loss 0.029634
Epoch 8 | Steps 200 | Pairs 672/8576 | Avg loss 0.030020
Epoch 8 | Steps 300 | Pairs 1023/8576 | Avg loss 0.029186
Epoch 8 | Steps 400 | Pairs 1370/8576 | Avg loss 0.029868
Epoch 8 | Steps 500 | Pairs 1729/8576 | Avg loss 0.028755
Epoch 8 | Steps 600 | Pairs 2072/8576 | Avg loss 0.028252
Epoch 8 | Steps 700 | Pairs 2412/8576 | Avg loss 0.028149
Epoch 8 | Steps 800 | Pairs 2744/8576 | Avg loss 0.028489
Epoch 8 | Steps 900 | Pairs 3094/8576 | Avg loss 0.028046
Epoch 8 | Steps 1000 | Pairs 3436/8576 | Avg loss 0.028002
Epoch 8 | Steps 1100 | Pairs 3791/8576 | Avg loss 0.027722
Epoch 8 | Steps 1200 | Pairs 4133/8576 | Avg loss 0.027950
Epoch 8 | Steps 1300 | Pairs 4482/8576 | Avg loss 0.029016
Epoch 8 | Steps 1400 | Pairs 4816/8576 | Avg loss 0.029258
Epoch 8 | Steps 1500 | Pairs 5157/8576 | Avg loss 0.029351
Epoch 8 | Steps 1600 | Pairs 5503/8576 | Avg loss 0.029277
Epoch 8 | Steps 1700 | Pairs 5850

In [30]:

# ============================================================
# PHASE 7 - EXPERIMENT 2
# EPOCH 9 / 10 - PARTIAL LAYER FINE-TUNING
# ============================================================

import os
import time
import random
import torch

# ---------------- CONFIGURATION ----------------

START_EPOCH = 9
END_EPOCH = 9

PAIR_BATCH_SIZE = 4
MAX_GRAD_NORM = 1.0

CHECKPOINT_DIR = "/kaggle/working/phase7_exp2_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ---------------- SAFETY CHECKS ----------------

assert encoder is not None
assert optimizer is not None
assert len(training_pairs) == 8576
assert len(pairs_by_query) == 817
assert len(restored_cache["queries"]) == 827
assert len(restored_cache["candidates"]) == 4320

assert os.path.exists(
    os.path.join(CHECKPOINT_DIR, "exp2_epoch_8.pt")
), "Epoch 8 checkpoint not found!"

print("EXPERIMENT 2 TRAINING")
print("---------------------")
print("Starting epoch:", START_EPOCH)
print("Global step:", global_step)
print("Training pairs:", len(training_pairs))
print("Query groups:", len(pairs_by_query))
print("Device:", device)
print("Trainable layers: 4, 5, 6")

# ---------------- EPOCH TRAINING ----------------

for epoch in range(START_EPOCH, END_EPOCH + 1):

    encoder.train()

    epoch_start = time.time()

    query_ids = list(pairs_by_query.keys())

    # Reproducible query ordering
    random.Random(42 + epoch).shuffle(query_ids)

    total_loss = 0.0
    total_pairs = 0
    epoch_steps = 0

    print(f"\n========== EPOCH {epoch}/10 ==========")

    for qid in query_ids:

        query_pairs = list(pairs_by_query[qid])

        # Shuffle pairs within each query
        random.Random(42 + epoch).shuffle(query_pairs)

        for start in range(
            0, len(query_pairs), PAIR_BATCH_SIZE
        ):

            batch_pairs = query_pairs[
                start:start + PAIR_BATCH_SIZE
            ]

            optimizer.zero_grad(set_to_none=True)

            # Encode query document once
            query_chunks = restored_cache["queries"][qid]

            query_embedding = encode_document_fast(
                query_chunks,
                encoder
            )

            candidate_embeddings = []
            labels_list = []

            # Encode candidate documents
            for pair in batch_pairs:

                cid = pair["candidate_id"]

                candidate_chunks = (
                    restored_cache["candidates"][cid]
                )

                candidate_embedding = encode_document_fast(
                    candidate_chunks,
                    encoder
                )

                candidate_embeddings.append(
                    candidate_embedding
                )

                labels_list.append(
                    float(pair["label"])
                )

            # Prepare embedding batches
            query_batch = query_embedding.unsqueeze(0).expand(
                len(batch_pairs), -1
            )

            candidate_batch = torch.stack(
                candidate_embeddings
            )

            labels = torch.tensor(
                labels_list,
                dtype=torch.float32,
                device=device
            )

            # Contrastive loss
            loss, similarities = contrastive_loss(
                query_batch,
                candidate_batch,
                labels
            )

            # Backpropagation
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(
                [
                    p for p in encoder.parameters()
                    if p.requires_grad
                ],
                max_norm=MAX_GRAD_NORM
            )

            # Optimizer update
            optimizer.step()

            # Update statistics
            batch_size = len(batch_pairs)

            total_loss += loss.item() * batch_size
            total_pairs += batch_size

            epoch_steps += 1
            global_step += 1

            # Progress every 100 steps
            if epoch_steps % 100 == 0:

                average_loss = (
                    total_loss / total_pairs
                )

                print(
                    f"Epoch {epoch} | "
                    f"Steps {epoch_steps} | "
                    f"Pairs {total_pairs}/8576 | "
                    f"Avg loss {average_loss:.6f}"
                )

    # ---------------- EPOCH SUMMARY ----------------

    avg_loss = total_loss / total_pairs

    elapsed_minutes = (
        time.time() - epoch_start
    ) / 60

    epoch_result = {
        "epoch": epoch,
        "avg_loss": avg_loss,
        "pairs": total_pairs,
        "steps": epoch_steps,
        "time_minutes": elapsed_minutes
    }

    epoch_history.append(epoch_result)

    # ---------------- SAVE CHECKPOINT ----------------

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"exp2_epoch_{epoch}.pt"
    )

    torch.save(
        {
            "epoch": epoch,
            "global_step": global_step,
            "model_state_dict": encoder.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch_history": epoch_history
        },
        checkpoint_path
    )

    # ---------------- FINAL OUTPUT ----------------

    print(
        f"\n========== EPOCH {epoch} COMPLETE =========="
    )

    print("Average loss:", round(avg_loss, 6))
    print("Pairs processed:", total_pairs)
    print("Optimizer steps:", epoch_steps)
    print("Elapsed minutes:", round(elapsed_minutes, 2))
    print("Global step:", global_step)
    print("Checkpoint saved:", checkpoint_path)

    assert total_pairs == 8576
    assert os.path.exists(checkpoint_path)

    print("EPOCH CHECK: PASSED")

print("\nEXPERIMENT 2 EPOCH 9 FINISHED")


EXPERIMENT 2 TRAINING
---------------------
Starting epoch: 9
Global step: 19896
Training pairs: 8576
Query groups: 817
Device: cuda
Trainable layers: 4, 5, 6

========== EPOCH 9/10 ==========
Epoch 9 | Steps 100 | Pairs 356/8576 | Avg loss 0.027876
Epoch 9 | Steps 200 | Pairs 709/8576 | Avg loss 0.026664
Epoch 9 | Steps 300 | Pairs 1064/8576 | Avg loss 0.028926
Epoch 9 | Steps 400 | Pairs 1408/8576 | Avg loss 0.028175
Epoch 9 | Steps 500 | Pairs 1757/8576 | Avg loss 0.027970
Epoch 9 | Steps 600 | Pairs 2113/8576 | Avg loss 0.028006
Epoch 9 | Steps 700 | Pairs 2470/8576 | Avg loss 0.028385
Epoch 9 | Steps 800 | Pairs 2815/8576 | Avg loss 0.028518
Epoch 9 | Steps 900 | Pairs 3136/8576 | Avg loss 0.028232
Epoch 9 | Steps 1000 | Pairs 3484/8576 | Avg loss 0.027874
Epoch 9 | Steps 1100 | Pairs 3820/8576 | Avg loss 0.027825
Epoch 9 | Steps 1200 | Pairs 4151/8576 | Avg loss 0.028095
Epoch 9 | Steps 1300 | Pairs 4494/8576 | Avg loss 0.028124
Epoch 9 | Steps 1400 | Pairs 4840/8576 | Avg loss 0

In [31]:

import os
import json
import torch
import time

DATA_ROOT = "/kaggle/input/datasets/aq1de3/phase7"

DEV_PATH = os.path.join(DATA_ROOT, "dev (3).json")
DEV_CANDIDATES_PATH = os.path.join(
    DATA_ROOT, "dev_candidates.json"
)

print("Loading development queries...")
with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

print("Loading development candidates...")
with open(DEV_CANDIDATES_PATH, "r", encoding="utf-8") as f:
    dev_candidates = json.load(f)

print("\nDEVELOPMENT DATA")
print("Queries:", len(dev_data))
print("Candidates:", len(dev_candidates))

print("Queries with relevance labels:", sum(
    bool(x.get("relevant_candidates", []))
    for x in dev_data.values()
))

# Build token caches
print("\nBuilding development query token cache...")
dev_query_token_cache = {}

for i, (qid, record) in enumerate(dev_data.items(), 1):
    dev_query_token_cache[qid] = chunk_document(
        record["text"], tokenizer
    )

    if i % 25 == 0:
        print(f"Queries: {i}/{len(dev_data)}")

print("\nBuilding development candidate token cache...")
dev_candidate_token_cache = {}

for i, (cid, record) in enumerate(dev_candidates.items(), 1):
    dev_candidate_token_cache[cid] = chunk_document(
        record["text"], tokenizer
    )

    if i % 200 == 0:
        print(f"Candidates: {i}/{len(dev_candidates)}")

dev_cache = {
    "queries": dev_query_token_cache,
    "candidates": dev_candidate_token_cache
}

print("\nDEVELOPMENT CACHE READY")
print("Queries:", len(dev_cache["queries"]))
print("Candidates:", len(dev_cache["candidates"]))
print("Sample query chunks:", len(
    dev_cache["queries"][next(iter(dev_data))]
))


Loading development queries...
Loading development candidates...

DEVELOPMENT DATA
Queries: 118
Candidates: 1023
Queries with relevance labels: 118

Building development query token cache...
Queries: 25/118
Queries: 50/118
Queries: 75/118
Queries: 100/118

Building development candidate token cache...
Candidates: 200/1023
Candidates: 400/1023
Candidates: 600/1023
Candidates: 800/1023
Candidates: 1000/1023

DEVELOPMENT CACHE READY
Queries: 118
Candidates: 1023
Sample query chunks: 8


In [32]:

import os
import torch
import time

EPOCH = 9

CHECKPOINT_PATH = (
    f"/kaggle/working/phase7_exp2_checkpoints/"
    f"exp2_epoch_{EPOCH}.pt"
)

EVAL_DIR = "/kaggle/working/phase7_exp2_evaluation"
os.makedirs(EVAL_DIR, exist_ok=True)

assert os.path.exists(CHECKPOINT_PATH), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

# Load Epoch 9 weights
print("Loading Epoch 9 checkpoint...")

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

encoder.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

encoder.to(device)
encoder.eval()

print("Checkpoint epoch:", checkpoint["epoch"])
print("Global step:", checkpoint["global_step"])

# Generate query embeddings
dev_query_embeddings = {}

start_time = time.time()

print("\nEncoding development queries...")

with torch.inference_mode():

    for i, (qid, chunks) in enumerate(
        dev_cache["queries"].items(), 1
    ):

        embedding = encode_document_fast(
            chunks, encoder
        )

        dev_query_embeddings[qid] = (
            embedding.detach().cpu()
        )

        if i % 25 == 0:
            print(f"Queries: {i}/118")

# Generate candidate embeddings
dev_candidate_embeddings = {}

print("\nEncoding development candidates...")

with torch.inference_mode():

    for i, (cid, chunks) in enumerate(
        dev_cache["candidates"].items(), 1
    ):

        embedding = encode_document_fast(
            chunks, encoder
        )

        dev_candidate_embeddings[cid] = (
            embedding.detach().cpu()
        )

        if i % 100 == 0:
            print(f"Candidates: {i}/1023")

# Stack embeddings in dictionary order
query_ids = list(dev_query_embeddings.keys())
candidate_ids = list(dev_candidate_embeddings.keys())

query_matrix = torch.stack([
    dev_query_embeddings[qid] for qid in query_ids
])

candidate_matrix = torch.stack([
    dev_candidate_embeddings[cid] for cid in candidate_ids
])

# Validate
assert query_matrix.shape == (118, 512)
assert candidate_matrix.shape == (1023, 512)
assert torch.isfinite(query_matrix).all()
assert torch.isfinite(candidate_matrix).all()

# Save
embedding_path = os.path.join(
    EVAL_DIR, "epoch_9_dev_embeddings.pt"
)

torch.save({
    "epoch": 9,
    "query_ids": query_ids,
    "candidate_ids": candidate_ids,
    "query_embeddings": query_matrix,
    "candidate_embeddings": candidate_matrix
}, embedding_path)

print("\nEPOCH 9 EMBEDDINGS COMPLETE")
print("Query matrix:", tuple(query_matrix.shape))
print("Candidate matrix:", tuple(candidate_matrix.shape))
print("Elapsed minutes:", round(
    (time.time() - start_time) / 60, 2
))
print("Saved:", embedding_path)


Loading Epoch 9 checkpoint...
Checkpoint epoch: 9
Global step: 22383

Encoding development queries...
Queries: 25/118
Queries: 50/118
Queries: 75/118
Queries: 100/118

Encoding development candidates...
Candidates: 100/1023
Candidates: 200/1023
Candidates: 300/1023
Candidates: 400/1023
Candidates: 500/1023
Candidates: 600/1023
Candidates: 700/1023
Candidates: 800/1023
Candidates: 900/1023
Candidates: 1000/1023

EPOCH 9 EMBEDDINGS COMPLETE
Query matrix: (118, 512)
Candidate matrix: (1023, 512)
Elapsed minutes: 1.95
Saved: /kaggle/working/phase7_exp2_evaluation/epoch_9_dev_embeddings.pt


In [33]:

import torch
import json
import os
import numpy as np

# ------------------------------------------------
# COSINE SIMILARITY
# ------------------------------------------------

# Embeddings are already L2-normalized
similarity_matrix = query_matrix @ candidate_matrix.T

assert similarity_matrix.shape == (118, 1023)

# ------------------------------------------------
# TOP-7 RETRIEVAL
# ------------------------------------------------

K = 7

predictions = {}

total_tp = 0
total_fp = 0
total_fn = 0

all_relevant = 0

for i, qid in enumerate(query_ids):

    scores = similarity_matrix[i].clone()

    # Exclude query itself if it exists in candidate pool
    if qid in candidate_ids:
        self_index = candidate_ids.index(qid)
        scores[self_index] = -float("inf")

    # Retrieve top 7
    top_indices = torch.topk(
        scores,
        k=K,
        largest=True
    ).indices.tolist()

    retrieved_ids = [
        candidate_ids[idx] for idx in top_indices
    ]

    # Verify unique predictions
    assert len(retrieved_ids) == K
    assert len(set(retrieved_ids)) == K
    assert qid not in retrieved_ids

    predictions[qid] = retrieved_ids

    # Ground truth
    relevant_ids = set(
        dev_data[qid].get("relevant_candidates", [])
    )

    retrieved_set = set(retrieved_ids)

    tp = len(retrieved_set & relevant_ids)
    fp = K - tp
    fn = len(relevant_ids - retrieved_set)

    total_tp += tp
    total_fp += fp
    total_fn += fn

    all_relevant += len(relevant_ids)

# ------------------------------------------------
# METRICS
# ------------------------------------------------

micro_precision = (
    total_tp / (total_tp + total_fp)
    if total_tp + total_fp else 0.0
)

micro_recall = (
    total_tp / (total_tp + total_fn)
    if total_tp + total_fn else 0.0
)

micro_f1 = (
    2 * micro_precision * micro_recall /
    (micro_precision + micro_recall)
    if micro_precision + micro_recall else 0.0
)

# ------------------------------------------------
# SAVE PREDICTIONS
# ------------------------------------------------

PRED_PATH = (
    "/kaggle/working/phase7_exp2_evaluation/"
    "epoch_9_dev_predictions.json"
)

with open(PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)

# ------------------------------------------------
# OUTPUT
# ------------------------------------------------

print("\n========== EPOCH 9 DEVELOPMENT RESULTS ==========")

print("Queries:", len(query_ids))
print("Candidates:", len(candidate_ids))
print("K:", K)
print("Total relevant cases:", all_relevant)

print("\nConfusion counts:")
print("True Positives:", total_tp)
print("False Positives:", total_fp)
print("False Negatives:", total_fn)

print("\nMETRICS")
print("Micro-Precision@7:", round(micro_precision, 6))
print("Micro-Recall@7:   ", round(micro_recall, 6))
print("Micro-F1@7:       ", round(micro_f1, 6))

print("\nPrediction validation:")
print("Queries with predictions:", len(predictions))
print("Exactly 7 per query:", all(
    len(v) == 7 for v in predictions.values()
))
print("All predictions unique:", all(
    len(v) == len(set(v)) for v in predictions.values()
))

print("\nSaved:", PRED_PATH)



========== EPOCH 9 DEVELOPMENT RESULTS ==========
Queries: 118
Candidates: 1023
K: 7
Total relevant cases: 973

Confusion counts:
True Positives: 105
False Positives: 721
False Negatives: 868

METRICS
Micro-Precision@7: 0.127119
Micro-Recall@7:    0.107914
Micro-F1@7:        0.116732

Prediction validation:
Queries with predictions: 118
Exactly 7 per query: True
All predictions unique: True

Saved: /kaggle/working/phase7_exp2_evaluation/epoch_9_dev_predictions.json


In [34]:

import os
import torch
import json
import time

EVAL_DIR = "/kaggle/working/phase7_exp2_evaluation"

CHECKPOINTS = {
    5: "/kaggle/input/datasets/aq1de3/exp2-epoch-5/exp2_epoch_5 (1).pt",
    6: "/kaggle/working/phase7_exp2_checkpoints/exp2_epoch_6.pt",
    7: "/kaggle/working/phase7_exp2_checkpoints/exp2_epoch_7.pt",
    8: "/kaggle/working/phase7_exp2_checkpoints/exp2_epoch_8.pt",
}

results = []

def calculate_metrics(q_matrix, c_matrix, q_ids, c_ids):
    similarities = q_matrix @ c_matrix.T

    tp = fp = fn = 0
    predictions = {}

    for i, qid in enumerate(q_ids):
        scores = similarities[i].clone()

        if qid in c_ids:
            scores[c_ids.index(qid)] = -float("inf")

        indices = torch.topk(scores, k=7).indices.tolist()
        retrieved = [c_ids[j] for j in indices]

        predictions[qid] = retrieved

        relevant = set(
            dev_data[qid]["relevant_candidates"]
        )
        retrieved_set = set(retrieved)

        current_tp = len(relevant & retrieved_set)
        tp += current_tp
        fp += 7 - current_tp
        fn += len(relevant - retrieved_set)

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1, predictions


for epoch, checkpoint_path in CHECKPOINTS.items():

    print(f"\n========== EVALUATING EPOCH {epoch} ==========")

    assert os.path.exists(checkpoint_path), checkpoint_path

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    encoder.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True
    )

    encoder.to(device)
    encoder.eval()

    q_embeddings = []
    c_embeddings = []

    start_time = time.time()

    with torch.inference_mode():

        for qid in dev_cache["queries"]:
            emb = encode_document_fast(
                dev_cache["queries"][qid], encoder
            )
            q_embeddings.append(emb.cpu())

        for cid in dev_cache["candidates"]:
            emb = encode_document_fast(
                dev_cache["candidates"][cid], encoder
            )
            c_embeddings.append(emb.cpu())

    q_matrix = torch.stack(q_embeddings)
    c_matrix = torch.stack(c_embeddings)

    q_ids = list(dev_cache["queries"].keys())
    c_ids = list(dev_cache["candidates"].keys())

    precision, recall, f1, predictions = calculate_metrics(
        q_matrix, c_matrix, q_ids, c_ids
    )

    results.append({
        "epoch": epoch,
        "precision@7": precision,
        "recall@7": recall,
        "f1@7": f1
    })

    # Save predictions
    pred_path = os.path.join(
        EVAL_DIR, f"epoch_{epoch}_dev_predictions.json"
    )

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2)

    print("Precision@7:", round(precision, 6))
    print("Recall@7:", round(recall, 6))
    print("F1@7:", round(f1, 6))
    print("Elapsed minutes:", round(
        (time.time() - start_time) / 60, 2
    ))

# Include already evaluated Epoch 9
results.append({
    "epoch": 9,
    "precision@7": 0.127119,
    "recall@7": 0.107914,
    "f1@7": 0.116732
})

results.sort(key=lambda x: x["epoch"])

print("\n========== CHECKPOINT COMPARISON ==========")

for result in results:
    print(result)

best = max(results, key=lambda x: x["f1@7"])

print("\nBEST CHECKPOINT:", best["epoch"])
print("BEST MICRO-F1@7:", round(best["f1@7"], 6))

with open(
    os.path.join(EVAL_DIR, "checkpoint_comparison.json"),
    "w"
) as f:
    json.dump(results, f, indent=2)



========== EVALUATING EPOCH 5 ==========
Precision@7: 0.158596
Recall@7: 0.134635
F1@7: 0.145636
Elapsed minutes: 1.94

========== EVALUATING EPOCH 6 ==========
Precision@7: 0.122276
Recall@7: 0.103803
F1@7: 0.112285
Elapsed minutes: 1.92

========== EVALUATING EPOCH 7 ==========
Precision@7: 0.135593
Recall@7: 0.115108
F1@7: 0.124514
Elapsed minutes: 1.92

========== EVALUATING EPOCH 8 ==========
Precision@7: 0.130751
Recall@7: 0.110997
F1@7: 0.120067
Elapsed minutes: 1.92

========== CHECKPOINT COMPARISON ==========
{'epoch': 5, 'precision@7': 0.1585956416464891, 'recall@7': 0.13463514902363824, 'f1@7': 0.14563646470261254}
{'epoch': 6, 'precision@7': 0.12227602905569007, 'recall@7': 0.10380267214799589, 'f1@7': 0.1122846025569761}
{'epoch': 7, 'precision@7': 0.13559322033898305, 'recall@7': 0.11510791366906475, 'f1@7': 0.12451361867704279}
{'epoch': 8, 'precision@7': 0.13075060532687652, 'recall@7': 0.11099691675231244, 'f1@7': 0.12006670372429128}
{'epoch': 9, 'precision@7': 0.127

In [35]:

import os
import json
import shutil

EVAL_DIR = "/kaggle/working/phase7_exp2_evaluation"
BEST_DIR = "/kaggle/working/phase7_exp2_final"

os.makedirs(BEST_DIR, exist_ok=True)

# Preserve the best checkpoint
source = "/kaggle/input/datasets/aq1de3/exp2-epoch-5/exp2_epoch_5 (1).pt"
destination = os.path.join(BEST_DIR, "phase7_exp2_best_epoch5.pt")

shutil.copy2(source, destination)

# Save comparison and selected metrics
summary = {
    "experiment": "Experiment 2 - Partial Layer-wise Fine-Tuning",
    "best_epoch": 5,
    "best_micro_precision_at_7": 0.1585956416464891,
    "best_micro_recall_at_7": 0.13463514902363824,
    "best_micro_f1_at_7": 0.14563646470261254,
    "baseline_micro_f1_at_7": 0.066704,
    "relative_f1_improvement_percent": 118.3,
    "selection_basis": "Highest development Micro-F1@7"
}

with open(os.path.join(BEST_DIR, "experiment2_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("BEST CHECKPOINT SAVED:", destination)
print("SUMMARY SAVED")
print("BEST MICRO-F1@7:", summary["best_micro_f1_at_7"])
print("EXPERIMENT 2: COMPLETE")


BEST CHECKPOINT SAVED: /kaggle/working/phase7_exp2_final/phase7_exp2_best_epoch5.pt
SUMMARY SAVED
BEST MICRO-F1@7: 0.14563646470261254
EXPERIMENT 2: COMPLETE


In [36]:

import os
import torch
import inspect

print("========== LORMA ENVIRONMENT CHECK ==========")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ))

print("\n========== EXISTING VARIABLES ==========")

for name in [
    "encoder",
    "A4EncoderForNER",
    "tokenizer",
    "training_pairs",
    "pairs_by_query",
    "restored_cache",
    "encode_document_fast",
    "contrastive_loss",
    "device"
]:
    print(f"{name}: {'AVAILABLE' if name in globals() else 'MISSING'}")

print("\n========== MODEL ARCHITECTURE ==========")

if "A4EncoderForNER" in globals():
    print("Constructor:", inspect.signature(A4EncoderForNER.__init__))

if "encoder" in globals():
    print("Model:", type(encoder).__name__)
    print("Layers:", len(encoder.encoder_layers))
    print("First layer:", type(encoder.encoder_layers[0]).__name__)
    print("Attention structure:")
    print(encoder.encoder_layers[0].attn)

print("\n========== PHASE 1 CHECKPOINT ==========")

import glob

matches = glob.glob(
    "/kaggle/input/**/a4_step_22000.pt",
    recursive=True
)

for path in matches:
    print(path)

print("\n========== CHECK COMPLETE ==========")


========== LORMA ENVIRONMENT CHECK ==========
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56

========== EXISTING VARIABLES ==========
encoder: AVAILABLE
A4EncoderForNER: AVAILABLE
tokenizer: AVAILABLE
training_pairs: AVAILABLE
pairs_by_query: AVAILABLE
restored_cache: AVAILABLE
encode_document_fast: AVAILABLE
contrastive_loss: AVAILABLE
device: AVAILABLE

========== MODEL ARCHITECTURE ==========
Constructor: (self)
Model: A4EncoderForNER
Layers: 6
First layer: A4EncoderLayer
Attention structure:
A4Attention(
  (q_proj): Linear(in_features=512, out_features=512, bias=True)
  (k_proj): Linear(in_features=512, out_features=512, bias=True)
  (v_proj): Linear(in_features=512, out_features=512, bias=True)
  (out_proj): Linear(in_features=512, out_features=512, bias=True)
)

========== PHASE 1 CHECKPOINT ==========
/kaggle/input/datasets/aq1de3/a4-step-220000/a4_step_22000.pt

========== CHECK COMPLETE ==========


In [37]:

import torch

PHASE1_CHECKPOINT = (
    "/kaggle/input/datasets/aq1de3/"
    "a4-step-220000/a4_step_22000.pt"
)

phase1_checkpoint = torch.load(
    PHASE1_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(phase1_checkpoint))

if isinstance(phase1_checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in phase1_checkpoint.keys():
        print("-", key)

    if "model_state_dict" in phase1_checkpoint:
        state = phase1_checkpoint["model_state_dict"]
    elif "state_dict" in phase1_checkpoint:
        state = phase1_checkpoint["state_dict"]
    else:
        # Check whether the checkpoint itself is a state dict
        state = phase1_checkpoint

    print("\nState dictionary entries:", len(state))
    print("\nSample parameter names:")
    for key in list(state.keys())[:12]:
        print(key, tuple(state[key].shape))

print("\nCheckpoint inspection complete.")


Checkpoint type: <class 'dict'>

Checkpoint keys:
- model_state_dict
- step
- tokens_processed
- train_loss
- reason
- saved_at

State dictionary entries: 260

Sample parameter names:
token_embedding.weight (50000, 512)
encoder_pos.weight (1024, 512)
decoder_pos.weight (1024, 512)
encoder_layers.0.norm1.weight (512,)
encoder_layers.0.norm1.bias (512,)
encoder_layers.0.attn.q_proj.weight (512, 512)
encoder_layers.0.attn.q_proj.bias (512,)
encoder_layers.0.attn.k_proj.weight (512, 512)
encoder_layers.0.attn.k_proj.bias (512,)
encoder_layers.0.attn.v_proj.weight (512, 512)
encoder_layers.0.attn.v_proj.bias (512,)
encoder_layers.0.attn.out_proj.weight (512, 512)

Checkpoint inspection complete.


In [39]:

import torch

# Extract full Phase 1 checkpoint
full_state = phase1_checkpoint["model_state_dict"]

# Keep only encoder parameters
encoder_state = {
    key: value
    for key, value in full_state.items()
    if (
        key.startswith("token_embedding.")
        or key.startswith("encoder_pos.")
        or key.startswith("encoder_layers.")
    )
}

print("Full checkpoint entries:", len(full_state))
print("Encoder entries:", len(encoder_state))

# Create fresh encoder-only model
lorma_encoder = A4EncoderForNER()

# Strictly load all encoder weights
load_result = lorma_encoder.load_state_dict(
    encoder_state,
    strict=True
)

# Move to GPU
lorma_encoder = lorma_encoder.to(device)

# Freeze all original parameters
for param in lorma_encoder.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in lorma_encoder.parameters())
trainable_params = sum(
    p.numel() for p in lorma_encoder.parameters()
    if p.requires_grad
)

print("\n========== RESTORATION RESULTS ==========")
print("Strict loading: SUCCESS")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Device:", next(lorma_encoder.parameters()).device)

assert not load_result.missing_keys
assert not load_result.unexpected_keys
assert trainable_params == 0

print("\nPHASE 1 ENCODER RESTORED SUCCESSFULLY")


Full checkpoint entries: 260
Encoder entries: 98

========== RESTORATION RESULTS ==========
Strict loading: SUCCESS
Missing keys: []
Unexpected keys: []
Total parameters: 45,038,592
Trainable parameters: 0
Device: cuda:0

PHASE 1 ENCODER RESTORED SUCCESSFULLY


In [40]:

import torch
import torch.nn as nn

class LORMALinear(nn.Module):

    def __init__(
        self,
        base_layer,
        rank=8,
        alpha=16,
        dropout=0.05
    ):
        super().__init__()

        self.base = base_layer
        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        self.dropout = nn.Dropout(dropout)

        # Trainable low-rank matrices
        self.lorma_A = nn.Parameter(
            torch.empty(rank, self.in_features)
        )

        self.lorma_B = nn.Parameter(
            torch.zeros(self.out_features, rank)
        )

        nn.init.kaiming_uniform_(
            self.lorma_A,
            a=5**0.5
        )

        nn.init.zeros_(self.lorma_B)

        # Freeze original linear layer
        for param in self.base.parameters():
            param.requires_grad = False

    def forward(self, x):

        base_output = self.base(x)

        low_rank = (
            self.dropout(x)
            @ self.lorma_A.T
            @ self.lorma_B.T
        )

        return base_output + self.scaling * low_rank


# Inject adapters into Q and V projections
lorma_targets = []

for layer_idx, layer in enumerate(
    lorma_encoder.encoder_layers
):

    layer.attn.q_proj = LORMALinear(
        layer.attn.q_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

    layer.attn.v_proj = LORMALinear(
        layer.attn.v_proj,
        rank=8,
        alpha=16,
        dropout=0.05
    )

    lorma_targets.extend([
        f"encoder_layers.{layer_idx}.attn.q_proj",
        f"encoder_layers.{layer_idx}.attn.v_proj"
    ])


# Collect trainable parameters
lorma_trainable = [
    p for p in lorma_encoder.parameters()
    if p.requires_grad
]

total_params = sum(
    p.numel() for p in lorma_encoder.parameters()
)

trainable_params = sum(
    p.numel() for p in lorma_trainable
)

print("========== LORMA INJECTION ==========")
print("Target modules:", len(lorma_targets))
print("Target projections:")
for target in lorma_targets:
    print("-", target)

print("\nTotal model parameters:", f"{total_params:,}")
print("Trainable LORMA parameters:", f"{trainable_params:,}")
print(
    "Trainable percentage:",
    round(100 * trainable_params / total_params, 4),
    "%"
)

print("\nLORMA adapters successfully injected.")


========== LORMA INJECTION ==========
Target modules: 12
Target projections:
- encoder_layers.0.attn.q_proj
- encoder_layers.0.attn.v_proj
- encoder_layers.1.attn.q_proj
- encoder_layers.1.attn.v_proj
- encoder_layers.2.attn.q_proj
- encoder_layers.2.attn.v_proj
- encoder_layers.3.attn.q_proj
- encoder_layers.3.attn.v_proj
- encoder_layers.4.attn.q_proj
- encoder_layers.4.attn.v_proj
- encoder_layers.5.attn.q_proj
- encoder_layers.5.attn.v_proj

Total model parameters: 45,136,896
Trainable LORMA parameters: 98,304
Trainable percentage: 0.2178 %

LORMA adapters successfully injected.


In [42]:

# Move the complete LORMA model to the GPU
lorma_encoder = lorma_encoder.to(device)

# Verify device placement of every adapter
print("========== DEVICE VALIDATION ==========")

for layer_idx, layer in enumerate(lorma_encoder.encoder_layers):

    for name in ["q_proj", "v_proj"]:

        adapter = getattr(layer.attn, name)

        print(
            f"Layer {layer_idx} {name}:",
            "A =", adapter.lorma_A.device,
            "| B =", adapter.lorma_B.device
        )

# Verify all trainable parameters are on CUDA
assert all(
    p.device.type == "cuda"
    for p in lorma_encoder.parameters()
    if p.requires_grad
)

print("\nALL LORMA ADAPTERS ARE ON GPU")


========== DEVICE VALIDATION ==========
Layer 0 q_proj: A = cuda:0 | B = cuda:0
Layer 0 v_proj: A = cuda:0 | B = cuda:0
Layer 1 q_proj: A = cuda:0 | B = cuda:0
Layer 1 v_proj: A = cuda:0 | B = cuda:0
Layer 2 q_proj: A = cuda:0 | B = cuda:0
Layer 2 v_proj: A = cuda:0 | B = cuda:0
Layer 3 q_proj: A = cuda:0 | B = cuda:0
Layer 3 v_proj: A = cuda:0 | B = cuda:0
Layer 4 q_proj: A = cuda:0 | B = cuda:0
Layer 4 v_proj: A = cuda:0 | B = cuda:0
Layer 5 q_proj: A = cuda:0 | B = cuda:0
Layer 5 v_proj: A = cuda:0 | B = cuda:0

ALL LORMA ADAPTERS ARE ON GPU


In [43]:

reference_encoder.eval()
lorma_encoder.eval()

with torch.inference_mode():
    original_output = reference_encoder(
        input_ids, attention_mask
    )

    lorma_output = lorma_encoder(
        input_ids, attention_mask
    )

max_difference = (
    original_output - lorma_output
).abs().max().item()

mean_difference = (
    original_output - lorma_output
).abs().mean().item()

print("Maximum difference:", max_difference)
print("Mean difference:", mean_difference)

outputs_match = torch.allclose(
    original_output,
    lorma_output,
    atol=1e-6,
    rtol=1e-5
)

print("Outputs match:", outputs_match)

assert outputs_match, "LORMA output validation failed!"

print("\nLORMA OUTPUT VALIDATION PASSED")


Maximum difference: 0.0
Mean difference: 0.0
Outputs match: True

LORMA OUTPUT VALIDATION PASSED


In [44]:

lorma_encoder.train()
lorma_encoder.zero_grad(set_to_none=True)

# Forward pass
with torch.autocast(device_type="cuda", dtype=torch.float16):
    output = lorma_encoder(input_ids, attention_mask)

# Simple test loss
loss = output.float().pow(2).mean()
loss.backward()

# Check adapter gradients
a_grad_norms = []
b_grad_norms = []

for layer in lorma_encoder.encoder_layers:
    for name in ["q_proj", "v_proj"]:
        adapter = getattr(layer.attn, name)

        assert adapter.lorma_A.grad is not None
        assert adapter.lorma_B.grad is not None

        assert torch.isfinite(adapter.lorma_A.grad).all()
        assert torch.isfinite(adapter.lorma_B.grad).all()

        a_grad_norms.append(adapter.lorma_A.grad.norm().item())
        b_grad_norms.append(adapter.lorma_B.grad.norm().item())

# Original parameters must remain frozen
base_trainable = [
    name for name, param in lorma_encoder.named_parameters()
    if param.requires_grad and "lorma_" not in name
]

print("Loss:", loss.item())
print("Average A gradient norm:", sum(a_grad_norms) / len(a_grad_norms))
print("Average B gradient norm:", sum(b_grad_norms) / len(b_grad_norms))
print("Original trainable parameters:", len(base_trainable))

assert len(base_trainable) == 0
assert sum(b_grad_norms) > 0

print("\nLORMA GRADIENT VALIDATION PASSED")


Loss: 4.1761016845703125
Average A gradient norm: 0.0
Average B gradient norm: 0.03472189160917575
Original trainable parameters: 0

LORMA GRADIENT VALIDATION PASSED


In [45]:

# Collect only trainable LORMA parameters
lorma_trainable_params = [
    p for p in lorma_encoder.parameters()
    if p.requires_grad
]

# Clear gradients from validation
lorma_encoder.zero_grad(set_to_none=True)

# Optimizer
lorma_optimizer = torch.optim.AdamW(
    lorma_trainable_params,
    lr=5e-4,
    weight_decay=0.01
)

print("Trainable parameters:", sum(
    p.numel() for p in lorma_trainable_params
))
print("Optimizer:", type(lorma_optimizer).__name__)
print("Learning rate:", lorma_optimizer.param_groups[0]["lr"])
print("Weight decay:", lorma_optimizer.param_groups[0]["weight_decay"])

assert len(lorma_trainable_params) == 24
print("\nLORMA OPTIMIZER READY")


Trainable parameters: 98304
Optimizer: AdamW
Learning rate: 0.0005
Weight decay: 0.01

LORMA OPTIMIZER READY


In [46]:

print("Training pairs:", len(training_pairs))
print("\nFirst training pair:")
print(training_pairs[0])

print("\nPair type:", type(training_pairs[0]))

print("\nAvailable cache keys:")
print(restored_cache.keys())

print("\nEncoding function available:", callable(encode_document_fast))
print("Contrastive loss available:", callable(contrastive_loss))


Training pairs: 8576

First training pair:
{'query_id': '0000000599', 'candidate_id': '0001292810', 'label': 1}

Pair type: <class 'dict'>

Available cache keys:
dict_keys(['queries', 'candidates'])

Encoding function available: True
Contrastive loss available: True


In [48]:

import torch.utils.checkpoint as checkpoint_module

# Restore the callable used inside encode_document_fast
encode_document_fast.__globals__["checkpoint"] = (
    checkpoint_module.checkpoint
)

print(
    "Checkpoint type:",
    type(encode_document_fast.__globals__["checkpoint"])
)

assert callable(
    encode_document_fast.__globals__["checkpoint"]
)

# Test one query and candidate pair
test_pair = training_pairs[0]

q_chunks = restored_cache["queries"][test_pair["query_id"]]
c_chunks = restored_cache["candidates"][test_pair["candidate_id"]]

lorma_encoder.train()

q_test_emb = encode_document_fast(q_chunks, lorma_encoder)
c_test_emb = encode_document_fast(c_chunks, lorma_encoder)

print("Query embedding shape:", q_test_emb.shape)
print("Candidate embedding shape:", c_test_emb.shape)
print("Both embeddings finite:",
      torch.isfinite(q_test_emb).all().item()
      and torch.isfinite(c_test_emb).all().item())

print("\nCHECKPOINT REPAIR SUCCESSFUL")


Checkpoint type: <class 'function'>
Query embedding shape: torch.Size([512])
Candidate embedding shape: torch.Size([512])
Both embeddings finite: True

CHECKPOINT REPAIR SUCCESSFUL


In [50]:

import gc
import torch
import time
from torch.nn.utils import clip_grad_norm_

# 1. Release temporary tensors and failed computation graphs
for name in [
    "q_emb", "c_emb", "output", "loss",
    "query_embeddings", "candidate_embeddings",
    "labels", "similarities", "batch"
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# 2. Reduce chunks processed simultaneously
encode_document_fast.__globals__["CHUNK_BATCH_SIZE"] = 1

# 3. Resume from the batch where the OOM occurred
# 'start' retains the starting index of the failed batch.
resume_index = start
remaining_pairs = epoch_pairs[resume_index:]

print("Previously completed batches:", num_batches)
print("Resuming at pair index:", resume_index)
print("Remaining pairs:", len(remaining_pairs))
print("Chunk batch size: 1")
print("Training pair batch size: 1")

# 4. Clear incomplete gradients
lorma_optimizer.zero_grad(set_to_none=True)
lorma_encoder.train()

resume_start_time = time.time()

for pair in remaining_pairs:

    qid = pair["query_id"]
    cid = pair["candidate_id"]

    q_chunks = restored_cache["queries"][qid]
    c_chunks = restored_cache["candidates"][cid]

    q_emb = encode_document_fast(q_chunks, lorma_encoder)
    c_emb = encode_document_fast(c_chunks, lorma_encoder)

    q_emb = q_emb.unsqueeze(0)
    c_emb = c_emb.unsqueeze(0)

    labels = torch.tensor(
        [float(pair["label"])],
        dtype=torch.float32,
        device=device
    )

    loss, _ = contrastive_loss(
        q_emb, c_emb, labels, margin=LORMA_MARGIN
    )

    loss.backward()

    clip_grad_norm_(lorma_trainable_params, max_norm=1.0)

    lorma_optimizer.step()
    lorma_optimizer.zero_grad(set_to_none=True)

    total_loss += loss.item()
    num_batches += 1

    if num_batches % 100 == 0:
        print(
            f"Completed updates: {num_batches} "
            f"| Current loss: {loss.item():.5f} "
            f"| GPU allocated: "
            f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
        )

    del q_emb, c_emb, labels, loss

average_loss = total_loss / num_batches
elapsed = (time.time() - resume_start_time) / 60

print("\n========== LORMA EPOCH 1 ==========")
print("Average loss:", average_loss)
print("Total optimizer updates:", num_batches)
print("Resume time (minutes):", round(elapsed, 2))
print("EPOCH 1 COMPLETED")


Previously completed batches: 0
Resuming at pair index: 0
Remaining pairs: 8576
Chunk batch size: 1
Training pair batch size: 1
Completed updates: 100 | Current loss: 0.16248 | GPU allocated: 7.63 GB
Completed updates: 200 | Current loss: 0.03931 | GPU allocated: 7.63 GB
Completed updates: 300 | Current loss: 0.18028 | GPU allocated: 7.63 GB
Completed updates: 400 | Current loss: 0.06395 | GPU allocated: 7.63 GB
Completed updates: 500 | Current loss: 0.00840 | GPU allocated: 7.63 GB
Completed updates: 600 | Current loss: 0.13289 | GPU allocated: 7.63 GB
Completed updates: 700 | Current loss: 0.01600 | GPU allocated: 7.63 GB
Completed updates: 800 | Current loss: 0.15398 | GPU allocated: 7.63 GB
Completed updates: 900 | Current loss: 0.16559 | GPU allocated: 7.63 GB
Completed updates: 1000 | Current loss: 0.10631 | GPU allocated: 7.63 GB
Completed updates: 1100 | Current loss: 0.02235 | GPU allocated: 7.63 GB
Completed updates: 1200 | Current loss: 0.01916 | GPU allocated: 7.63 GB
Compl

In [51]:

import os
import json
import torch

LORMA_DIR = "/kaggle/working/phase7_exp3_lorma"
os.makedirs(LORMA_DIR, exist_ok=True)

# Extract trained adapter parameters only
adapter_state = {
    name: param.detach().cpu().clone()
    for name, param in lorma_encoder.state_dict().items()
    if "lorma_" in name
}

adapter_parameters = sum(
    p.numel() for p in adapter_state.values()
)

assert adapter_parameters == 98304

checkpoint_path = os.path.join(
    LORMA_DIR, "lorma_epoch1_adapter.pt"
)

torch.save({
    "adapter_state_dict": adapter_state,
    "base_checkpoint": (
        "/kaggle/input/datasets/aq1de3/"
        "a4-step-220000/a4_step_22000.pt"
    ),
    "rank": 8,
    "alpha": 16,
    "dropout": 0.05,
    "adapted_projections": ["q_proj", "v_proj"],
    "adapted_layers": 6,
    "trainable_parameters": adapter_parameters,
    "epoch": 1
}, checkpoint_path)

summary = {
    "experiment": "Experiment 3 - LORMA",
    "training_pairs": len(training_pairs),
    "optimizer_updates": num_batches,
    "average_training_loss": float(average_loss),
    "training_time_minutes": 105.48,
    "learning_rate": 0.0005,
    "weight_decay": 0.01,
    "pair_batch_size": 1,
    "chunk_batch_size": 1,
    "rank": 8,
    "alpha": 16,
    "dropout": 0.05,
    "trainable_parameters": adapter_parameters
}

with open(os.path.join(LORMA_DIR, "experiment3_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("CHECKPOINT SAVED:", checkpoint_path)
print("ADAPTER PARAMETERS:", adapter_parameters)
print("AVERAGE TRAINING LOSS:", average_loss)
print("EXPERIMENT 3 SAVED SUCCESSFULLY")


CHECKPOINT SAVED: /kaggle/working/phase7_exp3_lorma/lorma_epoch1_adapter.pt
ADAPTER PARAMETERS: 98304
AVERAGE TRAINING LOSS: 0.0447769892178478
EXPERIMENT 3 SAVED SUCCESSFULLY


In [53]:

import os

DATA_ROOT = "/kaggle/input/datasets/aq1de3/phase7"

print("Dataset directory exists:", os.path.exists(DATA_ROOT))

print("\nAvailable files:")
for filename in os.listdir(DATA_ROOT):
    print(filename)

print("\n========== CACHE STATUS ==========")
print("Restored cache available:", "restored_cache" in globals())

if "restored_cache" in globals():
    print("Query cache:", len(restored_cache["queries"]))
    print("Candidate cache:", len(restored_cache["candidates"]))

print("\nLORMA checkpoint exists:", os.path.exists(checkpoint_path))


Dataset directory exists: True

Available files:
dev (3).json
dev_candidates.json
test (5).json
test_candidates.json
train (4).json
train_candidates.json

========== CACHE STATUS ==========
Restored cache available: True
Query cache: 827
Candidate cache: 4320

LORMA checkpoint exists: True


In [54]:

import json
import os

DATA_ROOT = "/kaggle/input/datasets/aq1de3/phase7"

# Load development queries
with open(os.path.join(DATA_ROOT, "dev (3).json"), "r") as f:
    dev_queries = json.load(f)

# Load development candidates
with open(os.path.join(DATA_ROOT, "dev_candidates.json"), "r") as f:
    dev_candidates = json.load(f)

print("Development queries:", len(dev_queries))
print("Development candidates:", len(dev_candidates))

print("\nQuery fields:", dev_queries[0].keys())
print("Candidate fields:", dev_candidates[0].keys())

print("\nSample query ID:", dev_queries[0].get("id"))
print("Sample candidate ID:", dev_candidates[0].get("id"))

print("\nRelevant candidates available:",
      "relevant_candidates" in dev_queries[0])

print("\nDEV DATA LOADED SUCCESSFULLY")


Development queries: 118
Development candidates: 1023


KeyError: 0

In [55]:

print("DEV QUERIES")
print("Type:", type(dev_queries))

if isinstance(dev_queries, dict):
    print("Top-level keys:", list(dev_queries.keys())[:10])
    first_key = next(iter(dev_queries))
    print("First key:", first_key)
    print("First value type:", type(dev_queries[first_key]))
    print("First value:", str(dev_queries[first_key])[:1000])

elif isinstance(dev_queries, list):
    print("Number of records:", len(dev_queries))
    print("First record:", str(dev_queries[0])[:1000])


print("\nDEV CANDIDATES")
print("Type:", type(dev_candidates))

if isinstance(dev_candidates, dict):
    print("Top-level keys:", list(dev_candidates.keys())[:10])
    first_key = next(iter(dev_candidates))
    print("First key:", first_key)
    print("First value:", str(dev_candidates[first_key])[:1000])

elif isinstance(dev_candidates, list):
    print("Number of records:", len(dev_candidates))
    print("First record:", str(dev_candidates[0])[:1000])


DEV QUERIES
Type: <class 'dict'>
Top-level keys: ['0000012637', '0000027891', '0000035468', '0000046064', '0000067667', '0000067739', '0000092507', '0000098072', '0000105440', '0000125801']
First key: 0000012637
First value type: <class 'dict'>
First value: {'text': ['PETITIONER: <ORG>, <NAME> Vs. RESPONDENT: COMMISSIONER OF INCOME-TAX, <NAME> DATE OF JUDGMENT: 08/12/1965 BENCH: <NAME>, S.M. BENCH: <NAME>, S.M. SUBBARAO, K. SHAH, J.C. CITATION: 1966 AIR 1053 1966 SCR (2) 944 CITATOR INFO : F 1967 SC 819 (5) R 1969 SC 840 (11) R 1969 SC 946 (5) E 1969 SC1160 (5) D 1975 SC 97 (20) R 1976 SC 772 (6) R 1986 SC1483 (4) ACT: Indian Income-tax Act , 1922, s. 10(2)(xv) -Loan obtained by company-Stamp duty and other expenditure incurred in obtaining the loan-Whether capital or revenue expenditure- Whether laid out for purpose of business. HEADNOTE: During the accounting period relevant for the assessment year 1950-51 the appellant company obtained a loan of 40 lakhs of rupees from <ORG>. The lo

In [56]:

print("========== DEV DATA AUDIT ==========")

dev_query_ids = list(dev_queries.keys())
dev_candidate_ids = list(dev_candidates.keys())

print("Development queries:", len(dev_query_ids))
print("Development candidates:", len(dev_candidate_ids))

# Check relevant candidate labels
first_qid = dev_query_ids[0]
first_record = dev_queries[first_qid]

print("\nFirst query ID:", first_qid)
print("Query fields:", list(first_record.keys()))
print("Relevant candidates:", len(first_record.get("relevant_candidates", [])))

# Verify all relevant IDs exist in candidate pool
missing_ids = []

for qid, record in dev_queries.items():
    for cid in record.get("relevant_candidates", []):
        if cid not in dev_candidates:
            missing_ids.append((qid, cid))

print("Missing relevant candidate IDs:", len(missing_ids))

# Check query/candidate self-overlap
overlap = set(dev_query_ids) & set(dev_candidate_ids)
print("Query-candidate ID overlap:", len(overlap))

assert len(missing_ids) == 0
assert len(dev_query_ids) == 118
assert len(dev_candidate_ids) == 1023

print("\nDEV DATA AUDIT PASSED")


========== DEV DATA AUDIT ==========
Development queries: 118
Development candidates: 1023

First query ID: 0000012637
Query fields: ['text', 'relevant_candidates']
Relevant candidates: 11
Missing relevant candidate IDs: 0
Query-candidate ID overlap: 118

DEV DATA AUDIT PASSED


In [57]:

import os
import json
import gc
import torch

# Evaluation configuration
lorma_encoder.eval()

CHUNK_BATCH_SIZE = 1
encode_document_fast.__globals__["CHUNK_BATCH_SIZE"] = 1

dev_output_dir = "/kaggle/working/phase7_exp3_lorma/dev_evaluation"
os.makedirs(dev_output_dir, exist_ok=True)

# 1. Encode development queries
dev_query_embeddings = {}
dev_query_ids = list(dev_queries.keys())

print("Encoding development queries...")

with torch.inference_mode():
    for i, qid in enumerate(dev_query_ids):

        chunks = chunk_document(
            dev_queries[qid]["text"], tokenizer
        )

        emb = encode_document_fast(chunks, lorma_encoder)

        dev_query_embeddings[qid] = emb.detach().float().cpu()

        if (i + 1) % 20 == 0:
            print(f"Queries encoded: {i+1}/{len(dev_query_ids)}")

# 2. Encode development candidates
dev_candidate_embeddings = {}
dev_candidate_ids = list(dev_candidates.keys())

print("\nEncoding development candidates...")

with torch.inference_mode():
    for i, cid in enumerate(dev_candidate_ids):

        chunks = chunk_document(
            dev_candidates[cid]["text"], tokenizer
        )

        emb = encode_document_fast(chunks, lorma_encoder)

        dev_candidate_embeddings[cid] = emb.detach().float().cpu()

        if (i + 1) % 100 == 0:
            print(
                f"Candidates encoded: "
                f"{i+1}/{len(dev_candidate_ids)}"
            )

# 3. Create candidate embedding matrix
candidate_matrix = torch.stack([
    dev_candidate_embeddings[cid]
    for cid in dev_candidate_ids
])

candidate_matrix = torch.nn.functional.normalize(
    candidate_matrix, p=2, dim=1
)

# 4. Rank top 7 for every development query
dev_predictions = {}

total_tp = 0
total_relevant = 0
total_predicted = 0

for qid in dev_query_ids:

    query_vector = dev_query_embeddings[qid].unsqueeze(0)
    query_vector = torch.nn.functional.normalize(
        query_vector, p=2, dim=1
    )

    scores = torch.matmul(
        query_vector, candidate_matrix.T
    ).squeeze(0)

    # Exclude the query's own ID
    if qid in dev_candidate_ids:
        self_index = dev_candidate_ids.index(qid)
        scores[self_index] = -float("inf")

    top_indices = torch.topk(scores, k=7).indices.tolist()

    top7_ids = [
        dev_candidate_ids[idx] for idx in top_indices
    ]

    dev_predictions[qid] = top7_ids

    relevant = set(
        dev_queries[qid].get("relevant_candidates", [])
    )

    predicted = set(top7_ids)

    total_tp += len(predicted & relevant)
    total_relevant += len(relevant)
    total_predicted += len(top7_ids)

# 5. Calculate micro metrics
micro_precision = total_tp / total_predicted
micro_recall = total_tp / total_relevant

micro_f1 = (
    2 * micro_precision * micro_recall /
    (micro_precision + micro_recall)
    if micro_precision + micro_recall > 0 else 0.0
)

# 6. Save predictions
prediction_path = os.path.join(
    dev_output_dir, "lorma_epoch1_dev_predictions.json"
)

with open(prediction_path, "w") as f:
    json.dump(dev_predictions, f, indent=2)

metrics = {
    "experiment": "LORMA Epoch 1",
    "true_positives": total_tp,
    "total_relevant": total_relevant,
    "total_predictions": total_predicted,
    "micro_precision_at_7": micro_precision,
    "micro_recall_at_7": micro_recall,
    "micro_f1_at_7": micro_f1
}

metrics_path = os.path.join(
    dev_output_dir, "lorma_epoch1_dev_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\n========== LORMA DEV RESULTS ==========")
print("True positives:", total_tp)
print("Total relevant:", total_relevant)
print("Total predictions:", total_predicted)
print("Micro-Precision@7:", micro_precision)
print("Micro-Recall@7:", micro_recall)
print("Micro-F1@7:", micro_f1)

print("\nExperiment 2 best Micro-F1@7: 0.1456364647")
print("LORMA predictions saved:", prediction_path)
print("LORMA metrics saved:", metrics_path)

print("\nLORMA DEVELOPMENT EVALUATION COMPLETED")


Encoding development queries...
Queries encoded: 20/118
Queries encoded: 40/118
Queries encoded: 60/118
Queries encoded: 80/118
Queries encoded: 100/118

Encoding development candidates...
Candidates encoded: 100/1023
Candidates encoded: 200/1023
Candidates encoded: 300/1023
Candidates encoded: 400/1023
Candidates encoded: 500/1023
Candidates encoded: 600/1023
Candidates encoded: 700/1023
Candidates encoded: 800/1023
Candidates encoded: 900/1023
Candidates encoded: 1000/1023

========== LORMA DEV RESULTS ==========
True positives: 51
Total relevant: 973
Total predictions: 826
Micro-Precision@7: 0.06174334140435835
Micro-Recall@7: 0.052415210688591986
Micro-F1@7: 0.056698165647582

Experiment 2 best Micro-F1@7: 0.1456364647
LORMA predictions saved: /kaggle/working/phase7_exp3_lorma/dev_evaluation/lorma_epoch1_dev_predictions.json
LORMA metrics saved: /kaggle/working/phase7_exp3_lorma/dev_evaluation/lorma_epoch1_dev_metrics.json

LORMA DEVELOPMENT EVALUATION COMPLETED


In [58]:

import os
import torch

EXP2_CHECKPOINT = (
    "/kaggle/working/phase7_exp2_final/"
    "phase7_exp2_best_epoch5.pt"
)

print("Checkpoint exists:", os.path.exists(EXP2_CHECKPOINT))

if os.path.exists(EXP2_CHECKPOINT):
    checkpoint_data = torch.load(
        EXP2_CHECKPOINT,
        map_location="cpu",
        weights_only=False
    )

    print("\nCheckpoint type:", type(checkpoint_data))

    if isinstance(checkpoint_data, dict):
        print("Checkpoint keys:", list(checkpoint_data.keys()))

        for key, value in checkpoint_data.items():
            if isinstance(value, dict):
                print(key, ":", len(value), "entries")
            elif torch.is_tensor(value):
                print(key, ":", tuple(value.shape))
            else:
                print(key, ":", str(value)[:200])

else:
    print("WARNING: Experiment 2 checkpoint not found.")


Checkpoint exists: True

Checkpoint type: <class 'dict'>
Checkpoint keys: ['epoch', 'global_step', 'model_state_dict', 'optimizer_state_dict', 'epoch_history']
epoch : 5
global_step : 12435
model_state_dict : 98 entries
optimizer_state_dict : 2 entries
epoch_history : [{'epoch': 1, 'average_loss': 0.043492920848638816, 'pairs_processed': 8576, 'optimizer_steps': 2487, 'seconds': 6356.409047365189}, {'epoch': 2, 'average_loss': 0.0338135268596483, 'pairs_processed':


In [59]:

import torch
import os

EXP2_CHECKPOINT = (
    "/kaggle/working/phase7_exp2_final/"
    "phase7_exp2_best_epoch5.pt"
)

# Load checkpoint
exp2_checkpoint = torch.load(
    EXP2_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

# Create a fresh encoder and restore the complete trained weights
final_encoder = A4EncoderForNER()

load_result = final_encoder.load_state_dict(
    exp2_checkpoint["model_state_dict"],
    strict=True
)

final_encoder = final_encoder.to(device)
final_encoder.eval()

# Freeze all parameters for inference
for param in final_encoder.parameters():
    param.requires_grad = False

print("========== FINAL MODEL RESTORED ==========")
print("Experiment: Partial Fine-Tuning")
print("Best epoch:", exp2_checkpoint["epoch"])
print("Global step:", exp2_checkpoint["global_step"])
print("Strict loading: SUCCESS")
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print("Device:", next(final_encoder.parameters()).device)
print("Total parameters:", sum(
    p.numel() for p in final_encoder.parameters()
))
print("Trainable parameters:", sum(
    p.numel() for p in final_encoder.parameters()
    if p.requires_grad
))

assert len(load_result.missing_keys) == 0
assert len(load_result.unexpected_keys) == 0

print("\nBEST MODEL READY FOR FINAL INFERENCE")


========== FINAL MODEL RESTORED ==========
Experiment: Partial Fine-Tuning
Best epoch: 5
Global step: 12435
Strict loading: SUCCESS
Missing keys: []
Unexpected keys: []
Device: cuda:0
Total parameters: 45038592
Trainable parameters: 0

BEST MODEL READY FOR FINAL INFERENCE


In [60]:

import json
import os

DATA_ROOT = "/kaggle/input/datasets/aq1de3/phase7"

# Load test queries
with open(os.path.join(DATA_ROOT, "test (5).json"), "r") as f:
    test_queries = json.load(f)

# Load test candidates
with open(os.path.join(DATA_ROOT, "test_candidates.json"), "r") as f:
    test_candidates = json.load(f)

test_query_ids = list(test_queries.keys())
test_candidate_ids = list(test_candidates.keys())

print("========== TEST DATA AUDIT ==========")
print("Test queries:", len(test_query_ids))
print("Test candidates:", len(test_candidate_ids))

print("\nQuery example ID:", test_query_ids[0])
print("Query fields:", list(test_queries[test_query_ids[0]].keys()))

print("\nCandidate example ID:", test_candidate_ids[0])
print("Candidate fields:", list(test_candidates[test_candidate_ids[0]].keys()))

print("\nQuery-candidate ID overlap:",
      len(set(test_query_ids) & set(test_candidate_ids)))

assert len(test_query_ids) == 237
assert len(test_candidate_ids) == 1727

assert all("text" in test_queries[qid] for qid in test_query_ids)
assert all("text" in test_candidates[cid] for cid in test_candidate_ids)

print("\nTEST DATA AUDIT PASSED")


========== TEST DATA AUDIT ==========
Test queries: 237
Test candidates: 1727

Query example ID: 0000021652
Query fields: ['text']

Candidate example ID: 0000000987
Candidate fields: ['text']

Query-candidate ID overlap: 237

TEST DATA AUDIT PASSED


In [61]:
import os
import torch

# Use the best Experiment 2 model
final_encoder.eval()

# Keep memory usage low on the T4
encode_document_fast.__globals__["CHUNK_BATCH_SIZE"] = 1
encode_document_fast.__globals__["checkpoint"] = torch.utils.checkpoint.checkpoint

SAVE_DIR = "/kaggle/working/phase7_final"
os.makedirs(SAVE_DIR, exist_ok=True)

QUERY_EMB_PATH = os.path.join(SAVE_DIR, "test_query_embeddings.pt")
CAND_EMB_PATH = os.path.join(SAVE_DIR, "test_candidate_embeddings.pt")

# Resume previously saved embeddings if available
if os.path.exists(QUERY_EMB_PATH):
    test_query_embeddings = torch.load(
        QUERY_EMB_PATH, map_location="cpu", weights_only=False
    )
else:
    test_query_embeddings = {}

if os.path.exists(CAND_EMB_PATH):
    test_candidate_embeddings = torch.load(
        CAND_EMB_PATH, map_location="cpu", weights_only=False
    )
else:
    test_candidate_embeddings = {}

def encode_split(documents, embeddings, save_path, label):
    ids = list(documents.keys())
    pending = [doc_id for doc_id in ids if doc_id not in embeddings]

    print(f"\n{label}: {len(embeddings)} already done; {len(pending)} remaining")

    with torch.inference_mode():
        for i, doc_id in enumerate(pending, 1):
            chunks = chunk_document(documents[doc_id]["text"], tokenizer)
            vector = encode_document_fast(chunks, final_encoder)

            embeddings[doc_id] = vector.detach().float().cpu()

            if i % 25 == 0 or i == len(pending):
                torch.save(embeddings, save_path)
                print(
                    f"{label}: {i}/{len(pending)} new documents processed "
                    f"| Total saved: {len(embeddings)}/{len(ids)}"
                )

    torch.save(embeddings, save_path)
    print(f"{label} embeddings complete.")

encode_split(
    test_queries,
    test_query_embeddings,
    QUERY_EMB_PATH,
    "TEST QUERIES"
)

encode_split(
    test_candidates,
    test_candidate_embeddings,
    CAND_EMB_PATH,
    "TEST CANDIDATES"
)

# Final embedding audit
assert len(test_query_embeddings) == len(test_queries)
assert len(test_candidate_embeddings) == len(test_candidates)

print("\n========== EMBEDDING AUDIT ==========")
print("Query embeddings:", len(test_query_embeddings))
print("Candidate embeddings:", len(test_candidate_embeddings))
print("Query vector shape:", tuple(next(iter(test_query_embeddings.values())).shape))
print("Candidate vector shape:", tuple(next(iter(test_candidate_embeddings.values())).shape))
print("All embeddings saved successfully.")


TEST QUERIES: 0 already done; 237 remaining
TEST QUERIES: 25/237 new documents processed | Total saved: 25/237
TEST QUERIES: 50/237 new documents processed | Total saved: 50/237
TEST QUERIES: 75/237 new documents processed | Total saved: 75/237
TEST QUERIES: 100/237 new documents processed | Total saved: 100/237
TEST QUERIES: 125/237 new documents processed | Total saved: 125/237
TEST QUERIES: 150/237 new documents processed | Total saved: 150/237
TEST QUERIES: 175/237 new documents processed | Total saved: 175/237
TEST QUERIES: 200/237 new documents processed | Total saved: 200/237
TEST QUERIES: 225/237 new documents processed | Total saved: 225/237
TEST QUERIES: 237/237 new documents processed | Total saved: 237/237
TEST QUERIES embeddings complete.

TEST CANDIDATES: 0 already done; 1727 remaining
TEST CANDIDATES: 25/1727 new documents processed | Total saved: 25/1727
TEST CANDIDATES: 50/1727 new documents processed | Total saved: 50/1727
TEST CANDIDATES: 75/1727 new documents proce

In [64]:
import os
import json
import torch
import torch.nn.functional as F

SAVE_DIR = "/kaggle/working/phase7_final"
os.makedirs(SAVE_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    SAVE_DIR, "phase7_final_submission.json"
)

# Check embeddings
assert len(test_query_embeddings) == 237
assert len(test_candidate_embeddings) == 1727

# Prepare candidate embedding matrix
candidate_ids = list(test_candidates.keys())

candidate_matrix = torch.stack([
    test_candidate_embeddings[cid].float()
    for cid in candidate_ids
])

candidate_matrix = F.normalize(candidate_matrix, p=2, dim=1)

candidate_index = {
    cid: idx for idx, cid in enumerate(candidate_ids)
}

final_predictions = {}

# Rank candidates for each query
for i, qid in enumerate(test_queries.keys(), 1):

    query_vector = test_query_embeddings[qid].float()
    query_vector = F.normalize(query_vector, p=2, dim=0)

    scores = candidate_matrix @ query_vector

    # Exclude the query itself
    if qid in candidate_index:
        scores[candidate_index[qid]] = -float("inf")

    top_indices = torch.topk(scores, k=7).indices.tolist()

    final_predictions[qid] = [
        candidate_ids[idx] for idx in top_indices
    ]

    if i % 25 == 0 or i == 237:
        print(f"Processed {i}/237 queries")

# Save JSON
with open(OUTPUT_PATH, "w") as f:
    json.dump(final_predictions, f, indent=2)

print("\n========== FINAL JSON CREATED ==========")
print("Queries:", len(final_predictions))
print("Output:", OUTPUT_PATH)
print("File exists:", os.path.exists(OUTPUT_PATH))
print("File size:", round(os.path.getsize(OUTPUT_PATH) / 1024, 2), "KB")

Processed 25/237 queries
Processed 50/237 queries
Processed 75/237 queries
Processed 100/237 queries
Processed 125/237 queries
Processed 150/237 queries
Processed 175/237 queries
Processed 200/237 queries
Processed 225/237 queries
Processed 237/237 queries

========== FINAL JSON CREATED ==========
Queries: 237
Output: /kaggle/working/phase7_final/phase7_final_submission.json
File exists: True
File size: 34.26 KB


In [65]:
import json
import os

OUTPUT_PATH = "/kaggle/working/phase7_final/phase7_final_submission.json"

with open(OUTPUT_PATH, "r") as f:
    submitted = json.load(f)

assert len(submitted) == 237

for qid, predictions in submitted.items():
    assert len(predictions) == 7
    assert len(set(predictions)) == 7
    assert qid not in predictions
    assert all(pid in test_candidates for pid in predictions)

assert set(submitted.keys()) == set(test_queries.keys())

print("========== FINAL VALIDATION ==========")
print("237 queries: PASS")
print("7 predictions per query: PASS")
print("No duplicate predictions: PASS")
print("No self-matches: PASS")
print("All candidate IDs valid: PASS")
print("JSON structure: PASS")

print("\nSUBMISSION READY!")
print("File:", OUTPUT_PATH)

========== FINAL VALIDATION ==========
237 queries: PASS
7 predictions per query: PASS
No duplicate predictions: PASS
No self-matches: PASS
All candidate IDs valid: PASS
JSON structure: PASS

SUBMISSION READY!
File: /kaggle/working/phase7_final/phase7_final_submission.json
